# FinanceRAG Analyst — Retrieval Evaluation

This notebook conducts a reproducible evaluation of sparse, dense, and hybrid retrieval over the frozen SEC filing corpus produced by Notebook 2.

## Frozen retrieval bundle

- Corpus artifact ID: `sec30_959ac4a68dbf`
- SEC filings: 30
- Companies: 5
- Canonical chunks: 4,174
- Dense encoder: `BAAI/bge-small-en-v1.5`
- Dense index: FAISS `IndexFlatIP`
- Sparse retriever: BM25Okapi

## Evaluation objective

The notebook asks:

> On a frozen SEC corpus and independently labeled financial questions, which retrieval method performs best, under what configuration, and where does it fail?

The evaluated retrieval methods are:

1. BM25
2. Dense retrieval
3. Reciprocal Rank Fusion of BM25 and dense rankings
4. Selected metadata-filtering and retrieval enhancements evaluated through explicit ablations

## Evaluation contract

This notebook will:

- Load—not rebuild—the frozen Notebook 2 retrieval artifacts.
- Verify artifact checksums and ordered chunk identities before evaluation.
- Use exact, human-verified chunk IDs as relevance judgments.
- Support multiple relevant chunks and graded relevance.
- Keep question families within one evaluation split.
- Use development questions for configuration selection.
- Preserve an untouched test set for final evaluation.
- Report BM25, dense, and hybrid baselines.
- Distinguish true Recall@k from Hit Rate@k.
- Evaluate unanswerable questions separately from answerable retrieval.
- Save configurations, predictions, metrics, failures, and checksums.

This notebook will not use keyword occurrence or company identity as a substitute for passage-level relevance.

In [ ]:
# Core retrieval and evaluation dependencies.
# Notebook 3 loads the corpus indexes created by Notebook 2.
# It does not recreate corpus embeddings or indexes.

%pip install -q \
    "faiss-cpu>=1.8,<2" \
    "sentence-transformers>=3,<6" \
    "rank-bm25>=0.2.2,<0.3" \
    "pyarrow>=15,<23" \
    "pandas>=2.0,<3" \
    "numpy>=1.24,<3" \
    "scikit-learn>=1.3,<2" \
    "tqdm>=4.66,<5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 88.2 MB/s eta 0:00:00


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import re
import sys
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import faiss
import numpy as np
import pandas as pd
import pyarrow
import rank_bm25
import sklearn
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


# ---------------------------------------------------------------------
# Deterministic notebook state
# ---------------------------------------------------------------------

RANDOM_SEED = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ---------------------------------------------------------------------
# Runtime audit
# ---------------------------------------------------------------------

RUNTIME_INFO = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pyarrow": pyarrow.__version__,
    "faiss": getattr(faiss, "__version__", "unknown"),
    "rank_bm25": getattr(rank_bm25, "__version__", "not exposed"),
    "scikit_learn": sklearn.__version__,
    "torch": torch.__version__,
    "sentence_transformers": __import__(
        "sentence_transformers"
    ).__version__,
    "device": DEVICE,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
}

print(json.dumps(RUNTIME_INFO, indent=2))

assert RANDOM_SEED == 42
assert DEVICE in {"cuda", "cpu"}

if DEVICE == "cuda":
    print("\nGPU runtime detected: READY")
else:
    print(
        "\nWARNING: GPU not detected. The notebook will still work, "
        "but dense query encoding will run on CPU."
    )

{
  "timestamp_utc": "2026-08-05T21:30:36.564578+00:00",
  "python": "3.12.13",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "numpy": "2.0.2",
  "pandas": "2.2.2",
  "pyarrow": "18.1.0",
  "faiss": "1.15.0",
  "rank_bm25": "not exposed",
  "scikit_learn": "1.6.1",
  "torch": "2.11.0+cu128",
  "sentence_transformers": "5.6.0",
  "device": "cuda",
  "cuda_available": true,
  "gpu_name": "Tesla T4"
}

GPU runtime detected: READY


In [ ]:
from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT))

MY_DRIVE = DRIVE_MOUNT / "MyDrive"

assert DRIVE_MOUNT.exists(), "Google Drive mount directory was not created."
assert MY_DRIVE.exists(), f"MyDrive directory not found at: {MY_DRIVE}"

print(f"Google Drive mounted: {DRIVE_MOUNT}")
print(f"MyDrive available:    {MY_DRIVE}")

Mounted at /content/drive
Google Drive mounted: /content/drive
MyDrive available:    /content/drive/MyDrive


Locate the exact Notebook 2 bundle manifest

In [ ]:
EXPECTED_CORPUS_ARTIFACT_ID = "sec30_959ac4a68dbf"
EXPECTED_MANIFEST_NAME = (
    f"retrieval_bundle_{EXPECTED_CORPUS_ARTIFACT_ID}_metadata.json"
)

print(f"Searching for: {EXPECTED_MANIFEST_NAME}")

manifest_candidates = sorted(
    path
    for path in MY_DRIVE.rglob(EXPECTED_MANIFEST_NAME)
    if path.is_file()
)

print(f"Manifest candidates found: {len(manifest_candidates)}")

for index, path in enumerate(manifest_candidates, start=1):
    print(f"[{index}] {path}")

assert manifest_candidates, (
    "The frozen retrieval-bundle manifest was not found in MyDrive. "
    "Confirm that Notebook 2 artifacts were saved to Google Drive."
)

assert len(manifest_candidates) == 1, (
    "Multiple manifests with the same corpus artifact ID were found. "
    "Do not select one silently; inspect the displayed paths first."
)

BUNDLE_MANIFEST_PATH = manifest_candidates[0]
BUNDLE_DIR = BUNDLE_MANIFEST_PATH.parent

print("\nSelected frozen bundle")
print(f"Manifest: {BUNDLE_MANIFEST_PATH}")
print(f"Directory: {BUNDLE_DIR}")

Searching for: retrieval_bundle_sec30_959ac4a68dbf_metadata.json
Manifest candidates found: 1
[1] /content/drive/MyDrive/finance-rag-analyst/data/manifests/retrieval_bundle_sec30_959ac4a68dbf_metadata.json

Selected frozen bundle
Manifest: /content/drive/MyDrive/finance-rag-analyst/data/manifests/retrieval_bundle_sec30_959ac4a68dbf_metadata.json
Directory: /content/drive/MyDrive/finance-rag-analyst/data/manifests


Load and inspect the manifest without assuming its schema

In [ ]:
with BUNDLE_MANIFEST_PATH.open("r", encoding="utf-8") as file:
    bundle_manifest = json.load(file)

assert isinstance(bundle_manifest, dict), (
    "The retrieval-bundle manifest must contain a top-level JSON object."
)

print(f"Manifest loaded: {BUNDLE_MANIFEST_PATH.name}")
print(f"Top-level keys ({len(bundle_manifest)}):")

for key in bundle_manifest:
    value = bundle_manifest[key]
    print(f"  - {key}: {type(value).__name__}")

Manifest loaded: retrieval_bundle_sec30_959ac4a68dbf_metadata.json
Top-level keys (15):
  - artifact_type: str
  - artifact_version: str
  - created_at_utc: str
  - corpus_artifact_id: str
  - row_count: int
  - shared_row_identity: str
  - embedding_model: str
  - embedding_dimension: int
  - embeddings_normalized: bool
  - dense_index: dict
  - sparse_index: dict
  - artifacts: dict
  - component_manifests: dict
  - validation: dict
  - evaluation_scope: str


schema tree

In [ ]:
def describe_json_schema(
    value: Any,
    *,
    name: str = "root",
    depth: int = 0,
    max_depth: int = 4,
    max_items: int = 20,
) -> None:
    """Print a compact structural description of a JSON-compatible object."""

    indent = "  " * depth

    if isinstance(value, dict):
        print(f"{indent}{name}: dict[{len(value)}]")

        if depth >= max_depth:
            return

        items = list(value.items())
        for key, child in items[:max_items]:
            describe_json_schema(
                child,
                name=str(key),
                depth=depth + 1,
                max_depth=max_depth,
                max_items=max_items,
            )

        if len(items) > max_items:
            print(
                f"{indent}  ... {len(items) - max_items} additional keys"
            )

    elif isinstance(value, list):
        print(f"{indent}{name}: list[{len(value)}]")

        if value and depth < max_depth:
            describe_json_schema(
                value[0],
                name="[0]",
                depth=depth + 1,
                max_depth=max_depth,
                max_items=max_items,
            )

    else:
        rendered = repr(value)
        if len(rendered) > 160:
            rendered = rendered[:157] + "..."

        print(
            f"{indent}{name}: {type(value).__name__} = {rendered}"
        )


describe_json_schema(
    bundle_manifest,
    max_depth=5,
    max_items=30,
)

root: dict[15]
  artifact_type: str = 'retrieval_artifact_bundle'
  artifact_version: str = '1.0'
  created_at_utc: str = '2026-08-05T11:08:40.839889+00:00'
  corpus_artifact_id: str = 'sec30_959ac4a68dbf'
  row_count: int = 4174
  shared_row_identity: str = 'chunk_id'
  embedding_model: str = 'BAAI/bge-small-en-v1.5'
  embedding_dimension: int = 384
  embeddings_normalized: bool = True
  dense_index: dict[3]
    implementation: str = 'faiss.IndexFlatIP'
    metric: str = 'inner_product'
    exact_search: bool = True
  sparse_index: dict[5]
    implementation: str = 'rank_bm25.BM25Okapi'
    k1: float = 1.5
    b: float = 0.75
    epsilon: float = 0.25
    pickle_used: bool = False
  artifacts: dict[5]
    chunks: dict[2]
      path: str = '/content/drive/MyDrive/finance-rag-analyst/data/processed/retrieval_chunks_sec30_959ac4a68dbf.parquet'
      sha256: str = 'c41480fd37f6714c27f1c1a4a69e90634ec841cde76523d20d53559266864dd5'
    embeddings: dict[2]
      path: str = '/content/drive/M

Find all path- and checksum-like fields

In [ ]:
def flatten_json(
    value: Any,
    prefix: str = "",
) -> list[tuple[str, Any]]:
    """Flatten nested JSON values into dotted field paths."""

    flattened = []

    if isinstance(value, dict):
        for key, child in value.items():
            child_prefix = f"{prefix}.{key}" if prefix else str(key)
            flattened.extend(flatten_json(child, child_prefix))

    elif isinstance(value, list):
        for index, child in enumerate(value):
            child_prefix = f"{prefix}[{index}]"
            flattened.extend(flatten_json(child, child_prefix))

    else:
        flattened.append((prefix, value))

    return flattened


manifest_leaves = flatten_json(bundle_manifest)

path_like_fields = []
checksum_like_fields = []

for field_name, value in manifest_leaves:
    normalized_name = field_name.lower()

    if isinstance(value, str):
        if any(
            marker in normalized_name
            for marker in ("path", "file", "artifact")
        ):
            path_like_fields.append((field_name, value))

        if any(
            marker in normalized_name
            for marker in ("sha256", "checksum", "hash")
        ):
            checksum_like_fields.append((field_name, value))


print("PATH-LIKE MANIFEST FIELDS")
print("-" * 80)

for field_name, value in path_like_fields:
    print(f"{field_name}\n  {value}")


print("\nCHECKSUM-LIKE MANIFEST FIELDS")
print("-" * 80)

for field_name, value in checksum_like_fields:
    print(f"{field_name}\n  {value}")

PATH-LIKE MANIFEST FIELDS
--------------------------------------------------------------------------------
artifact_type
  retrieval_artifact_bundle
artifact_version
  1.0
corpus_artifact_id
  sec30_959ac4a68dbf
artifacts.chunks.path
  /content/drive/MyDrive/finance-rag-analyst/data/processed/retrieval_chunks_sec30_959ac4a68dbf.parquet
artifacts.chunks.sha256
  c41480fd37f6714c27f1c1a4a69e90634ec841cde76523d20d53559266864dd5
artifacts.embeddings.path
  /content/drive/MyDrive/finance-rag-analyst/data/artifacts/embeddings/chunk_embeddings_sec30_959ac4a68dbf.npy
artifacts.embeddings.sha256
  255b686b41dd35ae675a31ede84d0355c49c5f9800594ab6221828ef0fdea6c5
artifacts.embedding_mapping.path
  /content/drive/MyDrive/finance-rag-analyst/data/artifacts/embeddings/embedding_chunk_mapping_sec30_959ac4a68dbf.parquet
artifacts.embedding_mapping.sha256
  2e3720f84096df150c7e40eea2bfda6df6fadede7325f3a174897927f94488cb
artifacts.faiss_index.path
  /content/drive/MyDrive/finance-rag-analyst/data/artif

Load the frozen retrievers

In [ ]:

# NOTEBOOK 3 BOOTSTRAP — LOAD FROZEN RETRIEVAL SYSTEM

artifact_paths = {
    name: Path(metadata["path"])
    for name, metadata in bundle_manifest["artifacts"].items()
}

# Load persisted artifacts
chunks_df = pd.read_parquet(artifact_paths["chunks"])
embedding_mapping_df = pd.read_parquet(
    artifact_paths["embedding_mapping"]
)
bm25_tokens_df = pd.read_parquet(
    artifact_paths["bm25_tokenized_corpus"]
)

embeddings = np.load(
    artifact_paths["embeddings"],
    mmap_mode="r",
    allow_pickle=False,
)

faiss_index = faiss.read_index(
    str(artifact_paths["faiss_index"])
)

EXPECTED_ROWS = int(bundle_manifest["row_count"])
EMBEDDING_DIM = int(bundle_manifest["embedding_dimension"])

# Minimal evaluation-safety gate
assert len(chunks_df) == EXPECTED_ROWS
assert embeddings.shape == (EXPECTED_ROWS, EMBEDDING_DIM)
assert faiss_index.ntotal == EXPECTED_ROWS
assert faiss_index.d == EMBEDDING_DIM

canonical_ids = chunks_df["chunk_id"].astype(str).to_numpy()

assert np.array_equal(
    canonical_ids,
    embedding_mapping_df["chunk_id"].astype(str).to_numpy(),
)

assert np.array_equal(
    canonical_ids,
    bm25_tokens_df["chunk_id"].astype(str).to_numpy(),
)

# Automatically identify the persisted token column
token_candidates = [
    column
    for column in bm25_tokens_df.columns
    if column != "chunk_id"
]

valid_token_columns = []

for column in token_candidates:
    sample = next(
        (
            value
            for value in bm25_tokens_df[column]
            if value is not None
        ),
        None,
    )

    if isinstance(sample, (list, tuple, np.ndarray)):
        valid_token_columns.append(column)

assert len(valid_token_columns) == 1, (
    f"Could not uniquely identify BM25 token column: "
    f"{valid_token_columns}"
)

BM25_TOKEN_COLUMN = valid_token_columns[0]

bm25_corpus_tokens = [
    tokens.tolist()
    if isinstance(tokens, np.ndarray)
    else list(tokens)
    for tokens in bm25_tokens_df[BM25_TOKEN_COLUMN]
]

sparse_config = bundle_manifest["sparse_index"]

bm25_index = BM25Okapi(
    bm25_corpus_tokens,
    k1=float(sparse_config["k1"]),
    b=float(sparse_config["b"]),
    epsilon=float(sparse_config["epsilon"]),
)

# Load query encoder only; corpus embeddings remain frozen
dense_encoder = SentenceTransformer(
    bundle_manifest["embedding_model"],
    device=DEVICE,
)

assert (
    dense_encoder.get_sentence_embedding_dimension()
    == EMBEDDING_DIM
)

print("Notebook 3 retrieval system: READY")
print(f"Corpus rows:       {EXPECTED_ROWS:,}")
print(f"BM25 token field:  {BM25_TOKEN_COLUMN}")
print(f"Dense index:       {type(faiss_index).__name__}")
print(f"Embedding model:   {bundle_manifest['embedding_model']}")
print(f"Device:            {DEVICE}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Notebook 3 retrieval system: READY
Corpus rows:       4,174
BM25 token field:  tokens
Dense index:       IndexFlatIP
Embedding model:   BAAI/bge-small-en-v1.5
Device:            cuda


/tmp/ipykernel_4015/948671007.py:100: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dense_encoder.get_sentence_embedding_dimension()


Implement BM25, dense, and hybrid RRF retrieval

In [ ]:
# ============================================================
# RETRIEVAL IMPLEMENTATIONS
# BM25 + DENSE + HYBRID RRF + OPTIONAL METADATA FILTERING
# ============================================================

import re
import unicodedata

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

BGE_QUERY_INSTRUCTION = (
    "Represent this sentence for searching relevant passages: "
)

BM25_TOKEN_PATTERN = re.compile(r"\b[\w'-]+\b", flags=re.UNICODE)

assert len(chunks_df) == EXPECTED_ROWS
assert embeddings.shape[0] == EXPECTED_ROWS
assert faiss_index.ntotal == EXPECTED_ROWS


# ------------------------------------------------------------
# Shared helpers
# ------------------------------------------------------------

def tokenize_for_bm25(text: str) -> list[str]:
    """Tokenize query text exactly as required by the frozen BM25 index."""

    if not isinstance(text, str):
        raise TypeError("BM25 text must be a string.")

    normalized_text = unicodedata.normalize(
        "NFKC",
        text,
    ).lower()

    return BM25_TOKEN_PATTERN.findall(normalized_text)


def rows_to_results(
    row_indices: np.ndarray,
    scores: np.ndarray,
    *,
    method: str,
) -> pd.DataFrame:
    """Attach ranked retrieval rows to canonical chunk metadata."""

    row_indices = np.asarray(
        row_indices,
        dtype=np.int64,
    )

    scores = np.asarray(scores)

    if row_indices.ndim != 1 or scores.ndim != 1:
        raise ValueError(
            "row_indices and scores must both be one-dimensional."
        )

    if len(row_indices) != len(scores):
        raise ValueError(
            "row_indices and scores must have the same length."
        )

    if len(row_indices) == 0:
        raise ValueError("Retrieval returned no rows.")

    if (
        (row_indices < 0).any()
        or (row_indices >= EXPECTED_ROWS).any()
    ):
        raise IndexError(
            "One or more retrieval rows fall outside the corpus."
        )

    result = (
        chunks_df
        .iloc[row_indices]
        .copy()
        .reset_index(drop=True)
    )

    result.insert(
        0,
        "rank",
        np.arange(
            1,
            len(result) + 1,
            dtype=np.int64,
        ),
    )

    result.insert(
        1,
        "retrieval_method",
        method,
    )

    result.insert(
        2,
        "retrieval_score",
        scores,
    )

    result.insert(
        3,
        "corpus_row",
        row_indices,
    )

    return result


def build_metadata_mask(
    *,
    ticker=None,
    form_type=None,
    filing_date=None,
) -> np.ndarray:
    """
    Return a Boolean mask for corpus rows matching supplied metadata.

    If every filter is None, every corpus row remains eligible.
    """

    mask = np.ones(
        EXPECTED_ROWS,
        dtype=bool,
    )

    if ticker is not None:
        corpus_tickers = (
            chunks_df["ticker"]
            .astype(str)
            .str.strip()
            .str.upper()
            .to_numpy()
        )

        mask &= corpus_tickers == str(ticker).strip().upper()

    if form_type is not None:
        corpus_form_types = (
            chunks_df["form_type"]
            .astype(str)
            .str.strip()
            .str.upper()
            .to_numpy()
        )

        mask &= (
            corpus_form_types
            == str(form_type).strip().upper()
        )

    if filing_date is not None:
        requested_date = (
            pd.to_datetime(
                filing_date,
                errors="raise",
            )
            .strftime("%Y-%m-%d")
        )

        corpus_dates = (
            pd.to_datetime(
                chunks_df["filing_date"],
                errors="raise",
            )
            .dt.strftime("%Y-%m-%d")
            .to_numpy()
        )

        mask &= corpus_dates == requested_date

    if not mask.any():
        raise ValueError(
            "No corpus rows match the requested metadata: "
            f"ticker={ticker!r}, "
            f"form_type={form_type!r}, "
            f"filing_date={filing_date!r}."
        )

    return mask


def metadata_filter_is_active(
    *,
    ticker=None,
    form_type=None,
    filing_date=None,
) -> bool:
    """Return whether at least one retrieval filter was supplied."""

    return any(
        value is not None
        for value in [
            ticker,
            form_type,
            filing_date,
        ]
    )


# ------------------------------------------------------------
# BM25 retrieval
# ------------------------------------------------------------

def retrieve_bm25(
    query: str,
    *,
    top_k: int = 10,
    ticker=None,
    form_type=None,
    filing_date=None,
) -> pd.DataFrame:
    """Retrieve with BM25, optionally restricted by filing metadata."""

    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query must be a non-empty string.")

    query_tokens = tokenize_for_bm25(query)

    if not query_tokens:
        raise ValueError(
            "The query produced no BM25 tokens."
        )

    metadata_mask = build_metadata_mask(
        ticker=ticker,
        form_type=form_type,
        filing_date=filing_date,
    )

    eligible_rows = np.flatnonzero(metadata_mask)
    eligible_count = len(eligible_rows)

    if not 1 <= top_k <= eligible_count:
        raise ValueError(
            f"top_k must be between 1 and "
            f"{eligible_count:,} for this retrieval scope."
        )

    scores = np.asarray(
        bm25_index.get_scores(query_tokens),
        dtype=np.float64,
    )

    if scores.shape != (EXPECTED_ROWS,):
        raise ValueError(
            "BM25 score vector does not align with the corpus: "
            f"expected {(EXPECTED_ROWS,)}, received {scores.shape}."
        )

    eligible_scores = scores[eligible_rows]

    local_order = np.argsort(
        -eligible_scores,
        kind="stable",
    )[:top_k]

    ranked_rows = eligible_rows[local_order]
    ranked_scores = eligible_scores[local_order]

    is_filtered = metadata_filter_is_active(
        ticker=ticker,
        form_type=form_type,
        filing_date=filing_date,
    )

    return rows_to_results(
        ranked_rows,
        ranked_scores,
        method=(
            "bm25_filtered"
            if is_filtered
            else "bm25"
        ),
    )


# ------------------------------------------------------------
# Dense retrieval
# ------------------------------------------------------------

def retrieve_dense(
    query: str,
    *,
    top_k: int = 10,
    ticker=None,
    form_type=None,
    filing_date=None,
) -> pd.DataFrame:
    """
    Retrieve using normalized BGE query embeddings.

    Unfiltered retrieval uses the frozen FAISS index.
    Filtered retrieval performs exact inner-product scoring over
    only the eligible normalized corpus embeddings.
    """

    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query must be a non-empty string.")

    metadata_mask = build_metadata_mask(
        ticker=ticker,
        form_type=form_type,
        filing_date=filing_date,
    )

    eligible_rows = np.flatnonzero(metadata_mask)
    eligible_count = len(eligible_rows)

    if not 1 <= top_k <= eligible_count:
        raise ValueError(
            f"top_k must be between 1 and "
            f"{eligible_count:,} for this retrieval scope."
        )

    instructed_query = (
        BGE_QUERY_INSTRUCTION
        + query.strip()
    )

    query_embedding = dense_encoder.encode(
        [instructed_query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32,
        order="C",
    )

    if query_embedding.shape != (1, embeddings.shape[1]):
        raise ValueError(
            "Unexpected query-embedding shape: "
            f"{query_embedding.shape}."
        )

    is_filtered = metadata_filter_is_active(
        ticker=ticker,
        form_type=form_type,
        filing_date=filing_date,
    )

    if not is_filtered:
        scores, row_indices = faiss_index.search(
            query_embedding,
            top_k,
        )

        ranked_rows = row_indices[0]
        ranked_scores = scores[0]

    else:
        eligible_embeddings = np.asarray(
            embeddings[eligible_rows],
            dtype=np.float32,
            order="C",
        )

        eligible_scores = (
            eligible_embeddings
            @ query_embedding[0]
        )

        # Stable sorting makes tied scores deterministic.
        local_order = np.argsort(
            -eligible_scores,
            kind="stable",
        )[:top_k]

        ranked_rows = eligible_rows[local_order]
        ranked_scores = eligible_scores[local_order]

    return rows_to_results(
        ranked_rows,
        ranked_scores,
        method=(
            "dense_filtered"
            if is_filtered
            else "dense"
        ),
    )


# ------------------------------------------------------------
# Hybrid retrieval using Reciprocal Rank Fusion
# ------------------------------------------------------------

def retrieve_hybrid_rrf(
    query: str,
    *,
    top_k: int = 10,
    candidate_k: int = 50,
    rrf_k: int = 60,
    ticker=None,
    form_type=None,
    filing_date=None,
) -> pd.DataFrame:
    """
    Fuse BM25 and dense rankings using Reciprocal Rank Fusion.

    RRF score:
        1 / (rrf_k + BM25 rank)
        + 1 / (rrf_k + dense rank)

    The same metadata restrictions are applied to both component
    retrievers before their rankings are fused.
    """

    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query must be a non-empty string.")

    metadata_mask = build_metadata_mask(
        ticker=ticker,
        form_type=form_type,
        filing_date=filing_date,
    )

    eligible_count = int(metadata_mask.sum())

    if not 1 <= top_k <= eligible_count:
        raise ValueError(
            f"top_k must be between 1 and "
            f"{eligible_count:,} for this retrieval scope."
        )

    if not isinstance(candidate_k, int) or candidate_k < 1:
        raise ValueError(
            "candidate_k must be a positive integer."
        )

    if candidate_k < top_k:
        raise ValueError(
            "candidate_k must be at least top_k."
        )

    if not isinstance(rrf_k, int) or rrf_k < 1:
        raise ValueError(
            "rrf_k must be a positive integer."
        )

    effective_candidate_k = min(
        candidate_k,
        eligible_count,
    )

    retrieval_filters = {
        "ticker": ticker,
        "form_type": form_type,
        "filing_date": filing_date,
    }

    bm25_results = retrieve_bm25(
        query,
        top_k=effective_candidate_k,
        **retrieval_filters,
    )

    dense_results = retrieve_dense(
        query,
        top_k=effective_candidate_k,
        **retrieval_filters,
    )

    fused_scores = {}
    bm25_ranks = {}
    dense_ranks = {}

    for result in bm25_results.itertuples(index=False):
        corpus_row = int(result.corpus_row)
        rank = int(result.rank)

        bm25_ranks[corpus_row] = rank

        fused_scores[corpus_row] = (
            fused_scores.get(corpus_row, 0.0)
            + 1.0 / (rrf_k + rank)
        )

    for result in dense_results.itertuples(index=False):
        corpus_row = int(result.corpus_row)
        rank = int(result.rank)

        dense_ranks[corpus_row] = rank

        fused_scores[corpus_row] = (
            fused_scores.get(corpus_row, 0.0)
            + 1.0 / (rrf_k + rank)
        )

    # Resolve exact RRF ties by corpus row for reproducibility.
    ranked_rows = sorted(
        fused_scores,
        key=lambda corpus_row: (
            -fused_scores[corpus_row],
            corpus_row,
        ),
    )[:top_k]

    is_filtered = metadata_filter_is_active(
        ticker=ticker,
        form_type=form_type,
        filing_date=filing_date,
    )

    hybrid_results = rows_to_results(
        np.asarray(
            ranked_rows,
            dtype=np.int64,
        ),
        np.asarray(
            [
                fused_scores[corpus_row]
                for corpus_row in ranked_rows
            ],
            dtype=np.float64,
        ),
        method=(
            "hybrid_rrf_filtered"
            if is_filtered
            else "hybrid_rrf"
        ),
    )

    hybrid_results.insert(
        4,
        "bm25_rank",
        pd.array(
            [
                bm25_ranks.get(corpus_row, pd.NA)
                for corpus_row in ranked_rows
            ],
            dtype="Int64",
        ),
    )

    hybrid_results.insert(
        5,
        "dense_rank",
        pd.array(
            [
                dense_ranks.get(corpus_row, pd.NA)
                for corpus_row in ranked_rows
            ],
            dtype="Int64",
        ),
    )

    return hybrid_results


# ------------------------------------------------------------
# Smoke tests — execution correctness, not quality evaluation
# ------------------------------------------------------------

SMOKE_TEST_QUERY = (
    "What factors affected revenue and operating income?"
)

smoke_bm25 = retrieve_bm25(
    SMOKE_TEST_QUERY,
    top_k=5,
)

smoke_dense = retrieve_dense(
    SMOKE_TEST_QUERY,
    top_k=5,
)

smoke_hybrid = retrieve_hybrid_rrf(
    SMOKE_TEST_QUERY,
    top_k=5,
    candidate_k=50,
    rrf_k=60,
)

smoke_filtered = retrieve_hybrid_rrf(
    SMOKE_TEST_QUERY,
    top_k=5,
    candidate_k=50,
    rrf_k=60,
    ticker="AAPL",
    form_type="10-K",
    filing_date="2025-10-31",
)

for method_name, results in {
    "BM25": smoke_bm25,
    "Dense": smoke_dense,
    "Hybrid RRF": smoke_hybrid,
    "Filtered hybrid": smoke_filtered,
}.items():
    assert len(results) == 5
    assert results["rank"].tolist() == [1, 2, 3, 4, 5]
    assert results["chunk_id"].is_unique
    assert results["corpus_row"].between(
        0,
        EXPECTED_ROWS - 1,
    ).all(), f"{method_name} returned an invalid corpus row."

assert smoke_filtered["ticker"].astype(str).eq(
    "AAPL"
).all()

assert smoke_filtered["form_type"].astype(str).eq(
    "10-K"
).all()

assert (
    pd.to_datetime(
        smoke_filtered["filing_date"],
        errors="raise",
    )
    .dt.strftime("%Y-%m-%d")
    .eq("2025-10-31")
    .all()
)

print("Retrieval implementations: READY")
print(f"Smoke-test query: {SMOKE_TEST_QUERY!r}")
print("BM25:           PASSED")
print("Dense:          PASSED")
print("Hybrid RRF:     PASSED")
print("Filtered hybrid: PASSED")

preview_columns = [
    column
    for column in [
        "rank",
        "retrieval_method",
        "retrieval_score",
        "bm25_rank",
        "dense_rank",
        "ticker",
        "form_type",
        "filing_date",
        "chunk_id",
        "text",
    ]
    if column in smoke_filtered.columns
]

display(smoke_filtered[preview_columns])

Retrieval implementations: READY
Smoke-test query: 'What factors affected revenue and operating income?'
BM25:           PASSED
Dense:          PASSED
Hybrid RRF:     PASSED
Filtered hybrid: PASSED


,rank,retrieval_method,retrieval_score,bm25_rank,dense_rank,ticker,form_type,filing_date,chunk_id,text
0,1,hybrid_rrf_filtered,0.032018,4,1,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0045,"The Company’s net sales and gross margins are subject to volatility and downward pressure due to a variety of factors, including: continued industry-wide global product pricing pressures and product pricing actions that the Company may take in response to such pressures; increased competition; the Company’s ability to effectively stimulate demand for certain of its products and services; compressed product life cycles; supply shortages; potential increases in the cost of components, outside manufacturing services, and developing, acquiring and delivering content for the Company’s services; the Company’s ability to manage product quality and warranty costs effectively; shifts in the mix o..."
1,2,hybrid_rrf_filtered,0.030777,6,4,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0048,"As of September 27, 2025, the Company’s vendor non-trade receivables were concentrated among a few individual vendors located primarily in Asia. If the Company is unable to monitor and limit exposure to credit risk on its trade and vendor non-trade receivables, as well as long-term prepayments, the Company’s results of operations, financial condition and stock price could be materially adversely affected. The Company is subject to changes in tax rates, the adoption of new U.S. or international tax legislation and exposure to additional tax liabilities. The Company is subject to taxes in the U.S. and numerous foreign jurisdictions, including Ireland and Singapore, where a number of the Co..."
2,3,hybrid_rrf_filtered,0.030415,10,2,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0062,"Services gross margin percentage increased during 2025 compared to 2024 primarily due to a different mix of services, partially offset by higher costs. The Company’s future gross margins can be impacted by a variety of factors, as discussed in Part I, Item 1A of this Form 10-K under the heading “Risk Factors.” As a result, the Company believes, in general, gross margins will be subject to volatility and downward pressure. Operating Expenses Operating expenses for 2025, 2024 and 2023 were as follows (dollars in millions): 2025 Change 2024 Change 2023 Research and development $ 34,550 10 % $ 31,370 5 % $ 29,915 Percentage of total net sales 8 % 8 % 8 % Selling, general and administrative $..."
3,4,hybrid_rrf_filtered,0.030077,7,6,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0046,"Gross margins on the Company’s products in foreign countries and on products that include components obtained from foreign suppliers have in the past been adversely affected and could in the future be materially adversely affected by foreign exchange rate fluctuations. The weakening of foreign currencies relative to the U.S. dollar adversely affects the U.S. dollar value of the Company’s foreign currency–denominated sales and earnings, and generally leads the Company to raise international pricing, potentially reducing demand for the Company’s products. In some circumstances, for competitive or other reasons, the Company may decide not to raise international pricing to offset the U.S. do..."
4,5,hybrid_rrf_filtered,0.029380,1,17,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0078,Apple Inc. | 2025 Form 10-K | 34 Derivative Instruments The Company presents derivative assets and liabilities at their gross fair values in the Consolidated Balance Sheets. Income Taxes The Company records certain deferred tax assets and liabilities in connection with the minimum tax on certain foreign earnings created by the TCJA. Leases The Company combines and accounts for lease and nonlease components as a single lease component for leases of corporate and retail facilities. Note 2 – Revenue The Company recognizes revenue at the amount to which it expects to be entitled when control of

benchmark source inventory

In [ ]:
# ============================================================
# BENCHMARK SOURCE INVENTORY
# ============================================================

required_metadata_columns = [
    "ticker",
    "form_type",
    "filing_date",
    "chunk_id",
    "text",
]

missing_columns = [
    column
    for column in required_metadata_columns
    if column not in chunks_df.columns
]

assert not missing_columns, (
    f"Missing benchmark metadata columns: {missing_columns}"
)

benchmark_source_inventory = (
    chunks_df
    .assign(
        ticker=chunks_df["ticker"].astype(str),
        form_type=chunks_df["form_type"].astype(str),
        filing_date=pd.to_datetime(
            chunks_df["filing_date"],
            errors="raise",
        ).dt.date.astype(str),
    )
    .groupby(
        ["ticker", "form_type", "filing_date"],
        as_index=False,
    )
    .agg(
        chunk_count=("chunk_id", "count"),
        total_characters=("text", lambda values: values.str.len().sum()),
    )
    .sort_values(
        ["ticker", "filing_date", "form_type"],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert benchmark_source_inventory["chunk_count"].sum() == EXPECTED_ROWS
assert (
    benchmark_source_inventory[
        ["ticker", "form_type", "filing_date"]
    ]
    .duplicated()
    .sum()
    == 0
)

print("Benchmark source inventory: READY")
print(f"Distinct filings:  {len(benchmark_source_inventory):,}")
print(f"Tickers:           {chunks_df['ticker'].nunique():,}")
print(f"Corpus chunks:     {benchmark_source_inventory['chunk_count'].sum():,}")

display(benchmark_source_inventory)

Benchmark source inventory: READY
Distinct filings:  30
Tickers:           5
Corpus chunks:     4,174


,ticker,form_type,filing_date,chunk_count,total_characters
0,AAPL,10-K,2023-11-03,127,262799
1,AAPL,10-K,2024-11-01,127,266918
2,AAPL,10-Q,2025-08-01,46,79846
3,AAPL,10-K,2025-10-31,129,273292
4,AAPL,10-Q,2026-01-30,40,74418
5,AAPL,10-Q,2026-05-01,59,110256
6,AMZN,10-K,2024-02-02,178,372266
7,AMZN,10-K,2025-02-07,177,368373
8,AMZN,10-Q,2025-08-01,109,223013
9,AMZN,10-Q,2025-10-31,112,232292


pilot queries and candidate judgment pools

In [ ]:

# ============================================================
# 10-QUERY PILOT — REVISED QUERIES AND JUDGMENT POOLS
# ============================================================

pilot_queries = pd.DataFrame(
    [
        {
            "query_id": "P01",
            "query": (
                "What factors drove changes in Apple's product and "
                "services net sales in its 2025 annual filing?"
            ),
            "ticker": "AAPL",
            "form_type": "10-K",
            "filing_date": "2025-10-31",
            "query_category": "financial_explanation",
            "difficulty": "medium",
        },
        {
            "query_id": "P02",
            "query": (
                "What risks did Apple identify from dependence on "
                "third-party manufacturing and suppliers?"
            ),
            "ticker": "AAPL",
            "form_type": "10-K",
            "filing_date": "2024-11-01",
            "query_category": "risk_factor",
            "difficulty": "medium",
        },
        {
            "query_id": "P03",
            "query": (
                "What factors affected AWS sales and operating income "
                "in Amazon's 2026 annual filing?"
            ),
            "ticker": "AMZN",
            "form_type": "10-K",
            "filing_date": "2026-02-06",
            "query_category": "financial_explanation",
            "difficulty": "medium",
        },
        {
            "query_id": "P04",
            "query": (
                "How did Amazon describe the effect of foreign exchange "
                "rates on its reported results?"
            ),
            "ticker": "AMZN",
            "form_type": "10-Q",
            "filing_date": "2026-04-30",
            "query_category": "exact_terminology",
            "difficulty": "medium",
        },
        {
            "query_id": "P05",
            "query": (
                "What drove changes in Google's advertising revenue "
                "in its 2026 annual filing?"
            ),
            "ticker": "GOOGL",
            "form_type": "10-K",
            "filing_date": "2026-02-05",
            "query_category": "financial_explanation",
            "difficulty": "medium",
        },
        {
            "query_id": "P06",
            "query": (
                "What risks did Alphabet identify from competition "
                "and changes caused by artificial intelligence?"
            ),
            "ticker": "GOOGL",
            "form_type": "10-K",
            "filing_date": "2026-02-05",
            "query_category": "lexical_mismatch",
            "difficulty": "hard",
        },
        {
            "query_id": "P07",
            "query": (
                "What factors affected Microsoft Cloud gross margin "
                "in Microsoft's 2025 annual filing?"
            ),
            "ticker": "MSFT",
            "form_type": "10-K",
            "filing_date": "2025-07-30",
            "query_category": "financial_explanation",
            "difficulty": "medium",
        },
        {
            "query_id": "P08",
            "query": (
                "How did Microsoft describe its commitments for data "
                "centers and cloud infrastructure?"
            ),
            "ticker": "MSFT",
            "form_type": "10-Q",
            "filing_date": "2026-04-29",
            "query_category": "filing_specific",
            "difficulty": "hard",
        },
        {
            "query_id": "P09",
            "query": (
                "How did export controls affect NVIDIA's ability to "
                "sell products in China?"
            ),
            "ticker": "NVDA",
            "form_type": "10-K",
            "filing_date": "2026-02-25",
            "query_category": "risk_factor",
            "difficulty": "medium",
        },
        {
            "query_id": "P10",
            "query": (
                "What factors drove changes in NVIDIA's Data Center "
                "revenue in its May 2026 quarterly filing?"
            ),
            "ticker": "NVDA",
            "form_type": "10-Q",
            "filing_date": "2026-05-20",
            "query_category": "filing_specific",
            "difficulty": "medium",
        },
    ]
)


# ------------------------------------------------------------
# Validate pilot-query definitions
# ------------------------------------------------------------

required_query_columns = [
    "query_id",
    "query",
    "ticker",
    "form_type",
    "filing_date",
    "query_category",
    "difficulty",
]

assert list(pilot_queries.columns) == required_query_columns
assert len(pilot_queries) == 10
assert pilot_queries["query_id"].is_unique
assert pilot_queries["query"].str.strip().ne("").all()
assert pilot_queries[
    [
        "ticker",
        "form_type",
        "filing_date",
    ]
].notna().all().all()

pilot_queries["filing_date"] = (
    pd.to_datetime(
        pilot_queries["filing_date"],
        errors="raise",
    )
    .dt.strftime("%Y-%m-%d")
)

available_filings = (
    chunks_df
    .assign(
        ticker=(
            chunks_df["ticker"]
            .astype(str)
            .str.strip()
            .str.upper()
        ),
        form_type=(
            chunks_df["form_type"]
            .astype(str)
            .str.strip()
            .str.upper()
        ),
        filing_date=(
            pd.to_datetime(
                chunks_df["filing_date"],
                errors="raise",
            )
            .dt.strftime("%Y-%m-%d")
        ),
    )
    [
        [
            "ticker",
            "form_type",
            "filing_date",
        ]
    ]
    .drop_duplicates()
)

validated_pilot_queries = pilot_queries.merge(
    available_filings.assign(
        filing_exists=True,
    ),
    on=[
        "ticker",
        "form_type",
        "filing_date",
    ],
    how="left",
    validate="many_to_one",
)

assert validated_pilot_queries["filing_exists"].fillna(
    False
).all(), (
    "At least one pilot query refers to a filing that is "
    "not present in the corpus."
)


# ------------------------------------------------------------
# Build pooled candidates for manual relevance judgments
# ------------------------------------------------------------

def build_judgment_pool(
    query_record,
    *,
    per_method_k: int = 20,
    neighbor_seed_k: int = 10,
) -> pd.DataFrame:
    """
    Pool candidates from six retrieval runs:

    1. Unfiltered BM25
    2. Unfiltered dense
    3. Unfiltered hybrid RRF
    4. Target-filing-filtered BM25
    5. Target-filing-filtered dense
    6. Target-filing-filtered hybrid RRF

    The immediately preceding and following chunks around the
    strongest filtered-hybrid results are also included when they
    belong to the same target filing.

    No relevance judgments are assigned automatically.
    """

    if not 1 <= per_method_k <= EXPECTED_ROWS:
        raise ValueError(
            f"per_method_k must be between 1 and "
            f"{EXPECTED_ROWS:,}."
        )

    if not 1 <= neighbor_seed_k <= per_method_k:
        raise ValueError(
            "neighbor_seed_k must be between 1 and "
            "per_method_k."
        )

    query = query_record["query"]

    target_filters = {
        "ticker": query_record["ticker"],
        "form_type": query_record["form_type"],
        "filing_date": query_record["filing_date"],
    }

    method_results = {
        "bm25_rank": retrieve_bm25(
            query,
            top_k=per_method_k,
        ),
        "dense_rank": retrieve_dense(
            query,
            top_k=per_method_k,
        ),
        "hybrid_rank": retrieve_hybrid_rrf(
            query,
            top_k=per_method_k,
            candidate_k=50,
            rrf_k=60,
        ),
        "filtered_bm25_rank": retrieve_bm25(
            query,
            top_k=per_method_k,
            **target_filters,
        ),
        "filtered_dense_rank": retrieve_dense(
            query,
            top_k=per_method_k,
            **target_filters,
        ),
        "filtered_hybrid_rank": retrieve_hybrid_rrf(
            query,
            top_k=per_method_k,
            candidate_k=50,
            rrf_k=60,
            **target_filters,
        ),
    }

    rank_columns = list(method_results)
    pooled_rows = {}

    def initialize_row(
        corpus_row: int,
        *,
        added_as_neighbor: bool = False,
    ) -> bool:
        """
        Initialize one pooled row.

        Returns True only when a new candidate was introduced.
        """

        if corpus_row in pooled_rows:
            return False

        pooled_rows[corpus_row] = {
            "query_id": query_record["query_id"],
            "query": query,
            "target_ticker": query_record["ticker"],
            "target_form_type": query_record["form_type"],
            "target_filing_date": query_record["filing_date"],
            "query_category": query_record["query_category"],
            "difficulty": query_record["difficulty"],
            "corpus_row": corpus_row,
            **{
                rank_column: pd.NA
                for rank_column in rank_columns
            },
            "added_as_neighbor": added_as_neighbor,
        }

        return True

    # Add every retrieved candidate and preserve its method-specific rank.
    for rank_column, results in method_results.items():
        for result in results.itertuples(index=False):
            corpus_row = int(result.corpus_row)

            initialize_row(corpus_row)

            pooled_rows[corpus_row][rank_column] = int(
                result.rank
            )

    corpus_tickers = (
        chunks_df["ticker"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    corpus_form_types = (
        chunks_df["form_type"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    corpus_dates = (
        pd.to_datetime(
            chunks_df["filing_date"],
            errors="raise",
        )
        .dt.strftime("%Y-%m-%d")
    )

    target_ticker = str(
        query_record["ticker"]
    ).strip().upper()

    target_form_type = str(
        query_record["form_type"]
    ).strip().upper()

    target_filing_date = str(
        query_record["filing_date"]
    )

    def row_matches_target(corpus_row: int) -> bool:
        """Check whether one corpus row belongs to the target filing."""

        return (
            corpus_tickers.iloc[corpus_row]
            == target_ticker
            and corpus_form_types.iloc[corpus_row]
            == target_form_type
            and corpus_dates.iloc[corpus_row]
            == target_filing_date
        )

    # Use the ten strongest filtered-hybrid candidates as
    # boundary-awareness seeds.
    neighbor_seeds = (
        method_results["filtered_hybrid_rank"]
        .head(neighbor_seed_k)["corpus_row"]
        .astype(int)
        .tolist()
    )

    for seed_row in neighbor_seeds:
        for neighbor_row in (
            seed_row - 1,
            seed_row + 1,
        ):
            if not 0 <= neighbor_row < EXPECTED_ROWS:
                continue

            if not row_matches_target(neighbor_row):
                continue

            # Mark as a neighbor only if it was not already included
            # by one of the six retrieval runs.
            initialize_row(
                neighbor_row,
                added_as_neighbor=True,
            )

    pool_df = pd.DataFrame(
        pooled_rows.values()
    )

    # Construct an explicit canonical corpus-row column instead of
    # depending on the current name of chunks_df's index.
    chunk_metadata = chunks_df.copy().reset_index(drop=True)

    chunk_metadata.insert(
        0,
        "corpus_row",
        np.arange(
            len(chunk_metadata),
            dtype=np.int64,
        ),
    )

    pool_df = pool_df.merge(
        chunk_metadata,
        on="corpus_row",
        how="left",
        validate="many_to_one",
    )

    assert pool_df["chunk_id"].notna().all()
    assert pool_df["chunk_id"].is_unique

    pool_df["target_filing_match"] = (
        pool_df["ticker"]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq(target_ticker)
        & pool_df["form_type"]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq(target_form_type)
        & pd.to_datetime(
            pool_df["filing_date"],
            errors="raise",
        )
        .dt.strftime("%Y-%m-%d")
        .eq(target_filing_date)
    )

    pool_df["best_pool_rank"] = (
        pool_df[rank_columns]
        .astype("Float64")
        .min(axis=1)
    )

    # Manual annotation fields. These intentionally remain empty.
    pool_df["relevance"] = pd.Series(
        [pd.NA] * len(pool_df),
        dtype="Int64",
    )

    pool_df["judgment_note"] = ""

    ordered_columns = [
        "query_id",
        "query",
        "target_ticker",
        "target_form_type",
        "target_filing_date",
        "query_category",
        "difficulty",
        "corpus_row",
        "target_filing_match",
        "best_pool_rank",
        "added_as_neighbor",
        *rank_columns,
        *[
            column
            for column in chunk_metadata.columns
            if column != "corpus_row"
        ],
        "relevance",
        "judgment_note",
    ]

    return (
        pool_df[ordered_columns]
        .sort_values(
            [
                "target_filing_match",
                "best_pool_rank",
                "added_as_neighbor",
                "corpus_row",
            ],
            ascending=[
                False,
                True,
                False,
                True,
            ],
            kind="stable",
            na_position="last",
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# Build all ten pilot pools
# ------------------------------------------------------------

judgment_pools = []

for query_record in pilot_queries.to_dict("records"):
    query_pool = build_judgment_pool(
        query_record,
        per_method_k=20,
        neighbor_seed_k=10,
    )

    judgment_pools.append(query_pool)

pilot_judgments_df = pd.concat(
    judgment_pools,
    ignore_index=True,
)


# ------------------------------------------------------------
# Validate the completed unlabeled pools
# ------------------------------------------------------------

assert set(
    pilot_judgments_df["query_id"]
) == set(
    pilot_queries["query_id"]
)

assert pilot_judgments_df["chunk_id"].notna().all()

assert not pilot_judgments_df.duplicated(
    subset=[
        "query_id",
        "corpus_row",
    ]
).any()

assert pilot_judgments_df["relevance"].isna().all()

assert pilot_judgments_df["judgment_note"].eq("").all()

assert (
    pilot_judgments_df
    .groupby("query_id")["filtered_bm25_rank"]
    .apply(lambda values: values.notna().sum())
    .eq(20)
    .all()
)

assert (
    pilot_judgments_df
    .groupby("query_id")["filtered_dense_rank"]
    .apply(lambda values: values.notna().sum())
    .eq(20)
    .all()
)

assert (
    pilot_judgments_df
    .groupby("query_id")["filtered_hybrid_rank"]
    .apply(lambda values: values.notna().sum())
    .eq(20)
    .all()
)

# Every filtered candidate must belong to its query's target filing.
filtered_rank_columns = [
    "filtered_bm25_rank",
    "filtered_dense_rank",
    "filtered_hybrid_rank",
]

filtered_candidate_mask = (
    pilot_judgments_df[filtered_rank_columns]
    .notna()
    .any(axis=1)
)

assert pilot_judgments_df.loc[
    filtered_candidate_mask,
    "target_filing_match",
].all()


# ------------------------------------------------------------
# Summarize the revised pools
# ------------------------------------------------------------

pool_summary = (
    pilot_judgments_df
    .groupby(
        "query_id",
        as_index=False,
    )
    .agg(
        pooled_candidates=(
            "chunk_id",
            "count",
        ),
        target_filing_candidates=(
            "target_filing_match",
            "sum",
        ),
        adjacent_chunks_added=(
            "added_as_neighbor",
            "sum",
        ),
        tickers_retrieved=(
            "ticker",
            "nunique",
        ),
    )
    .merge(
        pilot_queries[
            [
                "query_id",
                "ticker",
                "form_type",
                "filing_date",
                "query_category",
            ]
        ],
        on="query_id",
        how="left",
        validate="one_to_one",
    )
)

assert (
    pool_summary["target_filing_candidates"]
    >= 20
).all()

print("Revised pilot benchmark pools: READY")
print(f"Queries:                    {len(pilot_queries)}")
print(
    "Candidate judgments:        "
    f"{len(pilot_judgments_df):,}"
)
print(
    "Target-filtered runs/query:  "
    "BM25 + dense + hybrid RRF"
)
print(
    "Relevance labels assigned automatically: NO"
)

display(pilot_queries)
display(pool_summary)

Revised pilot benchmark pools: READY
Queries:                    10
Candidate judgments:        685
Target-filtered runs/query:  BM25 + dense + hybrid RRF
Relevance labels assigned automatically: NO


,query_id,query,ticker,form_type,filing_date,query_category,difficulty
0,P01,What factors drove changes in Apple's product and services net sales in its 2025 annual filing?,AAPL,10-K,2025-10-31,financial_explanation,medium
1,P02,What risks did Apple identify from dependence on third-party manufacturing and suppliers?,AAPL,10-K,2024-11-01,risk_factor,medium
2,P03,What factors affected AWS sales and operating income in Amazon's 2026 annual filing?,AMZN,10-K,2026-02-06,financial_explanation,medium
3,P04,How did Amazon describe the effect of foreign exchange rates on its reported results?,AMZN,10-Q,2026-04-30,exact_terminology,medium
4,P05,What drove changes in Google's advertising revenue in its 2026 annual filing?,GOOGL,10-K,2026-02-05,financial_explanation,medium
5,P06,What risks did Alphabet identify from competition and changes caused by artificial intelligence?,GOOGL,10-K,2026-02-05,lexical_mismatch,hard
6,P07,What factors affected Microsoft Cloud gross margin in Microsoft's 2025 annual filing?,MSFT,10-K,2025-07-30,financial_explanation,medium
7,P08,How did Microsoft describe its commitments for data centers and cloud infrastructure?,MSFT,10-Q,2026-04-29,filing_specific,hard
8,P09,How did export controls affect NVIDIA's ability to sell products in China?,NVDA,10-K,2026-02-25,risk_factor,medium
9,P10,What factors drove changes in NVIDIA's Data Center revenue in its May 2026 quarterly filing?,NVDA,10-Q,2026-05-20,filing_specific,medium


,query_id,pooled_candidates,target_filing_candidates,adjacent_chunks_added,tickers_retrieved,ticker,form_type,filing_date,query_category
0,P01,79,48,6,4,AAPL,10-K,2025-10-31,financial_explanation
1,P02,70,37,8,3,AAPL,10-K,2024-11-01,risk_factor
2,P03,66,32,5,3,AMZN,10-K,2026-02-06,financial_explanation
3,P04,58,35,5,2,AMZN,10-Q,2026-04-30,exact_terminology
4,P05,77,42,5,3,GOOGL,10-K,2026-02-05,financial_explanation
5,P06,81,49,8,5,GOOGL,10-K,2026-02-05,lexical_mismatch
6,P07,55,30,2,3,MSFT,10-K,2025-07-30,financial_explanation
7,P08,71,34,6,3,MSFT,10-Q,2026-04-29,filing_specific
8,P09,53,29,3,1,NVDA,10-K,2026-02-25,risk_factor
9,P10,75,39,4,3,NVDA,10-Q,2026-05-20,filing_specific


In [ ]:
# ============================================================
# MANUAL QREL INSPECTION
# ============================================================

QUERY_TO_REVIEW = "P01"

query_details = pilot_queries.loc[
    pilot_queries["query_id"].eq(QUERY_TO_REVIEW)
].iloc[0]

review_columns = [
    "query_id",
    "target_filing_match",
    "best_pool_rank",
    "bm25_rank",
    "dense_rank",
    "hybrid_rank",
    "ticker",
    "form_type",
    "filing_date",
    "chunk_id",
    "text",
    "relevance",
    "judgment_note",
]

review_df = (
    pilot_judgments_df
    .loc[
        pilot_judgments_df["query_id"].eq(QUERY_TO_REVIEW),
        review_columns,
    ]
    .reset_index(drop=True)
)

print(f"Query: {query_details['query']}")
print(
    "Target filing: "
    f"{query_details['ticker']} | "
    f"{query_details['form_type']} | "
    f"{query_details['filing_date']}"
)
print(f"Candidates to review: {len(review_df)}")

pd.set_option("display.max_colwidth", 700)
display(review_df)

Query: What factors drove changes in Apple's product and services net sales in its 2025 annual filing?
Target filing: AAPL | 10-K | 2025-10-31
Candidates to review: 79


,query_id,target_filing_match,best_pool_rank,bm25_rank,dense_rank,hybrid_rank,ticker,form_type,filing_date,chunk_id,text,relevance,judgment_note
0,P01,True,1.0,<NA>,<NA>,7,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0012,"Apple Inc. | 2025 Form 10-K | 3 In addition to Company-owned intellectual property, many of the Company’s products and services include technology or intellectual property that must be licensed from third parties. It may be necessary in the future to seek or renew licenses relating to various aspects of the Company’s products, processes and services. While the Company has generally been able to obtain such licenses on commercially reasonable terms in the past, there is no guarantee that such licenses could be obtained in the future on reasonable terms or at all. Business Seasonality and Product Introductions The Company has historically experienced higher net sales in its first quarter c...",<NA>,
1,P01,True,1.0,15,<NA>,<NA>,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0045,"The Company’s net sales and gross margins are subject to volatility and downward pressure due to a variety of factors, including: continued industry-wide global product pricing pressures and product pricing actions that the Company may take in response to such pressures; increased competition; the Company’s ability to effectively stimulate demand for certain of its products and services; compressed product life cycles; supply shortages; potential increases in the cost of components, outside manufacturing services, and developing, acquiring and delivering content for the Company’s services; the Company’s ability to manage product quality and warranty costs effectively; shifts in the mix o...",<NA>,
2,P01,True,1.0,<NA>,1,8,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0061,"Mac Mac net sales increased during 2025 compared to 2024 primarily due to higher net sales of laptops and desktops. iPad iPad net sales increased during 2025 compared to 2024 primarily due to higher net sales of iPad Air, iPad mini and iPad, partially offset by lower net sales of iPad Pro. Wearables, Home and Accessories Wearables, Home and Accessories net sales decreased during 2025 compared to 2024 primarily due to lower net sales of Accessories and Wearables. Services Services net sales increased during 2025 compared to 2024 primarily due to higher net sales from advertising, the App Store and cloud services. Apple Inc. | 2025 Form 10-K | 23 Gross Margin Products and Services gross ma...",<NA>,
3,P01,True,2.0,17,<NA>,<NA>,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0014,"The Company periodically provides certain information for investors on its corporate website, www.apple.com, and its investor relations website, investor.apple.com. This includes press releases and other information about financial performance, information on corporate governance, and details related to the Company’s annual meeting of shareholders. The information contained on the websites referenced in this Form 10-K is not incorporated by reference into this filing. Further, the Company’s references to website URLs are intended to be inactive textual references only. Apple Inc. | 2025 Form 10-K | 4 Item 1A. Risk Factors The following summarizes factors that could have a material advers...",<NA>,
4,P01,True,2.0,<NA>,2,9,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0062,"Services gross margin percentage increased during 2025 compared to 2024 primarily due to a different mix of services, partially offset by higher costs. The Company’s future gross margins can be impacted by a variety of factors, as discussed in Part I, Item 1A of this Form 10-K under the heading “Risk Factors.” As a result, the Company believes, in general, gross margins will be subject to volatility and downward pressure. Operating Expenses Operating expenses for 2025, 2024 and 2023 were as follows (dollars in millions): 2025 Change 2024 Change 2023 Research and development $ 34,

In [ ]:
# ============================================================
# PERSIST PILOT BENCHMARK CHECKPOINT TO GOOGLE DRIVE
# ============================================================

from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount("/content/drive")

CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/FinanceRAG-Analyst/"
    "artifacts/evaluation"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# CSV: convenient for manual inspection and annotation.
pilot_judgments_df.to_csv(
    CHECKPOINT_DIR / "pilot_judgments_checkpoint.csv",
    index=False,
)

# Parquet: preserves data types more reliably for notebook recovery.
pilot_judgments_df.to_parquet(
    CHECKPOINT_DIR / "pilot_judgments_checkpoint.parquet",
    index=False,
)

pilot_queries.to_csv(
    CHECKPOINT_DIR / "pilot_queries.csv",
    index=False,
)

pool_summary.to_csv(
    CHECKPOINT_DIR / "pilot_pool_summary.csv",
    index=False,
)

print("Pilot benchmark checkpoint: SAVED")
print(f"Directory: {CHECKPOINT_DIR}")
print(
    "Judgment rows saved: "
    f"{len(pilot_judgments_df):,}"
)
print(
    "Queries saved: "
    f"{len(pilot_queries):,}"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pilot benchmark checkpoint: SAVED
Directory: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation
Judgment rows saved: 685
Queries saved: 10


In [ ]:
# ============================================================
# VERIFY SAVED CHECKPOINT
# ============================================================

saved_judgments = pd.read_parquet(
    CHECKPOINT_DIR / "pilot_judgments_checkpoint.parquet"
)

saved_queries = pd.read_csv(
    CHECKPOINT_DIR / "pilot_queries.csv"
)

assert len(saved_judgments) == len(pilot_judgments_df)
assert len(saved_queries) == len(pilot_queries)

assert set(saved_judgments["query_id"]) == set(
    pilot_judgments_df["query_id"]
)

assert not saved_judgments.duplicated(
    subset=["query_id", "corpus_row"]
).any()

assert saved_judgments["chunk_id"].notna().all()

print("Checkpoint verification: PASSED")
print(f"Recovered judgment rows: {len(saved_judgments):,}")
print(f"Recovered queries:       {len(saved_queries):,}")

Checkpoint verification: PASSED
Recovered judgment rows: 685
Recovered queries:       10


In [ ]:
# ============================================================
# SAVE COMPLETE FINANCERAG RUNTIME ARTIFACTS TO GOOGLE DRIVE
# Run once while the current runtime is still alive
# ============================================================

from google.colab import drive
from pathlib import Path
import json
import numpy as np
import pandas as pd

drive.mount("/content/drive")

RUNTIME_DIR = Path(
    "/content/drive/MyDrive/FinanceRAG-Analyst/"
    "artifacts/runtime"
)

RUNTIME_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 1. Validate and save the canonical chunk corpus
# ------------------------------------------------------------

if "chunks_df" not in globals():
    raise NameError(
        "chunks_df is not present in memory. "
        "Do not continue until the corpus-loading cell has run."
    )

chunks_checkpoint = chunks_df.reset_index(drop=True).copy()

assert len(chunks_checkpoint) == 4_174
assert chunks_checkpoint["chunk_id"].notna().all()
assert chunks_checkpoint["chunk_id"].is_unique
assert chunks_checkpoint["text"].notna().all()

chunks_checkpoint.to_parquet(
    RUNTIME_DIR / "chunks_df.parquet",
    index=False,
)

# CSV is optional but useful for direct inspection.
chunks_checkpoint.to_csv(
    RUNTIME_DIR / "chunks_df.csv",
    index=False,
)

# ------------------------------------------------------------
# 2. Locate the existing embedding matrix
# ------------------------------------------------------------

expected_shape = (
    len(chunks_checkpoint),
    384,
)

embedding_candidates = {}

for variable_name, variable_value in list(globals().items()):
    if isinstance(variable_value, np.ndarray):
        if variable_value.shape == expected_shape:
            embedding_candidates[variable_name] = variable_value

if not embedding_candidates:
    raise NameError(
        "No NumPy embedding matrix with shape "
        f"{expected_shape} was found in memory."
    )

if len(embedding_candidates) > 1:
    print(
        "Multiple possible embedding matrices found: "
        f"{list(embedding_candidates)}"
    )
    print(
        "Using the first matrix after validating it."
    )

embedding_variable_name = next(iter(embedding_candidates))
runtime_embeddings = embedding_candidates[
    embedding_variable_name
]

assert runtime_embeddings.shape == expected_shape
assert np.issubdtype(
    runtime_embeddings.dtype,
    np.floating,
)
assert np.isfinite(runtime_embeddings).all()

embedding_norms = np.linalg.norm(
    runtime_embeddings,
    axis=1,
)

assert np.allclose(
    embedding_norms,
    1.0,
    atol=1e-4,
), (
    "The detected embedding matrix does not appear to contain "
    "the normalized corpus embeddings."
)

np.save(
    RUNTIME_DIR / "chunk_embeddings.npy",
    runtime_embeddings.astype(
        np.float32,
        copy=False,
    ),
    allow_pickle=False,
)

# ------------------------------------------------------------
# 3. Save runtime metadata
# ------------------------------------------------------------

runtime_metadata = {
    "corpus_rows": int(len(chunks_checkpoint)),
    "embedding_rows": int(runtime_embeddings.shape[0]),
    "embedding_dimensions": int(runtime_embeddings.shape[1]),
    "embedding_dtype": str(runtime_embeddings.dtype),
    "source_embedding_variable": embedding_variable_name,
    "chunk_ids_unique": bool(
        chunks_checkpoint["chunk_id"].is_unique
    ),
    "embeddings_normalized": True,
}

with open(
    RUNTIME_DIR / "runtime_metadata.json",
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        runtime_metadata,
        metadata_file,
        indent=2,
    )

# ------------------------------------------------------------
# 4. Verify artifacts by loading them back from Drive
# ------------------------------------------------------------

verified_chunks = pd.read_parquet(
    RUNTIME_DIR / "chunks_df.parquet"
)

verified_embeddings = np.load(
    RUNTIME_DIR / "chunk_embeddings.npy",
    allow_pickle=False,
)

assert len(verified_chunks) == len(chunks_checkpoint)
assert verified_embeddings.shape == expected_shape

assert verified_chunks["chunk_id"].tolist() == (
    chunks_checkpoint["chunk_id"].tolist()
)

assert np.isfinite(verified_embeddings).all()

assert np.allclose(
    verified_embeddings,
    runtime_embeddings,
    atol=1e-6,
)

print("Complete runtime artifact checkpoint: SAVED")
print(f"Directory:          {RUNTIME_DIR}")
print(f"Corpus rows:        {len(verified_chunks):,}")
print(f"Embedding shape:    {verified_embeddings.shape}")
print(f"Embedding dtype:    {verified_embeddings.dtype}")
print(
    "Embedding source:   "
    f"{embedding_variable_name}"
)
print("Row alignment:      PRESERVED")
print("Save verification:  PASSED")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Complete runtime artifact checkpoint: SAVED
Directory:          /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/runtime
Corpus rows:        4,174
Embedding shape:    (4174, 384)
Embedding dtype:    float32
Embedding source:   embeddings
Row alignment:      PRESERVED
Save verification:  PASSED


In [ ]:
# ============================================================
# FINANCERAG — COMPLETE ONE-CELL RUNTIME RECOVERY
# Run this after every Colab runtime restart/disconnection.
#
# Restores:
#   - 4,174 corpus chunks
#   - saved 384-dimensional embeddings
#   - BM25 index
#   - FAISS cosine-similarity index
#   - embedding query model
#   - BM25, dense, and hybrid-RRF retrieval functions
#   - pilot queries
#   - 685-row judgment checkpoint
#   - pool summary
#
# Does NOT re-encode the corpus.
# ============================================================

# Install only the lightweight runtime dependencies.
%pip install -q rank-bm25 sentence-transformers faiss-cpu pyarrow

from google.colab import drive
from pathlib import Path

import json
import re
import time
import numpy as np
import pandas as pd
import faiss

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# 1. Mount Google Drive and define artifact locations
# ------------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=False,
)

PROJECT_DIR = Path(
    "/content/drive/MyDrive/FinanceRAG-Analyst"
)

RUNTIME_DIR = (
    PROJECT_DIR
    / "artifacts"
    / "runtime"
)

CHECKPOINT_DIR = (
    PROJECT_DIR
    / "artifacts"
    / "evaluation"
)

required_runtime_files = [
    RUNTIME_DIR / "chunks_df.parquet",
    RUNTIME_DIR / "chunk_embeddings.npy",
    RUNTIME_DIR / "runtime_metadata.json",
]

required_evaluation_files = [
    CHECKPOINT_DIR / "pilot_judgments_checkpoint.parquet",
    CHECKPOINT_DIR / "pilot_queries.csv",
    CHECKPOINT_DIR / "pilot_pool_summary.csv",
]

missing_files = [
    path
    for path in (
        required_runtime_files
        + required_evaluation_files
    )
    if not path.exists()
]

if missing_files:
    missing_text = "\n".join(
        f"  - {path}"
        for path in missing_files
    )

    raise FileNotFoundError(
        "Recovery cannot continue because these saved "
        f"artifacts are missing:\n{missing_text}"
    )


# ------------------------------------------------------------
# 2. Load the canonical corpus and saved embeddings
# ------------------------------------------------------------

recovery_start = time.perf_counter()

chunks_df = (
    pd.read_parquet(
        RUNTIME_DIR / "chunks_df.parquet"
    )
    .reset_index(drop=True)
)

chunk_embeddings = np.load(
    RUNTIME_DIR / "chunk_embeddings.npy",
    allow_pickle=False,
).astype(
    np.float32,
    copy=False,
)

with open(
    RUNTIME_DIR / "runtime_metadata.json",
    "r",
    encoding="utf-8",
) as metadata_file:
    runtime_metadata = json.load(metadata_file)

EXPECTED_ROWS = int(
    runtime_metadata["corpus_rows"]
)

EXPECTED_DIMENSIONS = int(
    runtime_metadata["embedding_dimensions"]
)


# ------------------------------------------------------------
# 3. Validate corpus–embedding row alignment
# ------------------------------------------------------------

required_chunk_columns = [
    "chunk_id",
    "text",
    "ticker",
    "form_type",
    "filing_date",
]

missing_chunk_columns = [
    column
    for column in required_chunk_columns
    if column not in chunks_df.columns
]

if missing_chunk_columns:
    raise ValueError(
        "Recovered chunks_df is missing required columns: "
        f"{missing_chunk_columns}"
    )

assert len(chunks_df) == EXPECTED_ROWS

assert chunk_embeddings.shape == (
    EXPECTED_ROWS,
    EXPECTED_DIMENSIONS,
)

assert chunks_df["chunk_id"].notna().all()
assert chunks_df["chunk_id"].is_unique
assert chunks_df["text"].notna().all()

assert np.issubdtype(
    chunk_embeddings.dtype,
    np.floating,
)

assert np.isfinite(
    chunk_embeddings
).all()

embedding_norms = np.linalg.norm(
    chunk_embeddings,
    axis=1,
)

assert np.allclose(
    embedding_norms,
    1.0,
    atol=1e-4,
), (
    "Recovered corpus embeddings are not unit-normalized."
)


# ------------------------------------------------------------
# 4. Normalize corpus metadata used by filtering
# ------------------------------------------------------------

corpus_tickers = (
    chunks_df["ticker"]
    .astype(str)
    .str.strip()
    .str.upper()
)

corpus_form_types = (
    chunks_df["form_type"]
    .astype(str)
    .str.strip()
    .str.upper()
)

corpus_filing_dates = (
    pd.to_datetime(
        chunks_df["filing_date"],
        errors="raise",
    )
    .dt.strftime("%Y-%m-%d")
)


def resolve_candidate_rows(
    *,
    ticker=None,
    form_type=None,
    filing_date=None,
):
    """Return corpus rows satisfying optional metadata filters."""

    mask = np.ones(
        EXPECTED_ROWS,
        dtype=bool,
    )

    if ticker is not None:
        normalized_ticker = str(
            ticker
        ).strip().upper()

        mask &= corpus_tickers.eq(
            normalized_ticker
        ).to_numpy()

    if form_type is not None:
        normalized_form_type = str(
            form_type
        ).strip().upper()

        mask &= corpus_form_types.eq(
            normalized_form_type
        ).to_numpy()

    if filing_date is not None:
        normalized_filing_date = (
            pd.to_datetime(
                filing_date,
                errors="raise",
            )
            .strftime("%Y-%m-%d")
        )

        mask &= corpus_filing_dates.eq(
            normalized_filing_date
        ).to_numpy()

    candidate_rows = np.flatnonzero(
        mask
    ).astype(np.int64)

    if len(candidate_rows) == 0:
        raise ValueError(
            "No corpus chunks matched the supplied metadata "
            "filters."
        )

    return candidate_rows


# ------------------------------------------------------------
# 5. Rebuild BM25
# ------------------------------------------------------------

TOKEN_PATTERN = re.compile(
    r"[A-Za-z0-9]+(?:['.-][A-Za-z0-9]+)*"
)


def tokenize_for_bm25(text):
    """Apply deterministic lowercase lexical tokenization."""

    return TOKEN_PATTERN.findall(
        str(text).lower()
    )


tokenized_corpus = [
    tokenize_for_bm25(text)
    for text in chunks_df["text"].tolist()
]

bm25_index = BM25Okapi(
    tokenized_corpus
)


# ------------------------------------------------------------
# 6. Rebuild FAISS inner-product index
#
# Since corpus embeddings are unit-normalized, inner product
# is equivalent to cosine similarity.
# ------------------------------------------------------------

faiss_index = faiss.IndexFlatIP(
    EXPECTED_DIMENSIONS
)

faiss_index.add(
    np.ascontiguousarray(
        chunk_embeddings,
        dtype=np.float32,
    )
)

assert faiss_index.ntotal == EXPECTED_ROWS


# ------------------------------------------------------------
# 7. Reload the query-embedding model
#
# This must match the model used to create the stored corpus
# embeddings.
# ------------------------------------------------------------

EMBEDDING_MODEL_NAME = runtime_metadata.get(
    "embedding_model_name"
)

if not EMBEDDING_MODEL_NAME:
    raise ValueError(
        "runtime_metadata.json does not specify "
        "'embedding_model_name'. Recovery has been stopped "
        "to prevent a query/corpus embedding-model mismatch."
    )

if EMBEDDING_MODEL_NAME != "BAAI/bge-small-en-v1.5":
    raise ValueError(
        "Unexpected embedding model recorded in runtime metadata: "
        f"{EMBEDDING_MODEL_NAME!r}"
    )

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)


def encode_query(query):
    """Encode one query using normalized float32 embeddings."""

    if not isinstance(query, str) or not query.strip():
        raise ValueError(
            "query must be a non-empty string."
        )

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(
        np.float32,
        copy=False,
    )

    assert query_embedding.shape == (
        1,
        EXPECTED_DIMENSIONS,
    )

    assert np.isfinite(
        query_embedding
    ).all()

    return query_embedding


# ------------------------------------------------------------
# 8. Common retrieval-result constructor
# ------------------------------------------------------------

def construct_retrieval_results(
    corpus_rows,
    scores,
    *,
    retrieval_method,
    bm25_ranks=None,
    dense_ranks=None,
):
    """Attach ranks, retrieval fields, and corpus metadata."""

    corpus_rows = np.asarray(
        corpus_rows,
        dtype=np.int64,
    )

    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    result = chunks_df.iloc[
        corpus_rows
    ].copy()

    result.insert(
        0,
        "corpus_row",
        corpus_rows,
    )

    result.insert(
        0,
        "dense_rank",
        (
            dense_ranks
            if dense_ranks is not None
            else pd.array(
                [pd.NA] * len(result),
                dtype="Int64",
            )
        ),
    )

    result.insert(
        0,
        "bm25_rank",
        (
            bm25_ranks
            if bm25_ranks is not None
            else pd.array(
                [pd.NA] * len(result),
                dtype="Int64",
            )
        ),
    )

    result.insert(
        0,
        "retrieval_score",
        scores,
    )

    result.insert(
        0,
        "retrieval_method",
        retrieval_method,
    )

    result.insert(
        0,
        "rank",
        np.arange(
            1,
            len(result) + 1,
            dtype=np.int64,
        ),
    )

    return result.reset_index(
        drop=True
    )


# ------------------------------------------------------------
# 9. BM25 retrieval
# ------------------------------------------------------------

def retrieve_bm25(
    query,
    *,
    top_k=10,
    ticker=None,
    form_type=None,
    filing_date=None,
):
    """Retrieve chunks with BM25 and optional metadata filters."""

    if not 1 <= top_k <= EXPECTED_ROWS:
        raise ValueError(
            f"top_k must be between 1 and {EXPECTED_ROWS:,}."
        )

    candidate_rows = resolve_candidate_rows(
        ticker=ticker,
        form_type=form_type,
        filing_date=filing_date,
    )

    query_tokens = tokenize_for_bm25(
        query
    )

    all_scores = np.asarray(
        bm25_index.get_scores(
            query_tokens
        ),
        dtype=np.float64,
    )

    candidate_scores = all_scores[
        candidate_rows
    ]

    selected_count = min(
        top_k,
        len(candidate_rows),
    )

    selected_positions = np.argsort(
        -candidate_scores,
        kind="stable",
    )[:selected_count]

    selected_rows = candidate_rows[
        selected_positions
    ]

    selected_scores = candidate_scores[
        selected_positions
    ]

    method_name = (
        "bm25_filtered"
        if any(
            value is not None
            for value in (
                ticker,
                form_type,
                filing_date,
            )
        )
        else "bm25"
    )

    return construct_retrieval_results(
        selected_rows,
        selected_scores,
        retrieval_method=method_name,
    )


# ------------------------------------------------------------
# 10. Dense retrieval
# ------------------------------------------------------------

def retrieve_dense(
    query,
    *,
    top_k=10,
    ticker=None,
    form_type=None,
    filing_date=None,
):
    """Retrieve chunks using normalized dense similarity."""

    if not 1 <= top_k <= EXPECTED_ROWS:
        raise ValueError(
            f"top_k must be between 1 and {EXPECTED_ROWS:,}."
        )

    query_embedding = encode_query(
        query
    )

    filters_supplied = any(
        value is not None
        for value in (
            ticker,
            form_type,
            filing_date,
        )
    )

    if filters_supplied:
        candidate_rows = resolve_candidate_rows(
            ticker=ticker,
            form_type=form_type,
            filing_date=filing_date,
        )

        candidate_scores = (
            chunk_embeddings[
                candidate_rows
            ]
            @ query_embedding[0]
        )

        selected_count = min(
            top_k,
            len(candidate_rows),
        )

        selected_positions = np.argsort(
            -candidate_scores,
            kind="stable",
        )[:selected_count]

        selected_rows = candidate_rows[
            selected_positions
        ]

        selected_scores = candidate_scores[
            selected_positions
        ]

        method_name = "dense_filtered"

    else:
        selected_count = min(
            top_k,
            EXPECTED_ROWS,
        )

        selected_scores, selected_rows = (
            faiss_index.search(
                query_embedding,
                selected_count,
            )
        )

        selected_rows = selected_rows[
            0
        ].astype(np.int64)

        selected_scores = selected_scores[
            0
        ].astype(np.float64)

        method_name = "dense"

    return construct_retrieval_results(
        selected_rows,
        selected_scores,
        retrieval_method=method_name,
    )


# ------------------------------------------------------------
# 11. Reciprocal-rank-fusion hybrid retrieval
# ------------------------------------------------------------

def retrieve_hybrid_rrf(
    query,
    *,
    top_k=10,
    candidate_k=50,
    rrf_k=60,
    ticker=None,
    form_type=None,
    filing_date=None,
):
    """Fuse BM25 and dense candidate rankings using RRF."""

    if not 1 <= top_k <= EXPECTED_ROWS:
        raise ValueError(
            f"top_k must be between 1 and {EXPECTED_ROWS:,}."
        )

    if candidate_k < top_k:
        raise ValueError(
            "candidate_k must be greater than or equal "
            "to top_k."
        )

    if rrf_k <= 0:
        raise ValueError(
            "rrf_k must be positive."
        )

    candidate_rows = resolve_candidate_rows(
        ticker=ticker,
        form_type=form_type,
        filing_date=filing_date,
    )

    effective_candidate_k = min(
        candidate_k,
        len(candidate_rows),
    )

    bm25_results = retrieve_bm25(
        query,
        top_k=effective_candidate_k,
        ticker=ticker,
        form_type=form_type,
        filing_date=filing_date,
    )

    dense_results = retrieve_dense(
        query,
        top_k=effective_candidate_k,
        ticker=ticker,
        form_type=form_type,
        filing_date=filing_date,
    )

    bm25_rank_map = {
        int(row.corpus_row): int(row.rank)
        for row in bm25_results.itertuples(
            index=False
        )
    }

    dense_rank_map = {
        int(row.corpus_row): int(row.rank)
        for row in dense_results.itertuples(
            index=False
        )
    }

    fused_rows = sorted(
        set(bm25_rank_map)
        | set(dense_rank_map)
    )

    fused_records = []

    for corpus_row in fused_rows:
        bm25_rank = bm25_rank_map.get(
            corpus_row
        )

        dense_rank = dense_rank_map.get(
            corpus_row
        )

        rrf_score = 0.0

        if bm25_rank is not None:
            rrf_score += 1.0 / (
                rrf_k + bm25_rank
            )

        if dense_rank is not None:
            rrf_score += 1.0 / (
                rrf_k + dense_rank
            )

        fused_records.append(
            {
                "corpus_row": corpus_row,
                "retrieval_score": rrf_score,
                "bm25_rank": bm25_rank,
                "dense_rank": dense_rank,
            }
        )

    fused_records.sort(
        key=lambda record: (
            -record["retrieval_score"],
            record["bm25_rank"]
            if record["bm25_rank"] is not None
            else np.inf,
            record["dense_rank"]
            if record["dense_rank"] is not None
            else np.inf,
            record["corpus_row"],
        )
    )

    fused_records = fused_records[
        :min(
            top_k,
            len(fused_records),
        )
    ]

    selected_rows = [
        record["corpus_row"]
        for record in fused_records
    ]

    selected_scores = [
        record["retrieval_score"]
        for record in fused_records
    ]

    bm25_ranks = pd.array(
        [
            record["bm25_rank"]
            for record in fused_records
        ],
        dtype="Int64",
    )

    dense_ranks = pd.array(
        [
            record["dense_rank"]
            for record in fused_records
        ],
        dtype="Int64",
    )

    filters_supplied = any(
        value is not None
        for value in (
            ticker,
            form_type,
            filing_date,
        )
    )

    method_name = (
        "hybrid_rrf_filtered"
        if filters_supplied
        else "hybrid_rrf"
    )

    return construct_retrieval_results(
        selected_rows,
        selected_scores,
        retrieval_method=method_name,
        bm25_ranks=bm25_ranks,
        dense_ranks=dense_ranks,
    )


# ------------------------------------------------------------
# 12. Recover evaluation checkpoints
# ------------------------------------------------------------

pilot_judgments_df = pd.read_parquet(
    CHECKPOINT_DIR
    / "pilot_judgments_checkpoint.parquet"
)

pilot_queries = pd.read_csv(
    CHECKPOINT_DIR
    / "pilot_queries.csv"
)

pool_summary = pd.read_csv(
    CHECKPOINT_DIR
    / "pilot_pool_summary.csv"
)

pilot_queries["filing_date"] = (
    pd.to_datetime(
        pilot_queries["filing_date"],
        errors="raise",
    )
    .dt.strftime("%Y-%m-%d")
)

if "target_filing_date" in pilot_judgments_df.columns:
    pilot_judgments_df[
        "target_filing_date"
    ] = (
        pd.to_datetime(
            pilot_judgments_df[
                "target_filing_date"
            ],
            errors="raise",
        )
        .dt.strftime("%Y-%m-%d")
    )

if "filing_date" in pool_summary.columns:
    pool_summary["filing_date"] = (
        pd.to_datetime(
            pool_summary["filing_date"],
            errors="raise",
        )
        .dt.strftime("%Y-%m-%d")
    )


# ------------------------------------------------------------
# 13. Validate recovered benchmark state
# ------------------------------------------------------------

assert len(pilot_queries) == 10
assert pilot_queries["query_id"].is_unique

assert len(pilot_judgments_df) == 685

assert set(
    pilot_judgments_df["query_id"]
) == set(
    pilot_queries["query_id"]
)

assert pilot_judgments_df[
    "chunk_id"
].notna().all()

assert not pilot_judgments_df.duplicated(
    subset=[
        "query_id",
        "corpus_row",
    ]
).any()

assert len(pool_summary) == 10


# ------------------------------------------------------------
# 14. Run a complete post-recovery smoke test
# ------------------------------------------------------------

smoke_query = (
    "What factors affected revenue and operating income?"
)

smoke_bm25 = retrieve_bm25(
    smoke_query,
    top_k=5,
)

smoke_dense = retrieve_dense(
    smoke_query,
    top_k=5,
)

smoke_hybrid = retrieve_hybrid_rrf(
    smoke_query,
    top_k=5,
    candidate_k=50,
    rrf_k=60,
)

smoke_filtered = retrieve_hybrid_rrf(
    smoke_query,
    top_k=5,
    candidate_k=50,
    rrf_k=60,
    ticker="AAPL",
    form_type="10-K",
    filing_date="2025-10-31",
)

assert len(smoke_bm25) == 5
assert len(smoke_dense) == 5
assert len(smoke_hybrid) == 5
assert len(smoke_filtered) == 5

assert smoke_bm25["rank"].tolist() == [
    1, 2, 3, 4, 5
]

assert smoke_dense["rank"].tolist() == [
    1, 2, 3, 4, 5
]

assert smoke_hybrid["rank"].tolist() == [
    1, 2, 3, 4, 5
]

assert (
    smoke_filtered["ticker"]
    .astype(str)
    .str.upper()
    .eq("AAPL")
    .all()
)

assert (
    smoke_filtered["form_type"]
    .astype(str)
    .str.upper()
    .eq("10-K")
    .all()
)

assert (
    pd.to_datetime(
        smoke_filtered["filing_date"],
        errors="raise",
    )
    .dt.strftime("%Y-%m-%d")
    .eq("2025-10-31")
    .all()
)

recovery_seconds = (
    time.perf_counter()
    - recovery_start
)


# ------------------------------------------------------------
# 15. Recovery report
# ------------------------------------------------------------

print("=" * 62)
print("FINANCERAG RUNTIME RECOVERY: PASSED")
print("=" * 62)
print(f"Corpus rows:             {len(chunks_df):,}")
print(f"Embedding shape:         {chunk_embeddings.shape}")
print(f"Embedding dtype:         {chunk_embeddings.dtype}")
print(f"FAISS indexed vectors:   {faiss_index.ntotal:,}")
print(f"BM25 indexed documents:  {len(tokenized_corpus):,}")
print(f"Pilot queries:           {len(pilot_queries):,}")
print(f"Judgment rows recovered: {len(pilot_judgments_df):,}")
print(
    "Completed labels:        "
    f"{pilot_judgments_df['relevance'].notna().sum():,}"
)
print(
    "Unlabeled judgments:     "
    f"{pilot_judgments_df['relevance'].isna().sum():,}"
)
print(f"Recovery time:           {recovery_seconds:.2f} seconds")
print("Corpus re-encoded:       NO")
print("=" * 62)

display(
    smoke_filtered[
        [
            "rank",
            "retrieval_method",
            "retrieval_score",
            "bm25_rank",
            "dense_rank",
            "ticker",
            "form_type",
            "filing_date",
            "chunk_id",
            "text",
        ]
    ]
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

FINANCERAG RUNTIME RECOVERY: PASSED
Corpus rows:             4,174
Embedding shape:         (4174, 384)
Embedding dtype:         float32
FAISS indexed vectors:   4,174
BM25 indexed documents:  4,174
Pilot queries:           10
Judgment rows recovered: 685
Completed labels:        20
Unlabeled judgments:     665
Recovery time:           3.75 seconds
Corpus re-encoded:       NO


,rank,retrieval_method,retrieval_score,bm25_rank,dense_rank,ticker,form_type,filing_date,chunk_id,text
0,1,hybrid_rrf_filtered,0.032018,4,1,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0045,The Company’s net sales and gross margins are ...
1,2,hybrid_rrf_filtered,0.031319,1,7,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0078,Apple Inc. | 2025 Form 10-K | 34 Derivative In...
2,3,hybrid_rrf_filtered,0.030536,6,5,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0048,"As of September 27, 2025, the Company’s vendor..."
3,4,hybrid_rrf_filtered,0.030415,10,2,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0062,Services gross margin percentage increased dur...
4,5,hybrid_rrf_filtered,0.028850,13,6,AAPL,10-K,2025-10-31,aapl_10k_000032019325000079__chunk_0079,The Company’s process for estimating SSPs with...


In [ ]:
# ============================================================
# PERMANENTLY RECORD THE CORRECT EMBEDDING MODEL
# ============================================================

import json
from pathlib import Path

metadata_path = RUNTIME_DIR / "runtime_metadata.json"

with open(
    metadata_path,
    "r",
    encoding="utf-8",
) as metadata_file:
    runtime_metadata = json.load(metadata_file)

runtime_metadata["embedding_model_name"] = (
    "BAAI/bge-small-en-v1.5"
)

runtime_metadata["embedding_dimensions"] = 384
runtime_metadata["embeddings_normalized"] = True

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        runtime_metadata,
        metadata_file,
        indent=2,
        sort_keys=True,
    )

# Load it back from Drive and verify persistence.
with open(
    metadata_path,
    "r",
    encoding="utf-8",
) as metadata_file:
    verified_runtime_metadata = json.load(metadata_file)

assert (
    verified_runtime_metadata["embedding_model_name"]
    == "BAAI/bge-small-en-v1.5"
)

assert (
    int(
        verified_runtime_metadata[
            "embedding_dimensions"
        ]
    )
    == 384
)

assert (
    verified_runtime_metadata[
        "embeddings_normalized"
    ]
    is True
)

print("Embedding-model metadata update: PASSED")
print(f"Metadata file: {metadata_path}")
print(
    "Embedding model: "
    f"{verified_runtime_metadata['embedding_model_name']}"
)
print(
    "Embedding dimensions: "
    f"{verified_runtime_metadata['embedding_dimensions']}"
)
print(
    "Normalized embeddings: "
    f"{verified_runtime_metadata['embeddings_normalized']}"
)

Embedding-model metadata update: PASSED
Metadata file: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/runtime/runtime_metadata.json
Embedding model: BAAI/bge-small-en-v1.5
Embedding dimensions: 384
Normalized embeddings: True


In [ ]:
# ============================================================
# NOTEBOOK 3 — MANUAL RELEVANCE ANNOTATION SETUP
#
# Frozen scale:
#   2 = directly relevant
#   1 = partially relevant / useful supporting context
#   0 = irrelevant
#
# This cell:
#   - validates the recovered 685-row judgment pool
#   - initializes annotation columns safely
#   - writes the frozen rubric to Drive
#   - creates incremental label-saving functions
#   - displays P01's first 10 unlabeled candidates
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import json
import textwrap
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

JUDGMENT_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "pilot_judgments_checkpoint.parquet"
)

RUBRIC_PATH = (
    CHECKPOINT_DIR
    / "annotation_rubric.json"
)

ANNOTATION_PROGRESS_PATH = (
    CHECKPOINT_DIR
    / "annotation_progress.csv"
)


# ------------------------------------------------------------
# 2. Validate the recovered benchmark
# ------------------------------------------------------------

assert JUDGMENT_CHECKPOINT_PATH.exists()

assert len(pilot_judgments_df) == 685

assert pilot_judgments_df["query_id"].notna().all()
assert pilot_judgments_df["chunk_id"].notna().all()

assert not pilot_judgments_df.duplicated(
    subset=[
        "query_id",
        "corpus_row",
    ]
).any()

assert len(pilot_queries) == 10
assert pilot_queries["query_id"].is_unique


# ------------------------------------------------------------
# 3. Initialize annotation columns without overwriting work
# ------------------------------------------------------------

if "relevance" not in pilot_judgments_df.columns:
    pilot_judgments_df["relevance"] = pd.Series(
        pd.array(
            [pd.NA] * len(pilot_judgments_df),
            dtype="Int64",
        ),
        index=pilot_judgments_df.index,
    )
else:
    pilot_judgments_df["relevance"] = (
        pd.to_numeric(
            pilot_judgments_df["relevance"],
            errors="coerce",
        )
        .astype("Int64")
    )

if "annotation_note" not in pilot_judgments_df.columns:
    pilot_judgments_df["annotation_note"] = ""

if "annotated_at_utc" not in pilot_judgments_df.columns:
    pilot_judgments_df["annotated_at_utc"] = ""

if "annotation_version" not in pilot_judgments_df.columns:
    pilot_judgments_df["annotation_version"] = ""

existing_labels = (
    pilot_judgments_df["relevance"]
    .dropna()
    .astype(int)
)

if not existing_labels.isin([0, 1, 2]).all():
    invalid_labels = sorted(
        existing_labels[
            ~existing_labels.isin([0, 1, 2])
        ].unique()
    )

    raise ValueError(
        "Existing relevance column contains invalid labels: "
        f"{invalid_labels}"
    )


# ------------------------------------------------------------
# 4. Freeze and persist the annotation rubric
# ------------------------------------------------------------

ANNOTATION_VERSION = "pilot_v1_2026-08-05"

annotation_rubric = {
    "annotation_version": ANNOTATION_VERSION,
    "task": (
        "Judge whether a retrieved SEC-filing chunk contains "
        "information useful for answering the evaluation query."
    ),
    "labels": {
        "2": {
            "name": "directly_relevant",
            "definition": (
                "The chunk directly answers the question or contains "
                "specific evidence required to construct the answer."
            ),
        },
        "1": {
            "name": "partially_relevant",
            "definition": (
                "The chunk provides useful supporting context, a "
                "related fact, or part of the answer, but cannot answer "
                "the question adequately by itself."
            ),
        },
        "0": {
            "name": "irrelevant",
            "definition": (
                "The chunk does not help answer the question, even if "
                "it comes from the correct company or filing."
            ),
        },
    },
    "decision_rules": [
        (
            "Judge relevance to the query, not merely topical or "
            "keyword overlap."
        ),
        (
            "Do not reward a chunk solely because it belongs to the "
            "correct ticker, form type, or filing date."
        ),
        (
            "Tables and fragmented text may receive 2 if they contain "
            "the requested evidence clearly."
        ),
        (
            "A chunk that discusses the correct broad topic but lacks "
            "the requested fact should normally receive 1, not 2."
        ),
        (
            "Navigation text, boilerplate, unrelated risk disclosures, "
            "and accidental keyword matches receive 0."
        ),
        (
            "Judge each query–chunk pair independently. The same chunk "
            "may receive different labels for different queries."
        ),
    ],
}

with open(
    RUBRIC_PATH,
    "w",
    encoding="utf-8",
) as rubric_file:
    json.dump(
        annotation_rubric,
        rubric_file,
        indent=2,
    )


# ------------------------------------------------------------
# 5. Checkpoint writer
# ------------------------------------------------------------

def save_annotation_checkpoint():
    """Persist all current annotations and a progress summary."""

    invalid_mask = (
        pilot_judgments_df["relevance"].notna()
        & ~pilot_judgments_df["relevance"].isin([0, 1, 2])
    )

    if invalid_mask.any():
        invalid_values = sorted(
            pilot_judgments_df.loc[
                invalid_mask,
                "relevance",
            ]
            .astype(str)
            .unique()
        )

        raise ValueError(
            "Checkpoint not saved because invalid relevance "
            f"values were found: {invalid_values}"
        )

    if pilot_judgments_df.duplicated(
        subset=[
            "query_id",
            "corpus_row",
        ]
    ).any():
        raise ValueError(
            "Checkpoint not saved because duplicate "
            "query–corpus-row pairs were found."
        )

    pilot_judgments_df.to_parquet(
        JUDGMENT_CHECKPOINT_PATH,
        index=False,
    )

    progress = (
        pilot_judgments_df
        .assign(
            is_labeled=(
                pilot_judgments_df["relevance"].notna()
            )
        )
        .groupby(
            "query_id",
            as_index=False,
        )
        .agg(
            total_candidates=("chunk_id", "size"),
            completed_labels=("is_labeled", "sum"),
        )
    )

    progress["unlabeled_candidates"] = (
        progress["total_candidates"]
        - progress["completed_labels"]
    )

    progress["percent_complete"] = (
        100
        * progress["completed_labels"]
        / progress["total_candidates"]
    ).round(1)

    progress.to_csv(
        ANNOTATION_PROGRESS_PATH,
        index=False,
    )

    # Reload from Drive to verify the save.
    verified_checkpoint = pd.read_parquet(
        JUDGMENT_CHECKPOINT_PATH
    )

    assert len(verified_checkpoint) == len(
        pilot_judgments_df
    )

    assert (
        verified_checkpoint["relevance"]
        .fillna(-1)
        .astype(int)
        .tolist()
        ==
        pilot_judgments_df["relevance"]
        .fillna(-1)
        .astype(int)
        .tolist()
    )

    completed = int(
        pilot_judgments_df["relevance"]
        .notna()
        .sum()
    )

    print("Annotation checkpoint: SAVED AND VERIFIED")
    print(f"Completed labels: {completed:,}/685")
    print(f"Remaining labels: {685 - completed:,}")

    return progress


# ------------------------------------------------------------
# 6. Safe batch-labeling function
# ------------------------------------------------------------

def apply_annotation_batch(
    query_id,
    labels,
    *,
    notes=None,
    overwrite=False,
):
    """
    Apply and immediately save labels for one query.

    Parameters
    ----------
    query_id:
        Example: "P01"

    labels:
        Dictionary mapping corpus_row to relevance.
        Example:
            {
                123: 2,
                456: 1,
                789: 0,
            }

    notes:
        Optional dictionary mapping corpus_row to a short note.

    overwrite:
        False by default. Prevents accidental replacement of an
        existing label.
    """

    normalized_query_id = str(
        query_id
    ).strip()

    valid_query_ids = set(
        pilot_judgments_df["query_id"]
        .astype(str)
    )

    if normalized_query_id not in valid_query_ids:
        raise KeyError(
            f"Unknown query_id: {normalized_query_id!r}"
        )

    if not isinstance(labels, dict) or not labels:
        raise ValueError(
            "labels must be a non-empty dictionary mapping "
            "corpus_row to 0, 1, or 2."
        )

    notes = notes or {}

    normalized_labels = {
        int(corpus_row): int(relevance)
        for corpus_row, relevance in labels.items()
    }

    invalid_values = {
        value
        for value in normalized_labels.values()
        if value not in {0, 1, 2}
    }

    if invalid_values:
        raise ValueError(
            "Every relevance label must be 0, 1, or 2. "
            f"Invalid values: {sorted(invalid_values)}"
        )

    target_mask = (
        pilot_judgments_df["query_id"]
        .astype(str)
        .eq(normalized_query_id)
        &
        pilot_judgments_df["corpus_row"]
        .astype(int)
        .isin(normalized_labels)
    )

    matched_rows = pilot_judgments_df.loc[
        target_mask,
        "corpus_row",
    ].astype(int)

    missing_rows = (
        set(normalized_labels)
        - set(matched_rows)
    )

    if missing_rows:
        raise KeyError(
            f"These corpus rows are not candidates for "
            f"{normalized_query_id}: {sorted(missing_rows)}"
        )

    already_labeled_mask = (
        target_mask
        & pilot_judgments_df["relevance"].notna()
    )

    if already_labeled_mask.any() and not overwrite:
        conflicts = (
            pilot_judgments_df.loc[
                already_labeled_mask,
                [
                    "corpus_row",
                    "relevance",
                ],
            ]
            .to_dict("records")
        )

        raise ValueError(
            "Existing labels would be overwritten. "
            "Review these rows or explicitly pass "
            f"overwrite=True: {conflicts}"
        )

    timestamp = datetime.now(
        timezone.utc
    ).isoformat()

    for corpus_row, relevance in normalized_labels.items():
        row_mask = (
            pilot_judgments_df["query_id"]
            .astype(str)
            .eq(normalized_query_id)
            &
            pilot_judgments_df["corpus_row"]
            .astype(int)
            .eq(corpus_row)
        )

        pilot_judgments_df.loc[
            row_mask,
            "relevance",
        ] = relevance

        pilot_judgments_df.loc[
            row_mask,
            "annotation_note",
        ] = str(
            notes.get(
                corpus_row,
                "",
            )
        )

        pilot_judgments_df.loc[
            row_mask,
            "annotated_at_utc",
        ] = timestamp

        pilot_judgments_df.loc[
            row_mask,
            "annotation_version",
        ] = ANNOTATION_VERSION

    pilot_judgments_df["relevance"] = (
        pilot_judgments_df["relevance"]
        .astype("Int64")
    )

    progress = save_annotation_checkpoint()

    print(
        f"Applied {len(normalized_labels)} labels "
        f"to {normalized_query_id}."
    )

    return progress


# ------------------------------------------------------------
# 7. Readable annotation-batch display
# ------------------------------------------------------------

def show_annotation_batch(
    query_id,
    *,
    batch_size=10,
    start=0,
    unlabeled_only=True,
):
    """Display a readable batch with complete, untruncated text."""

    normalized_query_id = str(
        query_id
    ).strip()

    query_match = pilot_queries.loc[
        pilot_queries["query_id"]
        .astype(str)
        .eq(normalized_query_id)
    ]

    if query_match.empty:
        raise KeyError(
            f"Unknown query_id: {normalized_query_id!r}"
        )

    query_row = query_match.iloc[0]

    batch = pilot_judgments_df.loc[
        pilot_judgments_df["query_id"]
        .astype(str)
        .eq(normalized_query_id)
    ].copy()

    if unlabeled_only:
        batch = batch.loc[
            batch["relevance"].isna()
        ]

    batch = (
        batch
        .sort_values(
            "corpus_row",
            kind="stable",
        )
        .iloc[
            start:start + batch_size
        ]
        .copy()
    )

    if batch.empty:
        print(
            f"No matching candidates remain for "
            f"{normalized_query_id}."
        )
        return batch

    query_column_candidates = [
        "query",
        "question",
        "query_text",
    ]

    query_text = None

    for column in query_column_candidates:
        if column in query_row.index:
            query_text = query_row[column]
            break

    print("=" * 100)
    print(f"QUERY ID: {normalized_query_id}")

    if query_text is not None:
        print(f"QUESTION: {query_text}")

    for column, label in [
        ("ticker", "TARGET TICKER"),
        ("form_type", "TARGET FORM"),
        ("filing_date", "TARGET FILING DATE"),
    ]:
        if column in query_row.index:
            print(f"{label}: {query_row[column]}")

    total_for_query = int(
        pilot_judgments_df["query_id"]
        .astype(str)
        .eq(normalized_query_id)
        .sum()
    )

    completed_for_query = int(
        (
            pilot_judgments_df["query_id"]
            .astype(str)
            .eq(normalized_query_id)
            &
            pilot_judgments_df["relevance"]
            .notna()
        ).sum()
    )

    print(
        f"PROGRESS: {completed_for_query}/"
        f"{total_for_query} labeled"
    )
    print("=" * 100)

    for display_number, row in enumerate(
        batch.itertuples(index=False),
        start=1,
    ):
        row_data = row._asdict()

        print()
        print("-" * 100)
        print(
            f"CANDIDATE {display_number} | "
            f"corpus_row={row_data['corpus_row']} | "
            f"chunk_id={row_data['chunk_id']}"
        )

        metadata_parts = []

        for column in [
            "ticker",
            "form_type",
            "filing_date",
        ]:
            if column in row_data:
                metadata_parts.append(
                    f"{column}={row_data[column]}"
                )

        if metadata_parts:
            print(" | ".join(metadata_parts))

        rank_columns = [
            "bm25_rank",
            "dense_rank",
            "hybrid_rank",
        ]

        available_ranks = [
            (
                f"{column}="
                f"{row_data[column]}"
            )
            for column in rank_columns
            if (
                column in row_data
                and pd.notna(row_data[column])
            )
        ]

        if available_ranks:
            print(
                "Retrieved ranks: "
                + " | ".join(available_ranks)
            )

        print()
        print("FULL CHUNK TEXT:")
        print(
            textwrap.fill(
                str(row_data["text"]),
                width=100,
                replace_whitespace=False,
            )
        )

    print()
    print("=" * 100)
    print(
        "Assign each corpus_row one label: "
        "2 = direct, 1 = partial, 0 = irrelevant."
    )
    print("=" * 100)

    return batch


# ------------------------------------------------------------
# 8. Save initialized structure and show P01 batch 1
# ------------------------------------------------------------

progress = save_annotation_checkpoint()

p01_batch_1 = show_annotation_batch(
    "P01",
    batch_size=10,
    start=0,
    unlabeled_only=True,
)

Annotation checkpoint: SAVED AND VERIFIED
Completed labels: 0/685
Remaining labels: 685
QUERY ID: P01
QUESTION: What factors drove changes in Apple's product and services net sales in its 2025 annual filing?
TARGET TICKER: AAPL
TARGET FORM: 10-K
TARGET FILING DATE: 2025-10-31
PROGRESS: 0/79 labeled

----------------------------------------------------------------------------------------------------
CANDIDATE 1 | corpus_row=27 | chunk_id=aapl_10k_000032019323000106__chunk_0027
ticker=AAPL | form_type=10-K | filing_date=2023-11-03
Retrieved ranks: bm25_rank=5.0 | hybrid_rank=3.0

FULL CHUNK TEXT:
This analysis may be based on factors such as the market position of the Company and its products,
the anticipated revenue that may be generated, expected future growth of product sales, and the
costs of developing such applications and services. The Company’s minority market share in the
global smartphone, personal computer and tablet markets can make developers less inclined to develop
or upgr

In [ ]:
apply_annotation_batch(
    "P01",
    labels={
        27: 0,
        42: 0,
        54: 0,
        56: 0,
        57: 0,
        75: 0,
        170: 0,
        182: 0,
        185: 0,
        186: 0,
    },
    notes={
        27: "Wrong filing year: 2023; does not report 2025 sales-change factors.",
        42: "Wrong filing year: 2023; general risk factors, not 2025 results.",
        54: "Wrong filing year: 2023 segment results.",
        56: "Wrong filing year: 2023 product and Services performance.",
        57: "Wrong filing year: 2023 sales and gross-margin discussion.",
        75: "Wrong filing year: 2023 revenue disclosure.",
        170: "Wrong filing year: 2024 general risk factors.",
        182: "Wrong filing year: 2024 announcements and macroeconomic context.",
        185: "Wrong filing year: 2024 product and Services performance.",
        186: "Wrong filing year: 2024 gross margin and operating expenses.",
    },
)

Annotation checkpoint: SAVED AND VERIFIED
Completed labels: 10/685
Remaining labels: 675
Applied 10 labels to P01.


,query_id,total_candidates,completed_labels,unlabeled_candidates,percent_complete
0,P01,79,10,69,12.7
1,P02,70,0,70,0.0
2,P03,66,0,66,0.0
3,P04,58,0,58,0.0
4,P05,77,0,77,0.0
5,P06,81,0,81,0.0
6,P07,55,0,55,0.0
7,P08,71,0,71,0.0
8,P09,53,0,53,0.0
9,P10,75,0,75,0.0


In [ ]:
p01_batch_2 = show_annotation_batch(
    "P01",
    batch_size=10,
    start=0,
    unlabeled_only=True,
)

QUERY ID: P01
QUESTION: What factors drove changes in Apple's product and services net sales in its 2025 annual filing?
TARGET TICKER: AAPL
TARGET FORM: 10-K
TARGET FILING DATE: 2025-10-31
PROGRESS: 10/79 labeled

----------------------------------------------------------------------------------------------------
CANDIDATE 1 | corpus_row=203 | chunk_id=aapl_10k_000032019324000123__chunk_0076
ticker=AAPL | form_type=10-K | filing_date=2024-11-01
Retrieved ranks: dense_rank=19.0

FULL CHUNK TEXT:
Net sales disaggregated by significant products and services for 2024, 2023 and 2022 were as follows
(in millions): 2024 2023 2022 iPhone $ 201,183 $ 200,583 $ 205,489 Mac 29,984 29,357 40,177 iPad
26,694 28,300 29,292 Wearables, Home and Accessories 37,005 39,845 41,241 Services (1) 96,169 85,200
78,129 Total net sales $ 391,035 $ 383,285 $ 394,328 (1) Services net sales include amortization of
the deferred value of services bundled in the sales price of certain products. Total net sales
includ

In [ ]:
apply_annotation_batch(
    "P01",
    labels={
        203: 0,
        257: 0,
        259: 0,
        260: 0,
        261: 0,
        262: 0,
        263: 0,
        265: 0,
        266: 1,
        267: 0,
    },
    notes={
        203: (
            "Wrong filing year: 2024; provides historical sales "
            "figures but not factors driving 2025 changes."
        ),
        257: (
            "2025 filing cover and table of contents; no evidence "
            "about product or Services net-sales changes."
        ),
        259: (
            "General filing conventions and company background; "
            "does not explain 2025 sales changes."
        ),
        260: (
            "Describes Apple's product lines and some services; "
            "does not report sales-change factors."
        ),
        261: (
            "Describes service offerings and geographic segments; "
            "does not explain changes in 2025 net sales."
        ),
        262: (
            "Provides distribution-channel mix and general "
            "competition discussion, not actual 2025 sales drivers."
        ),
        263: (
            "General competitive factors and market conditions; "
            "not tied to Apple's reported 2025 net-sales changes."
        ),
        265: (
            "Discusses R&D and intellectual property; unrelated "
            "to the requested 2025 net-sales changes."
        ),
        266: (
            "Useful general context: seasonality, product launches, "
            "channel inventory and customer anticipation can affect "
            "net sales, but the chunk does not explain the actual "
            "2025 year-over-year product and Services changes."
        ),
        267: (
            "Human-capital and investor-information boilerplate; "
            "unrelated to product or Services net-sales changes."
        ),
    },
)

Annotation checkpoint: SAVED AND VERIFIED
Completed labels: 20/685
Remaining labels: 665
Applied 10 labels to P01.


,query_id,total_candidates,completed_labels,unlabeled_candidates,percent_complete
0,P01,79,20,59,25.3
1,P02,70,0,70,0.0
2,P03,66,0,66,0.0
3,P04,58,0,58,0.0
4,P05,77,0,77,0.0
5,P06,81,0,81,0.0
6,P07,55,0,55,0.0
7,P08,71,0,71,0.0
8,P09,53,0,53,0.0
9,P10,75,0,75,0.0


gemini API automation test

In [ ]:
# ============================================================
# NOTEBOOK 3 — GEMINI JUDGE CONNECTION + MODEL DISCOVERY
#
# This cell:
#   - installs/updates Google's Gemini SDK
#   - securely reads GEMINI_API_KEY from Colab Secrets
#   - discovers models available to this specific API key
#   - selects the best approved text model
#   - tests schema-constrained JSON output
#   - DOES NOT modify pilot_judgments_df
# ============================================================

!pip -q install -U google-genai pydantic

from typing import Literal

from google import genai
from google.genai import types
from google.colab import userdata
from pydantic import BaseModel, Field


# ------------------------------------------------------------
# 1. Securely retrieve the Gemini API key
# ------------------------------------------------------------

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "Could not read GEMINI_API_KEY from Colab Secrets. "
        "Confirm that the secret exists and notebook access is enabled."
    ) from exc

if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY is empty or unavailable."
    )


# ------------------------------------------------------------
# 2. Create the Gemini client
# ------------------------------------------------------------

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)


# ------------------------------------------------------------
# 3. Discover models available to this API key
# ------------------------------------------------------------

available_model_names = set()

try:
    for model in gemini_client.models.list():
        model_name = getattr(model, "name", None)

        if model_name:
            available_model_names.add(
                model_name.removeprefix("models/")
            )

except Exception as exc:
    raise RuntimeError(
        "The API key was loaded, but Gemini model discovery failed. "
        "Check that the key is valid and that Gemini API access is enabled."
    ) from exc


if not available_model_names:
    raise RuntimeError(
        "Gemini returned an empty model list for this API key."
    )


# ------------------------------------------------------------
# 4. Select the best approved available judge model
#
# Order matters. The first available model is selected.
# Preview is allowed only as a fallback.
# ------------------------------------------------------------

PREFERRED_JUDGE_MODELS = [
    "gemini-3.5-flash",
    "gemini-3.1-flash-lite",
    "gemini-3-flash-preview",
    "gemini-2.5-flash-lite",
    "gemini-2.5-flash",
]

GEMINI_JUDGE_MODEL = next(
    (
        model_name
        for model_name in PREFERRED_JUDGE_MODELS
        if model_name in available_model_names
    ),
    None,
)

if GEMINI_JUDGE_MODEL is None:
    available_gemini_models = sorted(
        model_name
        for model_name in available_model_names
        if "gemini" in model_name.lower()
    )

    raise RuntimeError(
        "None of the approved Gemini judge models is available "
        "to this API key.\n\n"
        f"Approved models checked:\n{PREFERRED_JUDGE_MODELS}\n\n"
        "Gemini models visible to this key:\n"
        f"{available_gemini_models}"
    )


GEMINI_PROMPT_VERSION = (
    "financerag_judge_v1_2026-08-05"
)


# ------------------------------------------------------------
# 5. Define the structured-output schema
# ------------------------------------------------------------

class ConnectivityJudgment(BaseModel):
    relevance: int = Field(
        ge=0,
        le=2,
        description=(
            "Relevance label: "
            "0 = irrelevant, "
            "1 = partially relevant, "
            "2 = directly relevant."
        ),
    )

    confidence: Literal[
        "high",
        "medium",
        "low",
    ] = Field(
        description="Confidence in the relevance judgment."
    )

    rationale: str = Field(
        min_length=1,
        max_length=300,
        description=(
            "Brief explanation grounded only in the supplied "
            "question and candidate passage."
        ),
    )


# ------------------------------------------------------------
# 6. Run a harmless structured-output test
# ------------------------------------------------------------

test_prompt = """
You are testing a structured relevance-judgment pipeline.

Relevance rubric:

2 = The passage directly answers the question or contains
specific evidence required to answer it.

1 = The passage provides useful supporting context or part
of the answer, but does not adequately answer the question.

0 = The passage does not help answer the question.

Question:
What was Example Company's 2025 net income?

Candidate passage:
Example Company reported 2025 net income of $125 million.

Return:
- relevance
- confidence
- a brief rationale
""".strip()


try:
    test_response = gemini_client.models.generate_content(
        model=GEMINI_JUDGE_MODEL,
        contents=test_prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_schema=ConnectivityJudgment,
        ),
    )

except Exception as exc:
    raise RuntimeError(
        "Gemini model discovery succeeded, but the structured-output "
        f"test failed for model {GEMINI_JUDGE_MODEL!r}."
    ) from exc


test_judgment = test_response.parsed

if test_judgment is None:
    raise RuntimeError(
        "Gemini returned a response, but the SDK could not parse it "
        "into the required structured-output schema."
    )

if test_judgment.relevance != 2:
    raise AssertionError(
        "The connection worked, but Gemini gave an unexpected label "
        "for the simple direct-evidence test.\n"
        f"Returned judgment: {test_judgment}"
    )


# ------------------------------------------------------------
# 7. Verify that no benchmark judgments were modified
# ------------------------------------------------------------

if "pilot_judgments_df" in globals():
    completed_labels_after_test = int(
        pilot_judgments_df["relevance"]
        .notna()
        .sum()
    )

    if completed_labels_after_test != 20:
        raise AssertionError(
            "Expected the existing benchmark to contain exactly "
            f"20 completed labels, but found "
            f"{completed_labels_after_test}."
        )
else:
    completed_labels_after_test = None


# ------------------------------------------------------------
# 8. Report results without exposing the API key
# ------------------------------------------------------------

print("=" * 72)
print("GEMINI JUDGE CONNECTION TEST: PASSED")
print("=" * 72)
print(f"Selected model:  {GEMINI_JUDGE_MODEL}")
print(f"Prompt version:  {GEMINI_PROMPT_VERSION}")
print(f"Test label:      {test_judgment.relevance}")
print(f"Confidence:      {test_judgment.confidence}")
print(f"Rationale:       {test_judgment.rationale}")
print("API key:         Loaded securely; value not displayed")
print("Judgments added: 0")
print("Judgments changed: 0")

if completed_labels_after_test is not None:
    print(
        "Existing completed benchmark labels: "
        f"{completed_labels_after_test}/685"
    )

print("=" * 72)

GEMINI JUDGE CONNECTION TEST: PASSED
Selected model:  gemini-3.5-flash
Prompt version:  financerag_judge_v1_2026-08-05
Test label:      2
Confidence:      high
Rationale:       The candidate passage directly answers the question by stating that Example Company's 2025 net income was $125 million.
API key:         Loaded securely; value not displayed
Judgments added: 0
Judgments changed: 0
Existing completed benchmark labels: 20/685


In [ ]:
# ============================================================
# NOTEBOOK 3 — GROQ JUDGE: BATCHED, RESUMABLE BLINDED CALIBRATION
#
# Replace the failed Gemini calibration cell with this cell.
#
# This cell:
#   - judges the same 20 human-labeled P01 candidates
#   - batches five candidates into each Groq request
#   - therefore needs only four fresh API requests
#   - never sends human labels or retrieval ranks to Groq
#   - checkpoints after every successful batch
#   - resumes automatically after interruption
#   - never modifies pilot_judgments_df
# ============================================================


# ------------------------------------------------------------
# 0. Install and import dependencies
# ------------------------------------------------------------

import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "groq",
    ],
    check=True,
)


from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import json
import os
import re
import time

import numpy as np
import pandas as pd

from groq import Groq

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
)


# ------------------------------------------------------------
# 1. Groq configuration
# ------------------------------------------------------------

GROQ_JUDGE_MODEL = "openai/gpt-oss-120b"

GROQ_PROMPT_VERSION = (
    "financerag_groq_judge_v1_2026-08-05"
)

GROQ_CALIBRATION_BATCH_SIZE = 5

GROQ_MAX_ATTEMPTS_PER_BATCH = 6

GROQ_MAX_OUTPUT_TOKENS = 2200


# ------------------------------------------------------------
# 2. Securely load the Groq API key
# ------------------------------------------------------------

def load_groq_api_key():
    """
    Load GROQ_API_KEY without displaying or saving it in notebook output.

    Search order:
      1. Environment variable
      2. Google Colab Secrets
      3. Hidden manual prompt
    """

    api_key = os.environ.get(
        "GROQ_API_KEY",
        "",
    ).strip()

    if api_key:
        return api_key

    try:
        from google.colab import userdata

        api_key = (
            userdata.get("GROQ_API_KEY")
            or ""
        ).strip()

        if api_key:
            return api_key

    except Exception:
        pass

    api_key = getpass(
        "Enter GROQ_API_KEY "
        "(the input will remain hidden): "
    ).strip()

    if not api_key:
        raise RuntimeError(
            "No Groq API key was supplied."
        )

    return api_key


groq_client = Groq(
    api_key=load_groq_api_key()
)


# ------------------------------------------------------------
# 3. Validate required Notebook 3 objects
# ------------------------------------------------------------

required_global_names = [
    "pilot_queries",
    "pilot_judgments_df",
    "CHECKPOINT_DIR",
]

missing_globals = [
    name
    for name in required_global_names
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        "Run the earlier Notebook 3 retrieval and "
        "human-labeling cells first.\n"
        f"Missing objects: {missing_globals}"
    )


required_judgment_columns = {
    "query_id",
    "corpus_row",
    "chunk_id",
    "relevance",
}

missing_judgment_columns = (
    required_judgment_columns
    - set(pilot_judgments_df.columns)
)

if missing_judgment_columns:
    raise RuntimeError(
        "pilot_judgments_df is missing required columns: "
        f"{sorted(missing_judgment_columns)}"
    )


if "query_id" not in pilot_queries.columns:
    raise RuntimeError(
        "pilot_queries must contain a query_id column."
    )


completed_before_calibration = int(
    pilot_judgments_df[
        "relevance"
    ].notna().sum()
)

if completed_before_calibration != 20:
    raise AssertionError(
        "This calibration expects exactly 20 existing "
        "human labels, but found "
        f"{completed_before_calibration}."
    )


# ------------------------------------------------------------
# 4. Define separate Groq output paths
# ------------------------------------------------------------

CALIBRATION_DIR = Path(
    CHECKPOINT_DIR
)

CALIBRATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


GROQ_CALIBRATION_CHECKPOINT_PATH = (
    CALIBRATION_DIR
    / "groq_judge_calibration_20_checkpoint.parquet"
)

GROQ_CALIBRATION_PATH = (
    CALIBRATION_DIR
    / "groq_judge_calibration_20.parquet"
)

GROQ_CALIBRATION_CSV_PATH = (
    CALIBRATION_DIR
    / "groq_judge_calibration_20.csv"
)


# ------------------------------------------------------------
# 5. General data helpers
# ------------------------------------------------------------

def first_present_value(
    row,
    candidate_columns,
    *,
    default="Not provided",
):
    """
    Return the first nonempty value found among the supplied columns.
    """

    for column in candidate_columns:
        if column not in row.index:
            continue

        value = row[column]

        if pd.isna(value):
            continue

        value_text = str(value).strip()

        if value_text:
            return value_text

    return default


def get_query_record(query_id):
    """
    Return exactly one pilot-query record for a query ID.
    """

    matches = pilot_queries.loc[
        pilot_queries["query_id"]
        .astype(str)
        .eq(str(query_id))
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one pilot query for "
            f"{query_id!r}, but found {len(matches)}."
        )

    return matches.iloc[0]


def candidate_payload(candidate_row):
    """
    Convert one candidate row into the payload sent to Groq.

    Human relevance labels and retrieval ranks are excluded.
    """

    candidate_text = first_present_value(
        candidate_row,
        [
            "text",
            "chunk_text",
            "content",
        ],
        default="",
    )

    if not candidate_text:
        raise ValueError(
            "Candidate "
            f"corpus_row={candidate_row['corpus_row']} "
            "has no passage text."
        )

    return {
        "corpus_row": int(
            candidate_row["corpus_row"]
        ),
        "ticker": first_present_value(
            candidate_row,
            [
                "ticker",
                "candidate_ticker",
            ],
        ),
        "form_type": first_present_value(
            candidate_row,
            [
                "form_type",
                "candidate_form_type",
                "form",
            ],
        ),
        "filing_date_or_year": first_present_value(
            candidate_row,
            [
                "filing_date",
                "candidate_filing_date",
                "filing_year",
            ],
        ),
        "passage": candidate_text,
    }


# ------------------------------------------------------------
# 6. Frozen blinded relevance-judgment prompt
# ------------------------------------------------------------

JUDGE_SYSTEM_PROMPT = """
You are a blinded relevance judge for an information-retrieval benchmark
built from SEC filings.

Judge each supplied candidate passage independently.

FROZEN RELEVANCE SCALE

2 = DIRECTLY RELEVANT
The passage directly answers the question or contains specific evidence
required to construct the answer.

1 = PARTIALLY RELEVANT
The passage provides useful supporting context, a related fact, or a
material part of the answer, but cannot adequately answer the question
by itself.

0 = IRRELEVANT
The passage does not help answer the question, even if it discusses the
same company or a broadly similar topic.

MANDATORY DECISION RULES

1. Judge relevance to the exact question, not keyword similarity.
2. Wrong ticker means relevance 0.
3. Wrong filing form means relevance 0 when the question is tied to a
   specific filing.
4. Wrong filing date or year means relevance 0 when the question asks
   about a specific filing period.
5. Do not reward a passage merely because it comes from the correct
   company.
6. A passage about the correct broad topic but lacking the requested
   fact is normally 0 or 1, not 2.
7. Tables or fragmented text may receive 2 if they clearly contain the
   requested evidence.
8. Use only the supplied information. Do not use outside knowledge.
9. Return exactly one judgment for every supplied corpus_row.
10. Do not compare candidates with one another.
11. Do not infer relevance from candidate order.
12. Give each candidate its own concise, evidence-based rationale.
""".strip()


def build_batch_user_prompt(
    query_row,
    candidate_rows,
):
    """
    Build one blinded prompt containing a query and up to five candidates.
    """

    query_payload = {
        "query_id": str(
            query_row["query_id"]
        ),
        "question": first_present_value(
            query_row,
            [
                "query",
                "question",
                "query_text",
            ],
        ),
        "target_ticker": first_present_value(
            query_row,
            [
                "ticker",
                "target_ticker",
            ],
        ),
        "target_form": first_present_value(
            query_row,
            [
                "form_type",
                "target_form_type",
                "form",
            ],
        ),
        "target_filing_date_or_year": first_present_value(
            query_row,
            [
                "filing_date",
                "target_filing_date",
                "filing_year",
                "target_year",
            ],
        ),
    }

    candidates = [
        candidate_payload(row)
        for _, row in candidate_rows.iterrows()
    ]

    request_payload = {
        "query": query_payload,
        "candidates": candidates,
    }

    return (
        "Judge every candidate contained in this JSON payload. "
        "Human labels and retrieval ranks are intentionally absent."
        "\n\n"
        + json.dumps(
            request_payload,
            ensure_ascii=False,
        )
    )


# ------------------------------------------------------------
# 7. Strict structured-output schema
# ------------------------------------------------------------

def response_schema_for_batch(
    expected_count,
):
    """
    Construct a strict JSON schema matching the current batch size.
    """

    return {
        "name": "financerag_relevance_batch",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "judgments": {
                    "type": "array",
                    "minItems": expected_count,
                    "maxItems": expected_count,
                    "items": {
                        "type": "object",
                        "properties": {
                            "corpus_row": {
                                "type": "integer",
                            },
                            "relevance": {
                                "type": "integer",
                                "enum": [0, 1, 2],
                            },
                            "confidence": {
                                "type": "string",
                                "enum": [
                                    "high",
                                    "medium",
                                    "low",
                                ],
                            },
                            "rationale": {
                                "type": "string",
                                "minLength": 1,
                                "maxLength": 500,
                            },
                        },
                        "required": [
                            "corpus_row",
                            "relevance",
                            "confidence",
                            "rationale",
                        ],
                        "additionalProperties": False,
                    },
                },
            },
            "required": [
                "judgments",
            ],
            "additionalProperties": False,
        },
    }


# ------------------------------------------------------------
# 8. Rate-limit and retry helpers
# ------------------------------------------------------------

def extract_retry_after_seconds(
    exception,
    attempt,
):
    """
    Honor Groq's retry-after information when available.

    Otherwise use bounded exponential backoff.
    """

    response = getattr(
        exception,
        "response",
        None,
    )

    headers = getattr(
        response,
        "headers",
        None,
    )

    if headers:
        retry_after = headers.get(
            "retry-after"
        )

        if retry_after:
            try:
                return max(
                    float(retry_after) + 1.0,
                    1.0,
                )
            except ValueError:
                pass

    error_text = str(exception)

    retry_patterns = [
        r"retry[- ]after[^\d]*([\d.]+)",
        r"try again in\s*([\d.]+)s",
        r"Please retry in\s*([\d.]+)s",
    ]

    for pattern in retry_patterns:
        match = re.search(
            pattern,
            error_text,
            flags=re.IGNORECASE,
        )

        if match:
            return (
                float(match.group(1))
                + 1.0
            )

    return min(
        2 ** (attempt - 1),
        60,
    )


def validate_batch_response(
    payload,
    expected_rows,
):
    """
    Validate that Groq returned exactly one result per requested row.
    """

    if (
        not isinstance(payload, dict)
        or not isinstance(
            payload.get("judgments"),
            list,
        )
    ):
        raise ValueError(
            "Groq response lacks a judgments list."
        )

    judgments = payload["judgments"]

    returned_rows = [
        int(item["corpus_row"])
        for item in judgments
    ]

    if len(returned_rows) != len(
        set(returned_rows)
    ):
        raise ValueError(
            "Groq returned duplicate corpus_row values."
        )

    if set(returned_rows) != set(
        expected_rows
    ):
        raise ValueError(
            "Groq returned the wrong candidate IDs. "
            f"Expected {sorted(expected_rows)}; "
            f"received {sorted(returned_rows)}."
        )

    for item in judgments:
        relevance = int(
            item["relevance"]
        )

        confidence = str(
            item["confidence"]
        )

        rationale = str(
            item["rationale"]
        ).strip()

        if relevance not in {
            0,
            1,
            2,
        }:
            raise ValueError(
                "Groq returned an invalid relevance label."
            )

        if confidence not in {
            "high",
            "medium",
            "low",
        }:
            raise ValueError(
                "Groq returned an invalid confidence value."
            )

        if not rationale:
            raise ValueError(
                "Groq returned an empty rationale."
            )

    return sorted(
        judgments,
        key=lambda item: int(
            item["corpus_row"]
        ),
    )


# ------------------------------------------------------------
# 9. Submit one batch to Groq
# ------------------------------------------------------------

def judge_one_batch(
    query_row,
    candidate_rows,
):
    """
    Request one blinded batch judgment with validation and retries.
    """

    expected_rows = (
        candidate_rows["corpus_row"]
        .astype(int)
        .tolist()
    )

    user_prompt = build_batch_user_prompt(
        query_row,
        candidate_rows,
    )

    output_schema = response_schema_for_batch(
        len(expected_rows)
    )

    last_exception = None

    for attempt in range(
        1,
        GROQ_MAX_ATTEMPTS_PER_BATCH + 1,
    ):
        try:
            completion = (
                groq_client
                .chat
                .completions
                .create(
                    model=GROQ_JUDGE_MODEL,
                    messages=[
                        {
                            "role": "system",
                            "content": JUDGE_SYSTEM_PROMPT,
                        },
                        {
                            "role": "user",
                            "content": user_prompt,
                        },
                    ],
                    temperature=0,
                    max_completion_tokens=(
                        GROQ_MAX_OUTPUT_TOKENS
                    ),
                    stream=False,
                    include_reasoning=False,
                    response_format={
                        "type": "json_schema",
                        "json_schema": output_schema,
                    },
                )
            )

            response_text = (
                completion
                .choices[0]
                .message
                .content
            )

            if not response_text:
                raise RuntimeError(
                    "Groq returned an empty response."
                )

            response_payload = json.loads(
                response_text
            )

            judgments = validate_batch_response(
                response_payload,
                expected_rows,
            )

            usage = getattr(
                completion,
                "usage",
                None,
            )

            prompt_tokens = getattr(
                usage,
                "prompt_tokens",
                np.nan,
            )

            completion_tokens = getattr(
                usage,
                "completion_tokens",
                np.nan,
            )

            return (
                judgments,
                attempt,
                prompt_tokens,
                completion_tokens,
            )

        except Exception as exc:
            last_exception = exc

            if (
                attempt
                == GROQ_MAX_ATTEMPTS_PER_BATCH
            ):
                break

            wait_seconds = (
                extract_retry_after_seconds(
                    exc,
                    attempt,
                )
            )

            print(
                f"         Batch attempt {attempt} failed "
                f"({type(exc).__name__}); waiting "
                f"{wait_seconds:.1f} seconds."
            )

            time.sleep(
                wait_seconds
            )

    raise RuntimeError(
        "Groq failed to return a valid batch after "
        f"{GROQ_MAX_ATTEMPTS_PER_BATCH} attempts."
    ) from last_exception


# ------------------------------------------------------------
# 10. Freeze the 20 human-labeled calibration pairs
# ------------------------------------------------------------

calibration_source = (
    pilot_judgments_df.loc[
        pilot_judgments_df[
            "relevance"
        ].notna()
    ]
    .copy()
    .sort_values(
        [
            "query_id",
            "corpus_row",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


if len(calibration_source) != 20:
    raise AssertionError(
        "Expected exactly 20 human-labeled "
        "calibration pairs, but found "
        f"{len(calibration_source)}."
    )


if not (
    calibration_source["query_id"]
    .astype(str)
    .eq("P01")
    .all()
):
    raise AssertionError(
        "The initial calibration set must contain "
        "only P01 candidates."
    )


if not (
    calibration_source["relevance"]
    .astype(int)
    .isin([0, 1, 2])
    .all()
):
    raise ValueError(
        "At least one human calibration label "
        "is outside {0, 1, 2}."
    )


if calibration_source[
    [
        "query_id",
        "corpus_row",
    ]
].duplicated().any():
    raise ValueError(
        "The calibration source contains "
        "duplicate query/corpus pairs."
    )


# ------------------------------------------------------------
# 11. Load an existing compatible Groq checkpoint
# ------------------------------------------------------------

checkpoint_columns = [
    "query_id",
    "corpus_row",
    "chunk_id",
    "human_relevance",
    "groq_relevance",
    "groq_confidence",
    "groq_rationale",
    "judge_model",
    "prompt_version",
    "batch_id",
    "batch_size",
    "attempts_used",
    "prompt_tokens",
    "completion_tokens",
    "judged_at_utc",
]


if GROQ_CALIBRATION_CHECKPOINT_PATH.exists():
    calibration_records_df = pd.read_parquet(
        GROQ_CALIBRATION_CHECKPOINT_PATH
    )

    missing_checkpoint_columns = (
        set(checkpoint_columns)
        - set(calibration_records_df.columns)
    )

    if missing_checkpoint_columns:
        raise RuntimeError(
            "The existing Groq checkpoint has an "
            "incompatible schema. Missing columns: "
            f"{sorted(missing_checkpoint_columns)}"
        )

    calibration_records_df = (
        calibration_records_df[
            checkpoint_columns
        ]
        .copy()
        .drop_duplicates(
            subset=[
                "query_id",
                "corpus_row",
            ],
            keep="last",
        )
        .reset_index(
            drop=True
        )
    )

    incompatible_checkpoint_rows = (
        calibration_records_df.loc[
            calibration_records_df[
                "judge_model"
            ]
            .astype(str)
            .ne(GROQ_JUDGE_MODEL)
            |
            calibration_records_df[
                "prompt_version"
            ]
            .astype(str)
            .ne(GROQ_PROMPT_VERSION)
        ]
    )

    if not incompatible_checkpoint_rows.empty:
        raise RuntimeError(
            "The existing Groq checkpoint uses another "
            "judge model or prompt version. Do not mix "
            "those results with this calibration."
        )

else:
    calibration_records_df = pd.DataFrame(
        columns=checkpoint_columns
    )


source_keys = {
    (
        str(row.query_id),
        int(row.corpus_row),
    )
    for row in calibration_source.itertuples(
        index=False
    )
}


completed_keys = {
    (
        str(row.query_id),
        int(row.corpus_row),
    )
    for row in calibration_records_df.itertuples(
        index=False
    )
}


unexpected_checkpoint_keys = (
    completed_keys
    - source_keys
)

if unexpected_checkpoint_keys:
    raise RuntimeError(
        "The Groq checkpoint contains pairs outside "
        "this frozen calibration set: "
        f"{sorted(unexpected_checkpoint_keys)}"
    )


# ------------------------------------------------------------
# 12. Construct remaining same-query batches
# ------------------------------------------------------------

remaining_mask = [
    (
        str(row.query_id),
        int(row.corpus_row),
    )
    not in completed_keys
    for row in calibration_source.itertuples(
        index=False
    )
]


remaining_source = (
    calibration_source.loc[
        remaining_mask
    ]
    .copy()
)


batches = []

for query_id, query_group in (
    remaining_source.groupby(
        "query_id",
        sort=False,
    )
):
    query_group = (
        query_group
        .reset_index(drop=True)
    )

    for start in range(
        0,
        len(query_group),
        GROQ_CALIBRATION_BATCH_SIZE,
    ):
        batch_df = query_group.iloc[
            start:
            start + GROQ_CALIBRATION_BATCH_SIZE
        ].copy()

        batches.append(
            (
                str(query_id),
                batch_df,
            )
        )


# ------------------------------------------------------------
# 13. Print run summary
# ------------------------------------------------------------

print("=" * 72)
print("STARTING OR RESUMING BATCHED GROQ CALIBRATION")
print("=" * 72)
print(
    f"Judge model:                "
    f"{GROQ_JUDGE_MODEL}"
)
print(
    f"Prompt version:             "
    f"{GROQ_PROMPT_VERSION}"
)
print(
    f"Total calibration pairs:    "
    f"{len(calibration_source)}"
)
print(
    f"Recovered pairs:            "
    f"{len(completed_keys)}"
)
print(
    f"Remaining pairs:            "
    f"{len(remaining_source)}"
)
print(
    f"Remaining API requests:     "
    f"{len(batches)}"
)
print(
    f"Maximum candidates/request: "
    f"{GROQ_CALIBRATION_BATCH_SIZE}"
)
print("Human labels sent to Groq:  NO")
print("Retrieval ranks sent:       NO")
print("Benchmark rows modified:    NO")
print("=" * 72)


# ------------------------------------------------------------
# 14. Run or resume the batched calibration
# ------------------------------------------------------------

for batch_number, (
    query_id,
    batch_df,
) in enumerate(
    batches,
    start=1,
):
    query_row = get_query_record(
        query_id
    )

    batch_rows = (
        batch_df["corpus_row"]
        .astype(int)
        .tolist()
    )

    (
        judgments,
        attempts_used,
        prompt_tokens,
        completion_tokens,
    ) = judge_one_batch(
        query_row,
        batch_df,
    )

    judgment_by_row = {
        int(item["corpus_row"]): item
        for item in judgments
    }

    batch_id = (
        f"{query_id}_"
        + "_".join(
            str(row_id)
            for row_id in sorted(batch_rows)
        )
    )

    judged_at_utc = datetime.now(
        timezone.utc
    ).isoformat()

    new_records = []

    for _, candidate_row in (
        batch_df.iterrows()
    ):
        corpus_row = int(
            candidate_row["corpus_row"]
        )

        judgment = judgment_by_row[
            corpus_row
        ]

        new_records.append(
            {
                "query_id": query_id,
                "corpus_row": corpus_row,
                "chunk_id": str(
                    candidate_row["chunk_id"]
                ),
                "human_relevance": int(
                    candidate_row["relevance"]
                ),
                "groq_relevance": int(
                    judgment["relevance"]
                ),
                "groq_confidence": str(
                    judgment["confidence"]
                ),
                "groq_rationale": str(
                    judgment["rationale"]
                ).strip(),
                "judge_model": GROQ_JUDGE_MODEL,
                "prompt_version": (
                    GROQ_PROMPT_VERSION
                ),
                "batch_id": batch_id,
                "batch_size": int(
                    len(batch_df)
                ),
                "attempts_used": int(
                    attempts_used
                ),
                "prompt_tokens": prompt_tokens,
                "completion_tokens": (
                    completion_tokens
                ),
                "judged_at_utc": judged_at_utc,
            }
        )

    calibration_records_df = pd.concat(
        [
            calibration_records_df,
            pd.DataFrame(
                new_records
            ),
        ],
        ignore_index=True,
    )

    calibration_records_df = (
        calibration_records_df
        .drop_duplicates(
            subset=[
                "query_id",
                "corpus_row",
            ],
            keep="last",
        )
        .sort_values(
            [
                "query_id",
                "corpus_row",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    # Save immediately after every successful batch.
    calibration_records_df.to_parquet(
        GROQ_CALIBRATION_CHECKPOINT_PATH,
        index=False,
    )

    # Verify checkpoint persistence immediately.
    verified_checkpoint = pd.read_parquet(
        GROQ_CALIBRATION_CHECKPOINT_PATH
    )

    if len(verified_checkpoint) != len(
        calibration_records_df
    ):
        raise RuntimeError(
            "Groq checkpoint persistence "
            "verification failed."
        )

    labels_summary = ", ".join(
        (
            f"{int(item['corpus_row'])}:"
            f"{int(item['relevance'])}"
        )
        for item in judgments
    )

    print(
        f"[{batch_number:02d}/{len(batches):02d}] "
        f"{query_id} | "
        f"rows/labels={labels_summary} | "
        f"attempts={attempts_used} | "
        "checkpoint saved"
    )


# ------------------------------------------------------------
# 15. Verify calibration completeness
# ------------------------------------------------------------

groq_calibration_df = pd.read_parquet(
    GROQ_CALIBRATION_CHECKPOINT_PATH
)

groq_calibration_df = (
    groq_calibration_df
    .drop_duplicates(
        subset=[
            "query_id",
            "corpus_row",
        ],
        keep="last",
    )
    .sort_values(
        [
            "query_id",
            "corpus_row",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


final_keys = {
    (
        str(row.query_id),
        int(row.corpus_row),
    )
    for row in groq_calibration_df.itertuples(
        index=False
    )
}


if (
    len(groq_calibration_df) != 20
    or final_keys != source_keys
):
    missing_keys = (
        source_keys
        - final_keys
    )

    extra_keys = (
        final_keys
        - source_keys
    )

    raise AssertionError(
        "Groq calibration is incomplete.\n"
        f"Rows found: {len(groq_calibration_df)}\n"
        f"Missing pairs: {sorted(missing_keys)}\n"
        f"Unexpected pairs: {sorted(extra_keys)}"
    )


groq_calibration_df[
    "exact_match"
] = (
    groq_calibration_df[
        "human_relevance"
    ].astype(int)
    ==
    groq_calibration_df[
        "groq_relevance"
    ].astype(int)
)


# ------------------------------------------------------------
# 16. Save and verify final results
# ------------------------------------------------------------

groq_calibration_df.to_parquet(
    GROQ_CALIBRATION_PATH,
    index=False,
)

groq_calibration_df.to_csv(
    GROQ_CALIBRATION_CSV_PATH,
    index=False,
)


verified_final_parquet = pd.read_parquet(
    GROQ_CALIBRATION_PATH
)

verified_final_csv = pd.read_csv(
    GROQ_CALIBRATION_CSV_PATH
)


if len(verified_final_parquet) != 20:
    raise AssertionError(
        "Final Groq Parquet verification failed."
    )


if len(verified_final_csv) != 20:
    raise AssertionError(
        "Final Groq CSV verification failed."
    )


# ------------------------------------------------------------
# 17. Confirm original human labels remain unchanged
# ------------------------------------------------------------

completed_after_calibration = int(
    pilot_judgments_df[
        "relevance"
    ].notna().sum()
)

if (
    completed_after_calibration
    != completed_before_calibration
):
    raise AssertionError(
        "The Groq calibration unexpectedly changed "
        "the benchmark labels."
    )


# ------------------------------------------------------------
# 18. Calculate calibration statistics
# ------------------------------------------------------------

human_labels = (
    groq_calibration_df[
        "human_relevance"
    ]
    .astype(int)
)

groq_labels = (
    groq_calibration_df[
        "groq_relevance"
    ]
    .astype(int)
)


exact_agreement = accuracy_score(
    human_labels,
    groq_labels,
)


weighted_kappa = cohen_kappa_score(
    human_labels,
    groq_labels,
    weights="quadratic",
)


agreement_matrix = confusion_matrix(
    human_labels,
    groq_labels,
    labels=[
        0,
        1,
        2,
    ],
)


agreement_matrix_df = pd.DataFrame(
    agreement_matrix,
    index=[
        "Human 0",
        "Human 1",
        "Human 2",
    ],
    columns=[
        "Groq 0",
        "Groq 1",
        "Groq 2",
    ],
)


disagreements_df = (
    groq_calibration_df.loc[
        ~groq_calibration_df[
            "exact_match"
        ]
    ]
    .copy()
    .sort_values(
        [
            "human_relevance",
            "groq_relevance",
            "corpus_row",
        ],
        kind="stable",
    )
)


unique_batch_usage = (
    groq_calibration_df
    .drop_duplicates(
        subset=[
            "batch_id",
        ]
    )
)


total_prompt_tokens = pd.to_numeric(
    unique_batch_usage[
        "prompt_tokens"
    ],
    errors="coerce",
).sum()


total_completion_tokens = pd.to_numeric(
    unique_batch_usage[
        "completion_tokens"
    ],
    errors="coerce",
).sum()


# ------------------------------------------------------------
# 19. Print the final report
# ------------------------------------------------------------

print()
print("=" * 72)
print("BATCHED GROQ CALIBRATION COMPLETE")
print("=" * 72)

print(
    f"Calibration pairs:             "
    f"{len(groq_calibration_df)}"
)

print(
    "Exact matches:                "
    f"{int(groq_calibration_df['exact_match'].sum())}/20"
)

print(
    "Exact agreement:              "
    f"{100 * exact_agreement:.1f}%"
)

if np.isnan(weighted_kappa):
    print(
        "Quadratic weighted kappa:    "
        "undefined "
        "(insufficient label variation)"
    )
else:
    print(
        "Quadratic weighted kappa:    "
        f"{weighted_kappa:.3f}"
    )

print(
    "High-confidence judgments:    "
    f"{int(groq_calibration_df['groq_confidence'].eq('high').sum())}/20"
)

print(
    "Disagreements requiring review: "
    f"{len(disagreements_df)}"
)

print(
    "Benchmark labels before/after: "
    f"{completed_before_calibration}/"
    f"{completed_after_calibration}"
)

print(
    "API batches represented:       "
    f"{groq_calibration_df['batch_id'].nunique()}"
)

print(
    "Prompt tokens (API reported):  "
    f"{int(total_prompt_tokens)}"
)

print(
    "Output tokens (API reported):  "
    f"{int(total_completion_tokens)}"
)

print("Human labels sent to Groq:     NO")
print("Retrieval ranks sent:          NO")
print("Benchmark rows modified:       NO")
print("Per-batch checkpointing:       YES")
print("Final persistence verified:    YES")

print(
    f"Final Parquet: "
    f"{GROQ_CALIBRATION_PATH}"
)

print(
    f"Final CSV:     "
    f"{GROQ_CALIBRATION_CSV_PATH}"
)

print("=" * 72)


print()
print("CONFUSION MATRIX")

display(
    agreement_matrix_df
)


print()
print("PAIR-LEVEL RESULTS")

display(
    groq_calibration_df[
        [
            "query_id",
            "corpus_row",
            "human_relevance",
            "groq_relevance",
            "groq_confidence",
            "exact_match",
            "batch_id",
            "groq_rationale",
        ]
    ]
)


print()
print("DISAGREEMENTS ONLY")

if disagreements_df.empty:
    print(
        "No disagreements."
    )
else:
    display(
        disagreements_df[
            [
                "query_id",
                "corpus_row",
                "human_relevance",
                "groq_relevance",
                "groq_confidence",
                "groq_rationale",
            ]
        ]
    )

STARTING OR RESUMING BATCHED GROQ CALIBRATION
Judge model:                openai/gpt-oss-120b
Prompt version:             financerag_groq_judge_v1_2026-08-05
Total calibration pairs:    20
Recovered pairs:            0
Remaining pairs:            20
Remaining API requests:     4
Maximum candidates/request: 5
Human labels sent to Groq:  NO
Retrieval ranks sent:       NO
Benchmark rows modified:    NO
[01/04] P01 | rows/labels=27:0, 42:0, 54:0, 56:0, 57:0 | attempts=1 | checkpoint saved
[02/04] P01 | rows/labels=75:0, 170:0, 182:0, 185:0, 186:0 | attempts=1 | checkpoint saved
[03/04] P01 | rows/labels=203:0, 257:0, 259:0, 260:0, 261:0 | attempts=1 | checkpoint saved
[04/04] P01 | rows/labels=262:2, 263:1, 265:1, 266:2, 267:0 | attempts=1 | checkpoint saved

BATCHED GROQ CALIBRATION COMPLETE
Calibration pairs:             20
Exact matches:                16/20
Exact agreement:              80.0%
Quadratic weighted kappa:    0.327
High-confidence judgments:    18/20
Disagreements requiring

,Groq 0,Groq 1,Groq 2
Human 0,16,2,1
Human 1,0,0,1
Human 2,0,0,0



PAIR-LEVEL RESULTS


,query_id,corpus_row,human_relevance,groq_relevance,groq_confidence,exact_match,batch_id,groq_rationale
0,P01,27,0,0,high,True,P01_27_42_54_56_57,Passage is from Apple 2023 Form 10-K; the ques...
1,P01,42,0,0,high,True,P01_27_42_54_56_57,Passage is from Apple 2023 Form 10-K; the ques...
2,P01,54,0,0,high,True,P01_27_42_54_56_57,"Passage originates from the 2023 Form 10-K, no..."
3,P01,56,0,0,high,True,P01_27_42_54_56_57,The passage is from Apple’s 2023 Form 10-K; th...
4,P01,57,0,0,high,True,P01_27_42_54_56_57,"This passage is from the 2023 Form 10-K, where..."
5,P01,75,0,0,high,True,P01_75_170_182_185_186,Passage is from the 2023 10-K; the question as...
6,P01,170,0,0,high,True,P01_75_170_182_185_186,Passage is from the 2024 10-K; the required in...
7,P01,182,0,0,high,True,P01_75_170_182_185_186,Passage is from the 2024 10-K and discusses pr...
8,P01,185,0,0,high,True,P01_75_170_182_185_186,Passage is from the 2024 10-K; the question sp...
9,P01,186,0,0,high,True,P01_75_170_182_185_186,Passage is from the 2024 10-K and does not per...



DISAGREEMENTS ONLY


,query_id,corpus_row,human_relevance,groq_relevance,groq_confidence,groq_rationale
16,P01,263,0,1,medium,Describes competitive pressures and pricing bu...
17,P01,265,0,1,medium,Mentions component‑supply constraints and R&D ...
15,P01,262,0,2,high,Passage lists distribution‑channel mix (40% di...
18,P01,266,1,2,high,Explicitly states seasonality and new product/...


In [ ]:
from pathlib import Path
import pandas as pd

files_to_verify = [
    Path("/content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/groq_judge_calibration_20_checkpoint.parquet"),
    Path("/content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/groq_judge_calibration_20.parquet"),
    Path("/content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/groq_judge_calibration_20.csv"),
]

for path in files_to_verify:
    assert path.exists(), f"MISSING: {path}"
    print(f"SAVED: {path.name} | {path.stat().st_size:,} bytes")

df = pd.read_parquet(files_to_verify[1])

assert len(df) == 20
assert df[["query_id", "corpus_row"]].duplicated().sum() == 0
assert df["groq_relevance"].notna().all()

print()
print("Verified calibration rows:", len(df))
print("Unique judged pairs:", len(df[["query_id", "corpus_row"]].drop_duplicates()))
print("All Groq labels present:", df["groq_relevance"].notna().all())
print("SAFE TO DISCONNECT")

SAVED: groq_judge_calibration_20_checkpoint.parquet | 12,517 bytes
SAVED: groq_judge_calibration_20.parquet | 12,965 bytes
SAVED: groq_judge_calibration_20.csv | 7,056 bytes

Verified calibration rows: 20
Unique judged pairs: 20
All Groq labels present: True
SAFE TO DISCONNECT


In [ ]:
# ============================================================
# REVIEW P01 GROQ–HUMAN DISAGREEMENTS WITHOUT CHANGING LABELS
# ============================================================

from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


# ------------------------------------------------------------
# 1. Load the saved blinded Groq calibration
# ------------------------------------------------------------

GROQ_CALIBRATION_PATH = (
    Path(CHECKPOINT_DIR)
    / "groq_judge_calibration_20.parquet"
)

if not GROQ_CALIBRATION_PATH.exists():
    raise FileNotFoundError(
        f"Saved calibration not found: {GROQ_CALIBRATION_PATH}"
    )

groq_calibration_df = pd.read_parquet(
    GROQ_CALIBRATION_PATH
)


# ------------------------------------------------------------
# 2. Validate the required in-memory benchmark objects
# ------------------------------------------------------------

required_objects = [
    "pilot_queries",
    "pilot_judgments_df",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Rerun the earlier Notebook 3 setup and retrieval cells first. "
        f"Missing objects: {missing_objects}"
    )


required_columns = {
    "query_id",
    "corpus_row",
    "chunk_id",
    "relevance",
}

missing_columns = (
    required_columns
    - set(pilot_judgments_df.columns)
)

if missing_columns:
    raise RuntimeError(
        "pilot_judgments_df is missing required columns: "
        f"{sorted(missing_columns)}"
    )


# ------------------------------------------------------------
# 3. Helpers for schema-tolerant display
# ------------------------------------------------------------

def first_available_column(dataframe, candidates):
    for column in candidates:
        if column in dataframe.columns:
            return column
    return None


query_text_column = first_available_column(
    pilot_queries,
    [
        "query",
        "question",
        "query_text",
    ],
)

passage_text_column = first_available_column(
    pilot_judgments_df,
    [
        "text",
        "chunk_text",
        "content",
        "passage",
    ],
)

if query_text_column is None:
    raise RuntimeError(
        "Could not identify the query-text column in pilot_queries."
    )

if passage_text_column is None:
    raise RuntimeError(
        "Could not identify the passage-text column in "
        "pilot_judgments_df."
    )


# ------------------------------------------------------------
# 4. Isolate genuine disagreements
# ------------------------------------------------------------

disagreement_labels = (
    groq_calibration_df.loc[
        groq_calibration_df["human_relevance"].astype(int)
        != groq_calibration_df["groq_relevance"].astype(int)
    ]
    .copy()
)

review_df = disagreement_labels.merge(
    pilot_judgments_df,
    on=[
        "query_id",
        "corpus_row",
        "chunk_id",
    ],
    how="left",
    validate="one_to_one",
    suffixes=(
        "_groq_file",
        "_human_table",
    ),
)

if len(review_df) != len(disagreement_labels):
    raise AssertionError(
        "The disagreement rows did not merge cleanly with "
        "pilot_judgments_df."
    )

if review_df[passage_text_column].isna().any():
    missing_rows = review_df.loc[
        review_df[passage_text_column].isna(),
        "corpus_row",
    ].tolist()

    raise RuntimeError(
        "Passage text is missing after the merge for corpus rows: "
        f"{missing_rows}"
    )


# ------------------------------------------------------------
# 5. Print the frozen rubric used during manual review
# ------------------------------------------------------------

display(
    Markdown(
        """
### Frozen manual relevance rubric

- **2 — Directly relevant:** Contains the requested answer or essential
  evidence and can serve as the primary citation.
- **1 — Partially relevant:** Provides materially useful supporting
  evidence, but cannot answer the question adequately by itself.
- **0 — Irrelevant:** Does not materially help answer the exact question.

Apply exact ticker, filing, and period constraints. Do not reward broad
topic overlap. This review cell does **not** alter any labels.
"""
    )
)


# ------------------------------------------------------------
# 6. Display the full question and every disagreement separately
# ------------------------------------------------------------

for query_id, query_group in review_df.groupby(
    "query_id",
    sort=False,
):
    query_matches = pilot_queries.loc[
        pilot_queries["query_id"].astype(str).eq(
            str(query_id)
        )
    ]

    if len(query_matches) != 1:
        raise ValueError(
            f"Expected one query record for {query_id}, "
            f"found {len(query_matches)}."
        )

    query_row = query_matches.iloc[0]

    display(
        Markdown(
            f"## Query {query_id}\n\n"
            f"**Question:** {query_row[query_text_column]}"
        )
    )

    for _, row in query_group.sort_values(
        "corpus_row"
    ).iterrows():
        metadata_lines = []

        for column in [
            "ticker",
            "form_type",
            "filing_date",
            "filing_year",
        ]:
            if (
                column in row.index
                and pd.notna(row[column])
            ):
                metadata_lines.append(
                    f"- **{column}:** {row[column]}"
                )

        metadata_text = (
            "\n".join(metadata_lines)
            if metadata_lines
            else "- Metadata not available in the current table"
        )

        display(
            Markdown(
                f"### Corpus row {int(row['corpus_row'])}\n\n"
                f"- **Current human label:** "
                f"{int(row['human_relevance'])}\n"
                f"- **Groq label:** "
                f"{int(row['groq_relevance'])}\n"
                f"- **Groq confidence:** "
                f"{row['groq_confidence']}\n"
                f"{metadata_text}\n\n"
                f"**Groq rationale:**\n\n"
                f"{row['groq_rationale']}\n\n"
                f"**Full passage:**\n\n"
                f"{row[passage_text_column]}"
            )
        )

        print("\n" + "=" * 100 + "\n")


# ------------------------------------------------------------
# 7. Create a blank decision sheet; do not update ground truth
# ------------------------------------------------------------

manual_review_sheet = review_df[
    [
        "query_id",
        "corpus_row",
        "chunk_id",
        "human_relevance",
        "groq_relevance",
        "groq_confidence",
    ]
].copy()

manual_review_sheet["reviewed_relevance"] = pd.NA
manual_review_sheet["review_reason"] = ""

display(
    Markdown(
        "### Blank review sheet\n"
        "We will fill this only after reading each complete passage."
    )
)

display(manual_review_sheet)

print(
    f"Ready for manual review: {len(manual_review_sheet)} disagreements."
)
print("No benchmark labels were modified.")


### Frozen manual relevance rubric

- **2 — Directly relevant:** Contains the requested answer or essential
  evidence and can serve as the primary citation.
- **1 — Partially relevant:** Provides materially useful supporting
  evidence, but cannot answer the question adequately by itself.
- **0 — Irrelevant:** Does not materially help answer the exact question.

Apply exact ticker, filing, and period constraints. Do not reward broad
topic overlap. This review cell does **not** alter any labels.


## Query P01

**Question:** What factors drove changes in Apple's product and services net sales in its 2025 annual filing?

### Corpus row 262

- **Current human label:** 0
- **Groq label:** 2
- **Groq confidence:** high
- **ticker:** AAPL
- **form_type:** 10-K
- **filing_date:** 2025-10-31

**Groq rationale:**

Passage lists distribution‑channel mix (40% direct, 60% indirect) and competition/innovation as factors influencing net sales, directly addressing drivers of product and services net‑sales changes.

**Full passage:**

Rest of Asia Pacific includes Australia, New Zealand and those Asian countries not included in the Company’s other reportable segments. Although the reportable segments provide similar hardware and software products and similar services, each one is managed separately to better align with the location of the Company’s customers and distribution partners and the unique market dynamics of each geographic region. Markets and Distribution The Company’s customers are primarily in the consumer, small and mid-sized business, education, enterprise and government markets. The Company sells its products and resells third-party products in most of its major markets directly to customers through its retail and online stores and its direct sales force. The Company sells its services in the same markets through its various service platforms. The Company also employs a variety of indirect distribution channels, such as third-party cellular network carriers and other resellers, for the sale of its products and certain of its services. During 2025, the Company’s net sales through its direct and indirect distribution channels accounted for 40% and 60%, respectively, of total net sales. Competition The markets for the Company’s products and services are highly competitive and are characterized by aggressive price competition, downward pressure on gross margins, continual improvement in product performance, and price sensitivity on the part of consumers and businesses. The markets in which the Company competes are further defined by frequent introduction of new products and services, short product life cycles, evolving industry standards, and rapid adoption of technological advancements by competitors. Many of the Company’s competitors seek to compete primarily through aggressive pricing and very low cost structures, and by imitating the Company’s products and infringing on its intellectual property. Apple Inc. | 2025 Form 10-K | 2 The Company’s ability to compete successfully depends heavily on ensuring the continuing and timely introduction of innovative new products, services and technologies to the marketplace. The Company designs and develops nearly the entire solution for its products, including the hardware, operating system, numerous software applications and related services.

### Corpus row 263

- **Current human label:** 0
- **Groq label:** 1
- **Groq confidence:** medium
- **ticker:** AAPL
- **form_type:** 10-K
- **filing_date:** 2025-10-31

**Groq rationale:**

Describes competitive pressures and pricing but does not explicitly connect them to net‑sales changes, offering supporting context only.

**Full passage:**

Many of the Company’s competitors seek to compete primarily through aggressive pricing and very low cost structures, and by imitating the Company’s products and infringing on its intellectual property. Apple Inc. | 2025 Form 10-K | 2 The Company’s ability to compete successfully depends heavily on ensuring the continuing and timely introduction of innovative new products, services and technologies to the marketplace. The Company designs and develops nearly the entire solution for its products, including the hardware, operating system, numerous software applications and related services. Principal competitive factors important to the Company include price, product and service features (including security features), relative price and performance, product and service quality and reliability, design and technology innovation, a strong third-party software and accessories ecosystem, marketing and distribution capability, service and support, corporate reputation, and the ability to effectively protect and enforce the Company’s intellectual property rights. The Company is focused on expanding its market opportunities related to smartphones, personal computers, tablets, wearables and accessories, and services. The Company’s products and services face substantial competition from companies that have significant technical, marketing, distribution and other resources, as well as established hardware, software, and service offerings with large customer bases. In addition, the Company faces significant competition as competitors imitate the Company’s product features and applications within their products to offer more competitive solutions. The Company also expects competition to intensify as competitors imitate the Company’s approach to providing components seamlessly within their offerings or work collaboratively to offer integrated solutions. Some of the Company’s competitors have broad product lines, low-priced products, large installed bases of active devices, and large customer bases. Competition has been particularly intense as competitors have aggressively cut prices and lowered product margins. Certain competitors have the resources, experience or cost structures to provide products and services at little or no profit or even at a loss. The Company has a minority market share in the global smartphone, personal computer, tablet and wearables markets, and some of the markets in which the Company competes have from time to time experienced little to no growth or contracted overall. Supply of Components Although most components essential to the Company’s business are generally available from multiple sources, certain components are currently obtained from single or limited sources.

### Corpus row 265

- **Current human label:** 0
- **Groq label:** 1
- **Groq confidence:** medium
- **ticker:** AAPL
- **form_type:** 10-K
- **filing_date:** 2025-10-31

**Groq rationale:**

Mentions component‑supply constraints and R&D efforts that can affect product availability and sales, providing relevant background but not a direct statement of net‑sales drivers.

**Full passage:**

In addition, component suppliers may fail, be subject to consolidation within a particular industry, or decide to concentrate on the production of common components instead of components customized to meet the Company’s requirements, further limiting the Company’s ability to obtain sufficient quantities of components on commercially reasonable terms, or at all. Research and Development Because the industries in which the Company competes are characterized by rapid technological advances, the Company’s ability to compete successfully depends heavily upon its ability to ensure a continual and timely flow of competitive products, services and technologies to the marketplace. The Company continues to develop new technologies to enhance existing products and services, and to expand the range of its offerings through research and development (“R&D”), licensing of intellectual property and acquisition of third-party businesses and technology. Intellectual Property The Company currently holds a broad collection of intellectual property rights relating to certain aspects of its hardware, software and services. This includes patents, designs, copyrights, trademarks, trade secrets and other forms of intellectual property rights in the U.S. and various foreign countries. Although the Company believes the ownership of such intellectual property rights is an important factor in differentiating its business and that its success does depend in part on such ownership, the Company relies primarily on the innovative skills, technical competence and marketing abilities of its personnel. The Company regularly files patent, design, copyright and trademark applications to protect innovations arising from its hardware, software and service research, development, design and marketing, and is currently pursuing thousands of applications around the world. Over time, the Company has accumulated a large portfolio of issued and registered intellectual property rights around the world. No single intellectual property right is solely responsible for protecting the Company’s products and services. The Company believes the duration of its intellectual property rights is adequate relative to the expected lives of its products and services. Apple Inc. | 2025 Form 10-K | 3 In addition to Company-owned intellectual property, many of the Company’s products and services include technology or intellectual property that must be licensed from third parties. It may be necessary in the future to seek or renew licenses relating to various aspects of the Company’s products, processes and services. While the Company has generally been able to obtain such licenses on commercially reasonable terms in the past, there is no guarantee that such licenses could be obtained in the future on reasonable terms or at all.

### Corpus row 266

- **Current human label:** 1
- **Groq label:** 2
- **Groq confidence:** high
- **ticker:** AAPL
- **form_type:** 10-K
- **filing_date:** 2025-10-31

**Groq rationale:**

Explicitly states seasonality and new product/service introductions as significant factors that impact net sales, directly answering the question.

**Full passage:**

Apple Inc. | 2025 Form 10-K | 3 In addition to Company-owned intellectual property, many of the Company’s products and services include technology or intellectual property that must be licensed from third parties. It may be necessary in the future to seek or renew licenses relating to various aspects of the Company’s products, processes and services. While the Company has generally been able to obtain such licenses on commercially reasonable terms in the past, there is no guarantee that such licenses could be obtained in the future on reasonable terms or at all. Business Seasonality and Product Introductions The Company has historically experienced higher net sales in its first quarter compared to other quarters in its fiscal year due in part to seasonal holiday demand. Additionally, new product and service introductions can significantly impact net sales, cost of sales and operating expenses. The timing of product introductions can also impact the Company’s net sales to its indirect distribution channels as these channels are filled with new inventory following a product launch, and channel inventory of an older product often declines as the launch of a newer product approaches. Net sales can also be affected when consumers and distributors anticipate a product introduction. Human Capital The Company believes that its people play an important role in its success, and strives to attract, develop and retain the best talent. The Company works to create a culture of collaboration, one where people with a broad range of backgrounds and perspectives can come together to innovate and do the best work of their lives. The Company is an equal opportunity employer committed to inclusion and to providing a workplace free of harassment or discrimination. As of September 27, 2025, the Company had approximately 166,000 full-time equivalent employees. The Company believes that compensation should be competitive and equitable, and offers discretionary cash and equity awards to enable employees to share in the Company’s success. The Company recognizes its people are most likely to thrive when they have the resources to meet their needs and the time and support to succeed in their professional and personal lives. In support of this, the Company offers a wide variety of benefits for employees around the world, including health, wellness and time away. The Company invests in resources to help its people develop and achieve their career goals. The Company offers programs through Apple University on leadership, management and influence, as well as Apple culture and values.

### Blank review sheet
We will fill this only after reading each complete passage.

,query_id,corpus_row,chunk_id,human_relevance,groq_relevance,groq_confidence,reviewed_relevance,review_reason
0,P01,262,aapl_10k_000032019325000079__chunk_0008,0,2,high,<NA>,
1,P01,263,aapl_10k_000032019325000079__chunk_0009,0,1,medium,<NA>,
2,P01,265,aapl_10k_000032019325000079__chunk_0011,0,1,medium,<NA>,
3,P01,266,aapl_10k_000032019325000079__chunk_0012,1,2,high,<NA>,


Ready for manual review: 4 disagreements.
No benchmark labels were modified.


In [ ]:
# ============================================================
# FREEZE THE HELD-OUT EVALUATION DESIGN
# ============================================================
# This cell:
#   1. freezes the retrieval settings before evaluation,
#   2. defines the final held-out evaluation questions,
#   3. validates benchmark coverage,
#   4. saves both artifacts for reproducibility.
#
# It does NOT retrieve candidates or alter existing judgments.
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
import json

import pandas as pd
from IPython.display import display, Markdown


# ------------------------------------------------------------
# 1. Confirm required Notebook 3 objects exist
# ------------------------------------------------------------

required_objects = [
    "CHECKPOINT_DIR",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Rerun the earlier Notebook 3 setup cells first. "
        f"Missing objects: {missing_objects}"
    )


CHECKPOINT_DIR = Path(CHECKPOINT_DIR)

if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(
        f"Checkpoint directory does not exist: {CHECKPOINT_DIR}"
    )


# ------------------------------------------------------------
# 2. Freeze the retrieval configuration
# ------------------------------------------------------------

FROZEN_RETRIEVAL_CONFIG = {
    "config_name": "heldout_evaluation_v1",
    "dense_top_k": 30,
    "bm25_top_k": 20,
    "rrf_k": 30,
    "final_top_k": 10,
    "expand": True,
    "diversify": True,
    "evaluation_pool_depth": 10,
    "relevance_scale": {
        "0": "irrelevant",
        "1": "partially_relevant",
        "2": "directly_relevant",
    },
    "frozen_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}


FROZEN_CONFIG_PATH = (
    CHECKPOINT_DIR
    / "heldout_retrieval_config_v1.json"
)


# Refuse to silently overwrite a different frozen configuration.
if FROZEN_CONFIG_PATH.exists():
    with open(
        FROZEN_CONFIG_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        existing_config = json.load(file)

    comparable_keys = [
        "dense_top_k",
        "bm25_top_k",
        "rrf_k",
        "final_top_k",
        "expand",
        "diversify",
        "evaluation_pool_depth",
    ]

    configuration_changed = any(
        existing_config.get(key)
        != FROZEN_RETRIEVAL_CONFIG.get(key)
        for key in comparable_keys
    )

    if configuration_changed:
        raise RuntimeError(
            "A different held-out retrieval configuration "
            "is already frozen. Review it before proceeding:\n"
            f"{FROZEN_CONFIG_PATH}"
        )

    # Preserve the original freeze timestamp.
    FROZEN_RETRIEVAL_CONFIG = existing_config

else:
    with open(
        FROZEN_CONFIG_PATH,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            FROZEN_RETRIEVAL_CONFIG,
            file,
            indent=2,
        )


# ------------------------------------------------------------
# 3. Define the held-out evaluation questions
# ------------------------------------------------------------
# IMPORTANT:
# Replace only a question if its stated company/year is absent
# from your indexed corpus. Do not tune wording after inspecting
# retrieval results.
#
# Questions are deliberately explicit about company, filing year,
# and requested information to reduce relevance ambiguity.
# ------------------------------------------------------------

heldout_queries = pd.DataFrame(
    [
        {
            "query_id": "H01",
            "ticker": "AAPL",
            "filing_year": 2025,
            "question_type": "reported_driver",
            "difficulty": "medium",
            "query": (
                "According to Apple's 2025 Form 10-K, what "
                "factors did management identify as driving "
                "the reported year-over-year changes in "
                "Products and Services net sales?"
            ),
        },
        {
            "query_id": "H02",
            "ticker": "AAPL",
            "filing_year": 2024,
            "question_type": "geographic_or_segment_comparison",
            "difficulty": "medium",
            "query": (
                "According to Apple's 2024 Form 10-K, how did "
                "net sales change across its reportable "
                "geographic segments, and which segments "
                "experienced declines?"
            ),
        },
        {
            "query_id": "H03",
            "ticker": "MSFT",
            "filing_year": 2025,
            "question_type": "segment_driver",
            "difficulty": "medium",
            "query": (
                "According to Microsoft's 2025 Form 10-K, "
                "what were the principal drivers of revenue "
                "growth in the Intelligent Cloud segment?"
            ),
        },
        {
            "query_id": "H04",
            "ticker": "MSFT",
            "filing_year": 2024,
            "question_type": "capital_expenditure",
            "difficulty": "medium",
            "query": (
                "According to Microsoft's 2024 Form 10-K, "
                "what investments or business needs drove "
                "capital expenditures during the fiscal year?"
            ),
        },
        {
            "query_id": "H05",
            "ticker": "AMZN",
            "filing_year": 2024,
            "question_type": "segment_comparison",
            "difficulty": "medium",
            "query": (
                "According to Amazon's 2024 Form 10-K, how "
                "did AWS net sales and operating income change "
                "compared with the prior year?"
            ),
        },
        {
            "query_id": "H06",
            "ticker": "AMZN",
            "filing_year": 2023,
            "question_type": "cash_flow_driver",
            "difficulty": "hard",
            "query": (
                "According to Amazon's 2023 Form 10-K, what "
                "factors affected the year-over-year change "
                "in operating cash flow?"
            ),
        },
        {
            "query_id": "H07",
            "ticker": "GOOGL",
            "filing_year": 2024,
            "question_type": "revenue_driver",
            "difficulty": "medium",
            "query": (
                "According to Alphabet's 2024 Form 10-K, what "
                "factors drove the year-over-year change in "
                "Google advertising revenues?"
            ),
        },
        {
            "query_id": "H08",
            "ticker": "GOOGL",
            "filing_year": 2023,
            "question_type": "risk_factor",
            "difficulty": "hard",
            "query": (
                "According to Alphabet's 2023 Form 10-K, what "
                "risks could artificial intelligence products "
                "and services create for the company?"
            ),
        },
        {
            "query_id": "H09",
            "ticker": "NVDA",
            "filing_year": 2025,
            "question_type": "segment_driver",
            "difficulty": "medium",
            "query": (
                "According to NVIDIA's 2025 Form 10-K, what "
                "factors drove the year-over-year increase in "
                "Data Center revenue?"
            ),
        },
        {
            "query_id": "H10",
            "ticker": "NVDA",
            "filing_year": 2024,
            "question_type": "supply_risk",
            "difficulty": "hard",
            "query": (
                "According to NVIDIA's 2024 Form 10-K, what "
                "supply-chain and manufacturing dependencies "
                "could constrain its ability to meet demand?"
            ),
        },
        {
            "query_id": "H11",
            "ticker": "AAPL",
            "filing_year": 2023,
            "question_type": "numerical_change",
            "difficulty": "easy",
            "query": (
                "According to Apple's 2023 Form 10-K, how did "
                "research and development expense change from "
                "2022 to 2023?"
            ),
        },
        {
            "query_id": "H12",
            "ticker": "MSFT",
            "filing_year": 2023,
            "question_type": "risk_factor",
            "difficulty": "hard",
            "query": (
                "According to Microsoft's 2023 Form 10-K, what "
                "cybersecurity risks and potential business "
                "consequences did the company disclose?"
            ),
        },
    ]
)


# ------------------------------------------------------------
# 4. Validate the benchmark design
# ------------------------------------------------------------

required_query_columns = {
    "query_id",
    "ticker",
    "filing_year",
    "question_type",
    "difficulty",
    "query",
}

missing_query_columns = (
    required_query_columns
    - set(heldout_queries.columns)
)

if missing_query_columns:
    raise AssertionError(
        "Held-out query table is missing columns: "
        f"{sorted(missing_query_columns)}"
    )


if heldout_queries["query_id"].duplicated().any():
    duplicate_ids = heldout_queries.loc[
        heldout_queries["query_id"].duplicated(
            keep=False
        ),
        "query_id",
    ].tolist()

    raise AssertionError(
        f"Duplicate query IDs found: {duplicate_ids}"
    )


if heldout_queries["query"].duplicated().any():
    raise AssertionError(
        "Duplicate held-out question text found."
    )


if heldout_queries.isna().any().any():
    raise AssertionError(
        "Held-out query definitions contain missing values."
    )


if not heldout_queries["difficulty"].isin(
    [
        "easy",
        "medium",
        "hard",
    ]
).all():
    raise AssertionError(
        "Difficulty must be easy, medium, or hard."
    )


if len(heldout_queries) != 12:
    raise AssertionError(
        "Expected exactly 12 held-out questions."
    )


if heldout_queries["ticker"].nunique() != 5:
    raise AssertionError(
        "Expected coverage across exactly five companies."
    )


if heldout_queries["question_type"].nunique() < 6:
    raise AssertionError(
        "The benchmark needs at least six question types."
    )


# ------------------------------------------------------------
# 5. Check available corpus coverage when possible
# ------------------------------------------------------------

corpus_dataframe = None

for candidate_name in [
    "chunks_df",
    "corpus_df",
    "documents_df",
]:
    if candidate_name in globals():
        corpus_dataframe = globals()[candidate_name]
        break


coverage_check = None

if corpus_dataframe is not None:
    ticker_column = next(
        (
            column
            for column in [
                "ticker",
                "company_ticker",
                "symbol",
            ]
            if column in corpus_dataframe.columns
        ),
        None,
    )

    year_column = next(
        (
            column
            for column in [
                "filing_year",
                "year",
                "fiscal_year",
            ]
            if column in corpus_dataframe.columns
        ),
        None,
    )

    if (
        ticker_column is not None
        and year_column is not None
    ):
        available_pairs = set(
            zip(
                corpus_dataframe[
                    ticker_column
                ].astype(str).str.upper(),
                pd.to_numeric(
                    corpus_dataframe[
                        year_column
                    ],
                    errors="coerce",
                ).astype("Int64"),
            )
        )

        coverage_check = heldout_queries.copy()

        coverage_check[
            "corpus_pair_available"
        ] = [
            (
                str(row.ticker).upper(),
                int(row.filing_year),
            )
            in available_pairs
            for row in coverage_check.itertuples(
                index=False
            )
        ]

        unavailable_queries = coverage_check.loc[
            ~coverage_check[
                "corpus_pair_available"
            ],
            [
                "query_id",
                "ticker",
                "filing_year",
            ],
        ]

        if not unavailable_queries.empty:
            display(
                Markdown(
                    "### Corpus coverage problem\n\n"
                    "Do not retrieve yet. The following requested "
                    "company/year pairs were not detected:"
                )
            )

            display(unavailable_queries)

            raise RuntimeError(
                "Replace unavailable held-out questions with "
                "company/year pairs that exist in the corpus."
            )


# ------------------------------------------------------------
# 6. Save and verify the frozen held-out questions
# ------------------------------------------------------------

HELDOUT_QUERIES_PATH = (
    CHECKPOINT_DIR
    / "heldout_queries_v1.parquet"
)

HELDOUT_QUERIES_CSV_PATH = (
    CHECKPOINT_DIR
    / "heldout_queries_v1.csv"
)


if HELDOUT_QUERIES_PATH.exists():
    existing_queries = pd.read_parquet(
        HELDOUT_QUERIES_PATH
    )

    comparison_columns = [
        "query_id",
        "ticker",
        "filing_year",
        "question_type",
        "difficulty",
        "query",
    ]

    existing_comparison = (
        existing_queries[
            comparison_columns
        ]
        .sort_values(
            "query_id"
        )
        .reset_index(
            drop=True
        )
    )

    proposed_comparison = (
        heldout_queries[
            comparison_columns
        ]
        .sort_values(
            "query_id"
        )
        .reset_index(
            drop=True
        )
    )

    if not existing_comparison.equals(
        proposed_comparison
    ):
        raise RuntimeError(
            "A different held-out question set is already "
            "frozen. Do not overwrite it after evaluation "
            "has begun."
        )

    heldout_queries = existing_queries.copy()

else:
    heldout_queries.to_parquet(
        HELDOUT_QUERIES_PATH,
        index=False,
    )

    heldout_queries.to_csv(
        HELDOUT_QUERIES_CSV_PATH,
        index=False,
    )


verified_queries = pd.read_parquet(
    HELDOUT_QUERIES_PATH
)

if len(verified_queries) != 12:
    raise AssertionError(
        "Held-out question persistence verification failed."
    )


if (
    verified_queries["query_id"].nunique()
    != 12
):
    raise AssertionError(
        "Saved held-out query IDs are not unique."
    )


# ------------------------------------------------------------
# 7. Display the frozen design
# ------------------------------------------------------------

display(
    Markdown(
        "## Frozen held-out evaluation design\n\n"
        "The retrieval configuration and evaluation questions "
        "are now fixed. Do not revise them after inspecting "
        "retrieval results."
    )
)


print("FROZEN RETRIEVAL CONFIGURATION")

for key, value in (
    FROZEN_RETRIEVAL_CONFIG.items()
):
    print(f"{key}: {value}")


print()
print(
    "Held-out questions:",
    len(heldout_queries),
)
print(
    "Companies represented:",
    heldout_queries["ticker"].nunique(),
)
print(
    "Filing years represented:",
    heldout_queries["filing_year"].nunique(),
)
print(
    "Question types represented:",
    heldout_queries[
        "question_type"
    ].nunique(),
)
print(
    "Difficulties:",
    heldout_queries[
        "difficulty"
    ].value_counts().to_dict(),
)
print(
    "Configuration saved:",
    FROZEN_CONFIG_PATH,
)
print(
    "Questions saved:",
    HELDOUT_QUERIES_PATH,
)
print()
print(
    "READY FOR POOLED DENSE/BM25/HYBRID RETRIEVAL"
)


display(
    heldout_queries[
        [
            "query_id",
            "ticker",
            "filing_year",
            "question_type",
            "difficulty",
            "query",
        ]
    ]
)

## Frozen held-out evaluation design

The retrieval configuration and evaluation questions are now fixed. Do not revise them after inspecting retrieval results.

FROZEN RETRIEVAL CONFIGURATION
config_name: heldout_evaluation_v1
dense_top_k: 30
bm25_top_k: 20
rrf_k: 30
final_top_k: 10
expand: True
diversify: True
evaluation_pool_depth: 10
relevance_scale: {'0': 'irrelevant', '1': 'partially_relevant', '2': 'directly_relevant'}
frozen_at_utc: 2026-08-06T06:05:22.120295+00:00

Held-out questions: 12
Companies represented: 5
Filing years represented: 3
Question types represented: 10
Difficulties: {'medium': 7, 'hard': 4, 'easy': 1}
Configuration saved: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_config_v1.json
Questions saved: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_queries_v1.parquet

READY FOR POOLED DENSE/BM25/HYBRID RETRIEVAL


,query_id,ticker,filing_year,question_type,difficulty,query
0,H01,AAPL,2025,reported_driver,medium,"According to Apple's 2025 Form 10-K, what fact..."
1,H02,AAPL,2024,geographic_or_segment_comparison,medium,"According to Apple's 2024 Form 10-K, how did n..."
2,H03,MSFT,2025,segment_driver,medium,"According to Microsoft's 2025 Form 10-K, what ..."
3,H04,MSFT,2024,capital_expenditure,medium,"According to Microsoft's 2024 Form 10-K, what ..."
4,H05,AMZN,2024,segment_comparison,medium,"According to Amazon's 2024 Form 10-K, how did ..."
5,H06,AMZN,2023,cash_flow_driver,hard,"According to Amazon's 2023 Form 10-K, what fac..."
6,H07,GOOGL,2024,revenue_driver,medium,"According to Alphabet's 2024 Form 10-K, what f..."
7,H08,GOOGL,2023,risk_factor,hard,"According to Alphabet's 2023 Form 10-K, what r..."
8,H09,NVDA,2025,segment_driver,medium,"According to NVIDIA's 2025 Form 10-K, what fac..."
9,H10,NVDA,2024,supply_risk,hard,"According to NVIDIA's 2024 Form 10-K, what sup..."


In [ ]:
# ============================================================
# INSPECT AVAILABLE RETRIEVAL INTERFACES AND CORPUS SCHEMA
# ============================================================
# Read-only inspection. This cell performs no retrieval and
# does not modify the frozen evaluation design.
# ============================================================

import inspect
import pandas as pd


# ------------------------------------------------------------
# 1. Locate likely retrieval functions
# ------------------------------------------------------------

retrieval_keywords = (
    "retriev",
    "search",
    "dense",
    "bm25",
    "hybrid",
    "rrf",
    "expand",
    "divers",
)

candidate_functions = []

for name, obj in sorted(globals().items()):
    if (
        callable(obj)
        and any(
            keyword in name.lower()
            for keyword in retrieval_keywords
        )
        and not name.startswith("_")
    ):
        try:
            signature = str(inspect.signature(obj))
        except (TypeError, ValueError):
            signature = "<signature unavailable>"

        candidate_functions.append(
            {
                "name": name,
                "signature": signature,
                "object_type": type(obj).__name__,
            }
        )


print("LIKELY RETRIEVAL FUNCTIONS")
print("-" * 80)

if candidate_functions:
    for item in candidate_functions:
        print(
            f"{item['name']}{item['signature']} "
            f"[{item['object_type']}]"
        )
else:
    print("No likely retrieval functions detected.")


# ------------------------------------------------------------
# 2. Locate likely retriever/index objects
# ------------------------------------------------------------

object_keywords = (
    "retriever",
    "index",
    "faiss",
    "bm25",
    "embedding",
    "model",
    "corpus",
    "chunk",
)

candidate_objects = []

for name, obj in sorted(globals().items()):
    if (
        any(
            keyword in name.lower()
            for keyword in object_keywords
        )
        and not name.startswith("_")
        and not inspect.ismodule(obj)
        and not inspect.isfunction(obj)
        and not inspect.isclass(obj)
    ):
        shape = getattr(obj, "shape", None)

        try:
            length = len(obj)
        except Exception:
            length = None

        candidate_objects.append(
            {
                "name": name,
                "type": type(obj).__name__,
                "shape": shape,
                "length": length,
            }
        )


print()
print("LIKELY RETRIEVAL OBJECTS")
print("-" * 80)

for item in candidate_objects:
    print(
        f"{item['name']} | "
        f"type={item['type']} | "
        f"shape={item['shape']} | "
        f"length={item['length']}"
    )


# ------------------------------------------------------------
# 3. Inspect likely corpus DataFrames
# ------------------------------------------------------------

likely_dataframes = []

for name, obj in sorted(globals().items()):
    if (
        isinstance(obj, pd.DataFrame)
        and any(
            keyword in name.lower()
            for keyword in (
                "corpus",
                "chunk",
                "document",
                "metadata",
            )
        )
    ):
        likely_dataframes.append(name)


print()
print("LIKELY CORPUS DATAFRAMES")
print("-" * 80)

if not likely_dataframes:
    print("No likely corpus DataFrame detected.")

for name in likely_dataframes:
    dataframe = globals()[name]

    print()
    print(f"{name}: {dataframe.shape}")
    print("Columns:")
    print(dataframe.columns.tolist())

    print("First-row non-null values:")
    first_row = dataframe.iloc[0]

    for column in dataframe.columns:
        value = first_row[column]

        if pd.notna(value):
            preview = str(value).replace("\n", " ")

            if len(preview) > 180:
                preview = preview[:180] + "..."

            print(f"  {column}: {preview}")


# ------------------------------------------------------------
# 4. Confirm frozen artifacts remain available
# ------------------------------------------------------------

print()
print("FROZEN EVALUATION OBJECTS")
print("-" * 80)

for required_name in [
    "heldout_queries",
    "FROZEN_RETRIEVAL_CONFIG",
    "CHECKPOINT_DIR",
]:
    if required_name in globals():
        print(
            f"AVAILABLE: {required_name} "
            f"({type(globals()[required_name]).__name__})"
        )
    else:
        print(f"MISSING: {required_name}")


print()
print("INSPECTION COMPLETE — NO RETRIEVAL WAS RUN")

LIKELY RETRIEVAL FUNCTIONS
--------------------------------------------------------------------------------
BM25Okapi(corpus, tokenizer=None, k1=1.5, b=0.75, epsilon=0.25) [type]
construct_retrieval_results(corpus_rows, scores, *, retrieval_method, bm25_ranks=None, dense_ranks=None) [function]
retrieve_bm25(query, *, top_k=10, ticker=None, form_type=None, filing_date=None) [function]
retrieve_dense(query, *, top_k=10, ticker=None, form_type=None, filing_date=None) [function]
retrieve_hybrid_rrf(query, *, top_k=10, candidate_k=50, rrf_k=60, ticker=None, form_type=None, filing_date=None) [function]
tokenize_for_bm25(text) [function]

LIKELY RETRIEVAL OBJECTS
--------------------------------------------------------------------------------
EMBEDDING_MODEL_NAME | type=str | shape=None | length=22
bm25_index | type=BM25Okapi | shape=None | length=None
chunk_embeddings | type=ndarray | shape=(4174, 384) | length=4174
chunks_df | type=DataFrame | shape=(4174, 23) | length=4174
corpus_filing_da

In [ ]:
# ============================================================
# INSPECT RETRIEVAL IMPLEMENTATIONS BEFORE HELD-OUT EXECUTION
# ============================================================
# Read-only inspection:
#   - performs no retrieval,
#   - changes no frozen artifacts,
#   - helps reconcile the frozen config with actual code.
# ============================================================

import inspect
from IPython.display import display, Markdown


functions_to_inspect = [
    retrieve_bm25,
    retrieve_dense,
    retrieve_hybrid_rrf,
    construct_retrieval_results,
]


display(
    Markdown(
        """
## Retrieval implementation audit

The frozen configuration currently contains separate dense/BM25 candidate
depths plus expansion and diversification settings. The callable interfaces
shown in memory do not expose all of those options, so we are inspecting the
actual source before running the held-out benchmark.
"""
    )
)


for function in functions_to_inspect:
    print("\n" + "=" * 100)
    print(f"FUNCTION: {function.__name__}")
    print(f"SIGNATURE: {inspect.signature(function)}")
    print("-" * 100)

    try:
        source = inspect.getsource(function)
        print(source)
    except (OSError, TypeError) as error:
        print(
            "SOURCE UNAVAILABLE:",
            type(error).__name__,
            str(error),
        )


print("\n" + "=" * 100)
print("AVAILABLE 10-K FILING DATES BY TICKER AND YEAR")
print("-" * 100)

filing_inventory = (
    chunks_df.loc[
        chunks_df["form_type"].astype(str).str.upper().eq("10-K"),
        [
            "ticker",
            "filing_date",
            "report_date",
            "document_id",
            "accession_number",
        ],
    ]
    .drop_duplicates()
    .copy()
)

filing_inventory["filing_date"] = pd.to_datetime(
    filing_inventory["filing_date"],
    errors="coerce",
)

filing_inventory["filing_year"] = (
    filing_inventory["filing_date"].dt.year.astype("Int64")
)

filing_inventory = filing_inventory.sort_values(
    [
        "ticker",
        "filing_year",
        "filing_date",
    ]
).reset_index(drop=True)

display(filing_inventory)


print("\n" + "=" * 100)
print("HELD-OUT COMPANY/YEAR TO FILING-DATE MAPPING")
print("-" * 100)

heldout_filing_map = (
    heldout_queries[
        [
            "query_id",
            "ticker",
            "filing_year",
        ]
    ]
    .merge(
        filing_inventory[
            [
                "ticker",
                "filing_year",
                "filing_date",
                "report_date",
                "document_id",
                "accession_number",
            ]
        ],
        on=[
            "ticker",
            "filing_year",
        ],
        how="left",
        validate="many_to_many",
    )
    .sort_values(
        [
            "query_id",
            "filing_date",
        ]
    )
    .reset_index(drop=True)
)

display(heldout_filing_map)


mapping_counts = (
    heldout_filing_map.groupby(
        "query_id",
        dropna=False,
    )["filing_date"]
    .nunique(dropna=True)
)

missing_query_ids = mapping_counts[
    mapping_counts.eq(0)
].index.tolist()

ambiguous_query_ids = mapping_counts[
    mapping_counts.gt(1)
].index.tolist()


print("Missing filing-date mappings:", missing_query_ids)
print("Multiple filing-date mappings:", ambiguous_query_ids)

if missing_query_ids:
    print(
        "\nWARNING: Some held-out queries do not map to a 10-K "
        "filing date."
    )

if ambiguous_query_ids:
    print(
        "\nWARNING: Some held-out queries map to multiple 10-K "
        "filing dates and require explicit resolution."
    )

if not missing_query_ids and not ambiguous_query_ids:
    print(
        "\nEach held-out query maps uniquely to one exact 10-K "
        "filing date."
    )

print(
    "\nIMPLEMENTATION AUDIT COMPLETE — "
    "NO RETRIEVAL WAS RUN AND NO ARTIFACT WAS MODIFIED"
)


## Retrieval implementation audit

The frozen configuration currently contains separate dense/BM25 candidate
depths plus expansion and diversification settings. The callable interfaces
shown in memory do not expose all of those options, so we are inspecting the
actual source before running the held-out benchmark.



FUNCTION: retrieve_bm25
SIGNATURE: (query, *, top_k=10, ticker=None, form_type=None, filing_date=None)
----------------------------------------------------------------------------------------------------
def retrieve_bm25(
    query,
    *,
    top_k=10,
    ticker=None,
    form_type=None,
    filing_date=None,
):
    """Retrieve chunks with BM25 and optional metadata filters."""

    if not 1 <= top_k <= EXPECTED_ROWS:
        raise ValueError(
            f"top_k must be between 1 and {EXPECTED_ROWS:,}."
        )

    candidate_rows = resolve_candidate_rows(
        ticker=ticker,
        form_type=form_type,
        filing_date=filing_date,
    )

    query_tokens = tokenize_for_bm25(
        query
    )

    all_scores = np.asarray(
        bm25_index.get_scores(
            query_tokens
        ),
        dtype=np.float64,
    )

    candidate_scores = all_scores[
        candidate_rows
    ]

    selected_count = min(
        top_k,
        len(candidate_rows),
    )

    sele

,ticker,filing_date,report_date,document_id,accession_number,filing_year
0,AAPL,2023-11-03,2023-09-30,aapl_10k_000032019323000106,0000320193-23-000106,2023
1,AAPL,2024-11-01,2024-09-28,aapl_10k_000032019324000123,0000320193-24-000123,2024
2,AAPL,2025-10-31,2025-09-27,aapl_10k_000032019325000079,0000320193-25-000079,2025
3,AMZN,2024-02-02,2023-12-31,amzn_10k_000101872424000008,0001018724-24-000008,2024
4,AMZN,2025-02-07,2024-12-31,amzn_10k_000101872425000004,0001018724-25-000004,2025
5,AMZN,2026-02-06,2025-12-31,amzn_10k_000101872426000004,0001018724-26-000004,2026
6,GOOGL,2024-01-31,2023-12-31,googl_10k_000165204424000022,0001652044-24-000022,2024
7,GOOGL,2025-02-05,2024-12-31,googl_10k_000165204425000014,0001652044-25-000014,2025
8,GOOGL,2026-02-05,2025-12-31,googl_10k_000165204426000018,0001652044-26-000018,2026
9,MSFT,2023-07-27,2023-06-30,msft_10k_000095017023035122,0000950170-23-035122,2023



HELD-OUT COMPANY/YEAR TO FILING-DATE MAPPING
----------------------------------------------------------------------------------------------------


,query_id,ticker,filing_year,filing_date,report_date,document_id,accession_number
0,H01,AAPL,2025,2025-10-31,2025-09-27,aapl_10k_000032019325000079,0000320193-25-000079
1,H02,AAPL,2024,2024-11-01,2024-09-28,aapl_10k_000032019324000123,0000320193-24-000123
2,H03,MSFT,2025,2025-07-30,2025-06-30,msft_10k_000095017025100235,0000950170-25-100235
3,H04,MSFT,2024,2024-07-30,2024-06-30,msft_10k_000095017024087843,0000950170-24-087843
4,H05,AMZN,2024,2024-02-02,2023-12-31,amzn_10k_000101872424000008,0001018724-24-000008
5,H06,AMZN,2023,NaT,NaN,NaN,NaN
6,H07,GOOGL,2024,2024-01-31,2023-12-31,googl_10k_000165204424000022,0001652044-24-000022
7,H08,GOOGL,2023,NaT,NaN,NaN,NaN
8,H09,NVDA,2025,2025-02-26,2025-01-26,nvda_10k_000104581025000023,0001045810-25-000023
9,H10,NVDA,2024,2024-02-21,2024-01-28,nvda_10k_000104581024000029,0001045810-24-000029


Missing filing-date mappings: ['H06', 'H08']
Multiple filing-date mappings: []


IMPLEMENTATION AUDIT COMPLETE — NO RETRIEVAL WAS RUN AND NO ARTIFACT WAS MODIFIED


In [ ]:
# ============================================================
# PRE-EXECUTION AMENDMENT + FROZEN HELD-OUT RETRIEVAL POOL
# ============================================================
# This cell:
#   1. preserves the original v1 artifacts,
#   2. interprets each question's year as the annual-report year,
#   3. maps that year to one exact filing_date,
#   4. records an executable v2 retrieval configuration,
#   5. runs Dense, BM25, and asymmetric Hybrid RRF,
#   6. pools and deduplicates each method's top 10,
#   7. creates a deterministic blind-review order,
#   8. saves and verifies all evaluation artifacts.
#
# No relevance judgments are assigned here.
# ============================================================

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display, Markdown


# ------------------------------------------------------------
# 1. Validate required Notebook 3 objects
# ------------------------------------------------------------

required_objects = [
    "CHECKPOINT_DIR",
    "heldout_queries",
    "chunks_df",
    "retrieve_dense",
    "retrieve_bm25",
    "construct_retrieval_results",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Rerun the earlier Notebook 3 cells. "
        f"Missing objects: {missing_objects}"
    )


CHECKPOINT_DIR = Path(CHECKPOINT_DIR)

if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(
        f"Checkpoint directory not found: {CHECKPOINT_DIR}"
    )


# ------------------------------------------------------------
# 2. Define artifact paths
# ------------------------------------------------------------

V1_CONFIG_PATH = (
    CHECKPOINT_DIR
    / "heldout_retrieval_config_v1.json"
)

V1_QUERIES_PATH = (
    CHECKPOINT_DIR
    / "heldout_queries_v1.parquet"
)

V2_CONFIG_PATH = (
    CHECKPOINT_DIR
    / "heldout_retrieval_config_v2.json"
)

V2_QUERIES_PATH = (
    CHECKPOINT_DIR
    / "heldout_queries_v2.parquet"
)

V2_QUERIES_CSV_PATH = (
    CHECKPOINT_DIR
    / "heldout_queries_v2.csv"
)

AMENDMENT_PATH = (
    CHECKPOINT_DIR
    / "heldout_preexecution_amendment_v2.json"
)

RUNS_PATH = (
    CHECKPOINT_DIR
    / "heldout_retrieval_runs_v2.parquet"
)

RUNS_CSV_PATH = (
    CHECKPOINT_DIR
    / "heldout_retrieval_runs_v2.csv"
)

POOL_PATH = (
    CHECKPOINT_DIR
    / "heldout_judgment_pool_v2.parquet"
)

POOL_CSV_PATH = (
    CHECKPOINT_DIR
    / "heldout_judgment_pool_v2.csv"
)


for path in [
    V1_CONFIG_PATH,
    V1_QUERIES_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required frozen v1 artifact not found: {path}"
        )


# ------------------------------------------------------------
# 3. Utility functions
# ------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def normalize_date_string(value):
    parsed = pd.to_datetime(
        value,
        errors="raise",
    )

    return parsed.strftime("%Y-%m-%d")


# ------------------------------------------------------------
# 4. Load the originally frozen questions
# ------------------------------------------------------------

heldout_queries_v1 = pd.read_parquet(
    V1_QUERIES_PATH
)

if len(heldout_queries_v1) != 12:
    raise AssertionError(
        "Expected 12 frozen v1 held-out questions."
    )

if heldout_queries_v1["query_id"].duplicated().any():
    raise AssertionError(
        "The frozen v1 query IDs are not unique."
    )


# ------------------------------------------------------------
# 5. Build annual-report inventory
# ------------------------------------------------------------
# The year stated in the question refers to the reporting period,
# not necessarily the calendar year in which the SEC filing was
# submitted. Therefore, report_date determines report_year.
# ------------------------------------------------------------

filing_inventory = (
    chunks_df.loc[
        chunks_df["form_type"]
        .astype(str)
        .str.upper()
        .eq("10-K"),
        [
            "ticker",
            "filing_date",
            "report_date",
            "document_id",
            "accession_number",
        ],
    ]
    .drop_duplicates()
    .copy()
)

filing_inventory["ticker"] = (
    filing_inventory["ticker"]
    .astype(str)
    .str.upper()
)

filing_inventory["filing_date"] = pd.to_datetime(
    filing_inventory["filing_date"],
    errors="raise",
)

filing_inventory["report_date"] = pd.to_datetime(
    filing_inventory["report_date"],
    errors="raise",
)

filing_inventory["report_year"] = (
    filing_inventory["report_date"]
    .dt.year
    .astype("Int64")
)

filing_inventory["sec_filing_year"] = (
    filing_inventory["filing_date"]
    .dt.year
    .astype("Int64")
)


# ------------------------------------------------------------
# 6. Create the executable v2 query table
# ------------------------------------------------------------
# The legacy column named filing_year is preserved for traceability
# but renamed to report_year in the executable design.
# Question wording is not changed.
# ------------------------------------------------------------

heldout_queries_v2 = (
    heldout_queries_v1
    .rename(
        columns={
            "filing_year": "report_year",
        }
    )
    .copy()
)

heldout_queries_v2["ticker"] = (
    heldout_queries_v2["ticker"]
    .astype(str)
    .str.upper()
)

heldout_queries_v2["report_year"] = (
    pd.to_numeric(
        heldout_queries_v2["report_year"],
        errors="raise",
    )
    .astype("Int64")
)

heldout_queries_v2 = heldout_queries_v2.merge(
    filing_inventory[
        [
            "ticker",
            "report_year",
            "sec_filing_year",
            "filing_date",
            "report_date",
            "document_id",
            "accession_number",
        ]
    ],
    on=[
        "ticker",
        "report_year",
    ],
    how="left",
    validate="many_to_one",
)


mapping_columns = [
    "filing_date",
    "report_date",
    "document_id",
    "accession_number",
]

missing_mapping_mask = (
    heldout_queries_v2[mapping_columns]
    .isna()
    .any(axis=1)
)

if missing_mapping_mask.any():
    display(
        heldout_queries_v2.loc[
            missing_mapping_mask,
            [
                "query_id",
                "ticker",
                "report_year",
            ],
        ]
    )

    raise RuntimeError(
        "At least one held-out question could not be mapped "
        "to exactly one annual report."
    )


if heldout_queries_v2["query_id"].duplicated().any():
    raise AssertionError(
        "The v2 filing mapping created duplicate query rows."
    )


heldout_queries_v2["filing_date"] = (
    heldout_queries_v2["filing_date"]
    .dt.strftime("%Y-%m-%d")
)

heldout_queries_v2["report_date"] = (
    heldout_queries_v2["report_date"]
    .dt.strftime("%Y-%m-%d")
)


# ------------------------------------------------------------
# 7. Record the executable retrieval configuration
# ------------------------------------------------------------

EXECUTABLE_RETRIEVAL_CONFIG = {
    "config_name": "heldout_evaluation_v2_executable",
    "evaluation_status": "frozen_before_first_heldout_retrieval",
    "retrieval_methods": [
        "dense",
        "bm25",
        "hybrid_rrf",
    ],
    "dense_candidate_k": 30,
    "bm25_candidate_k": 20,
    "rrf_k": 30,
    "final_top_k": 10,
    "evaluation_pool_depth": 10,
    "metadata_filters": [
        "ticker",
        "form_type",
        "filing_date",
    ],
    "form_type": "10-K",
    "year_interpretation": (
        "Question year is annual-report year derived from "
        "report_date; it is mapped to the exact SEC filing_date."
    ),
    "expansion_applied": False,
    "diversification_applied": False,
    "expansion_diversification_note": (
        "Removed before held-out execution because the inspected "
        "retrieval implementation does not implement these stages."
    ),
    "pool_deduplication_key": [
        "query_id",
        "corpus_row",
    ],
    "blind_review_seed": 20260806,
    "relevance_scale": {
        "0": "irrelevant",
        "1": "partially_relevant",
        "2": "directly_relevant",
    },
    "frozen_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}


amendment_record = {
    "amendment_name": (
        "heldout_preexecution_amendment_v2"
    ),
    "amendment_timing": (
        "Before the first held-out retrieval run and before "
        "inspection of any held-out results."
    ),
    "questions_changed": False,
    "relevance_labels_changed": False,
    "original_v1_config_sha256": sha256_file(
        V1_CONFIG_PATH
    ),
    "original_v1_queries_sha256": sha256_file(
        V1_QUERIES_PATH
    ),
    "changes": [
        (
            "Renamed the executable query metadata field from "
            "filing_year to report_year because question years "
            "refer to annual reporting periods."
        ),
        (
            "Mapped report_year to the exact filing_date required "
            "by the retrieval functions."
        ),
        (
            "Implemented asymmetric RRF candidate depths directly: "
            "dense_candidate_k=30 and bm25_candidate_k=20."
        ),
        (
            "Removed expansion and diversification from the "
            "executable claim because the inspected retrieval "
            "implementation does not implement them."
        ),
    ],
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}


# Refuse to overwrite a previously executed v2 pool.
if POOL_PATH.exists() or RUNS_PATH.exists():
    raise RuntimeError(
        "Held-out v2 retrieval artifacts already exist. "
        "Do not rerun or overwrite them silently."
    )


with open(
    V2_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        EXECUTABLE_RETRIEVAL_CONFIG,
        file,
        indent=2,
    )

with open(
    AMENDMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        amendment_record,
        file,
        indent=2,
    )

heldout_queries_v2.to_parquet(
    V2_QUERIES_PATH,
    index=False,
)

heldout_queries_v2.to_csv(
    V2_QUERIES_CSV_PATH,
    index=False,
)


# ------------------------------------------------------------
# 8. Define asymmetric hybrid RRF
# ------------------------------------------------------------

def retrieve_hybrid_rrf_asymmetric(
    query,
    *,
    top_k,
    dense_candidate_k,
    bm25_candidate_k,
    rrf_k,
    ticker,
    form_type,
    filing_date,
):
    """
    Fuse independently retrieved BM25 and dense candidates using
    reciprocal rank fusion with separate candidate depths.
    """

    dense_candidates = retrieve_dense(
        query,
        top_k=dense_candidate_k,
        ticker=ticker,
        form_type=form_type,
        filing_date=filing_date,
    )

    bm25_candidates = retrieve_bm25(
        query,
        top_k=bm25_candidate_k,
        ticker=ticker,
        form_type=form_type,
        filing_date=filing_date,
    )

    dense_rank_map = {
        int(row.corpus_row): int(row.rank)
        for row in dense_candidates.itertuples(
            index=False
        )
    }

    bm25_rank_map = {
        int(row.corpus_row): int(row.rank)
        for row in bm25_candidates.itertuples(
            index=False
        )
    }

    candidate_rows = sorted(
        set(dense_rank_map)
        | set(bm25_rank_map)
    )

    fused_records = []

    for corpus_row in candidate_rows:
        dense_rank = dense_rank_map.get(
            corpus_row
        )

        bm25_rank = bm25_rank_map.get(
            corpus_row
        )

        rrf_score = 0.0

        if dense_rank is not None:
            rrf_score += 1.0 / (
                rrf_k + dense_rank
            )

        if bm25_rank is not None:
            rrf_score += 1.0 / (
                rrf_k + bm25_rank
            )

        fused_records.append(
            {
                "corpus_row": corpus_row,
                "retrieval_score": rrf_score,
                "dense_rank": dense_rank,
                "bm25_rank": bm25_rank,
            }
        )

    fused_records.sort(
        key=lambda record: (
            -record["retrieval_score"],
            (
                record["bm25_rank"]
                if record["bm25_rank"] is not None
                else np.inf
            ),
            (
                record["dense_rank"]
                if record["dense_rank"] is not None
                else np.inf
            ),
            record["corpus_row"],
        )
    )

    fused_records = fused_records[:top_k]

    return construct_retrieval_results(
        [
            record["corpus_row"]
            for record in fused_records
        ],
        [
            record["retrieval_score"]
            for record in fused_records
        ],
        retrieval_method=(
            "hybrid_rrf_asymmetric_filtered"
        ),
        bm25_ranks=pd.array(
            [
                record["bm25_rank"]
                for record in fused_records
            ],
            dtype="Int64",
        ),
        dense_ranks=pd.array(
            [
                record["dense_rank"]
                for record in fused_records
            ],
            dtype="Int64",
        ),
    )


# ------------------------------------------------------------
# 9. Run all three retrieval methods
# ------------------------------------------------------------

all_runs = []

for query_row in heldout_queries_v2.itertuples(
    index=False
):
    common_arguments = {
        "query": query_row.query,
        "ticker": query_row.ticker,
        "form_type": "10-K",
        "filing_date": query_row.filing_date,
    }

    dense_results = retrieve_dense(
        **common_arguments,
        top_k=(
            EXECUTABLE_RETRIEVAL_CONFIG[
                "final_top_k"
            ]
        ),
    )

    bm25_results = retrieve_bm25(
        **common_arguments,
        top_k=(
            EXECUTABLE_RETRIEVAL_CONFIG[
                "final_top_k"
            ]
        ),
    )

    hybrid_results = retrieve_hybrid_rrf_asymmetric(
        **common_arguments,
        top_k=(
            EXECUTABLE_RETRIEVAL_CONFIG[
                "final_top_k"
            ]
        ),
        dense_candidate_k=(
            EXECUTABLE_RETRIEVAL_CONFIG[
                "dense_candidate_k"
            ]
        ),
        bm25_candidate_k=(
            EXECUTABLE_RETRIEVAL_CONFIG[
                "bm25_candidate_k"
            ]
        ),
        rrf_k=(
            EXECUTABLE_RETRIEVAL_CONFIG[
                "rrf_k"
            ]
        ),
    )

    method_results = {
        "dense": dense_results,
        "bm25": bm25_results,
        "hybrid_rrf": hybrid_results,
    }

    for method_name, results in method_results.items():
        results = results.copy()

        results["query_id"] = query_row.query_id
        results["query"] = query_row.query
        results["query_ticker"] = query_row.ticker
        results["query_report_year"] = (
            int(query_row.report_year)
        )
        results["query_filing_date"] = (
            query_row.filing_date
        )
        results["evaluation_method"] = (
            method_name
        )

        all_runs.append(results)


retrieval_runs = pd.concat(
    all_runs,
    ignore_index=True,
)


# ------------------------------------------------------------
# 10. Validate raw retrieval runs
# ------------------------------------------------------------

expected_run_rows = (
    len(heldout_queries_v2)
    * len(
        EXECUTABLE_RETRIEVAL_CONFIG[
            "retrieval_methods"
        ]
    )
    * EXECUTABLE_RETRIEVAL_CONFIG[
        "final_top_k"
    ]
)

if len(retrieval_runs) != expected_run_rows:
    raise AssertionError(
        "Unexpected number of raw retrieval rows. "
        f"Expected {expected_run_rows}, "
        f"found {len(retrieval_runs)}."
    )


per_method_counts = (
    retrieval_runs.groupby(
        [
            "query_id",
            "evaluation_method",
        ]
    )
    .size()
)

if not per_method_counts.eq(
    EXECUTABLE_RETRIEVAL_CONFIG[
        "final_top_k"
    ]
).all():
    raise AssertionError(
        "At least one query/method does not have exactly "
        "10 results."
    )


ticker_matches = (
    retrieval_runs["ticker"]
    .astype(str)
    .str.upper()
    .eq(
        retrieval_runs["query_ticker"]
        .astype(str)
        .str.upper()
    )
)

if not ticker_matches.all():
    raise AssertionError(
        "Ticker-filter leakage detected."
    )


form_matches = (
    retrieval_runs["form_type"]
    .astype(str)
    .str.upper()
    .eq("10-K")
)

if not form_matches.all():
    raise AssertionError(
        "Form-type filter leakage detected."
    )


filing_date_matches = (
    pd.to_datetime(
        retrieval_runs["filing_date"],
        errors="raise",
    )
    .dt.strftime("%Y-%m-%d")
    .eq(
        retrieval_runs["query_filing_date"]
    )
)

if not filing_date_matches.all():
    raise AssertionError(
        "Filing-date filter leakage detected."
    )


if retrieval_runs.duplicated(
    [
        "query_id",
        "evaluation_method",
        "corpus_row",
    ]
).any():
    raise AssertionError(
        "Duplicate result found within a retrieval ranking."
    )


# ------------------------------------------------------------
# 11. Create wide, deduplicated judgment pool
# ------------------------------------------------------------

rank_table = retrieval_runs.pivot_table(
    index=[
        "query_id",
        "corpus_row",
    ],
    columns="evaluation_method",
    values="rank",
    aggfunc="first",
).rename(
    columns={
        "dense": "dense_rank",
        "bm25": "bm25_rank",
        "hybrid_rrf": "hybrid_rank",
    }
).reset_index()


score_table = retrieval_runs.pivot_table(
    index=[
        "query_id",
        "corpus_row",
    ],
    columns="evaluation_method",
    values="retrieval_score",
    aggfunc="first",
).rename(
    columns={
        "dense": "dense_score",
        "bm25": "bm25_score",
        "hybrid_rrf": "hybrid_score",
    }
).reset_index()


corpus_columns = [
    "query_id",
    "corpus_row",
    "chunk_id",
    "document_id",
    "chunk_index",
    "text",
    "ticker",
    "company",
    "form_type",
    "filing_date",
    "report_date",
    "accession_number",
    "source_url",
]

pool_metadata = (
    retrieval_runs[corpus_columns]
    .drop_duplicates(
        subset=[
            "query_id",
            "corpus_row",
        ]
    )
    .reset_index(drop=True)
)


judgment_pool = (
    pool_metadata
    .merge(
        rank_table,
        on=[
            "query_id",
            "corpus_row",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        score_table,
        on=[
            "query_id",
            "corpus_row",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        heldout_queries_v2[
            [
                "query_id",
                "query",
                "question_type",
                "difficulty",
                "report_year",
            ]
        ],
        on="query_id",
        how="left",
        validate="many_to_one",
    )
)


# Verify that both ranking tables matched every pooled candidate.
rank_columns = [
    "dense_rank",
    "bm25_rank",
    "hybrid_rank",
]

score_columns = [
    "dense_score",
    "bm25_score",
    "hybrid_score",
]

if judgment_pool[rank_columns].isna().all(axis=1).any():
    raise AssertionError(
        "At least one pooled candidate has no retrieval rank."
    )

if judgment_pool[score_columns].isna().all(axis=1).any():
    raise AssertionError(
        "At least one pooled candidate has no retrieval score."
    )


for rank_column in rank_columns:
    judgment_pool[rank_column] = pd.array(
        judgment_pool[rank_column],
        dtype="Int64",
    )


judgment_pool["retrieved_by_count"] = (
    judgment_pool[rank_columns]
    .notna()
    .sum(axis=1)
)


# Deterministic random review order within each query.
rng = np.random.default_rng(
    EXECUTABLE_RETRIEVAL_CONFIG[
        "blind_review_seed"
    ]
)

judgment_pool["random_value"] = rng.random(
    len(judgment_pool)
)

judgment_pool = (
    judgment_pool
    .sort_values(
        [
            "query_id",
            "random_value",
            "corpus_row",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

judgment_pool["review_order"] = (
    judgment_pool
    .groupby(
        "query_id",
        sort=False,
    )
    .cumcount()
    .add(1)
)

judgment_pool["human_relevance"] = pd.array(
    [pd.NA] * len(judgment_pool),
    dtype="Int64",
)

judgment_pool["review_reason"] = ""

judgment_pool = judgment_pool.drop(
    columns="random_value"
)


# ------------------------------------------------------------
# 12. Validate blind pool
# ------------------------------------------------------------

if judgment_pool.duplicated(
    [
        "query_id",
        "corpus_row",
    ]
).any():
    raise AssertionError(
        "The judgment pool was not deduplicated correctly."
    )


if judgment_pool["human_relevance"].notna().any():
    raise AssertionError(
        "The new judgment pool unexpectedly contains labels."
    )


query_pool_counts = (
    judgment_pool.groupby("query_id")
    .size()
)

if query_pool_counts.lt(10).any():
    raise AssertionError(
        "Every query should have at least 10 unique pooled "
        "candidates."
    )


if query_pool_counts.gt(30).any():
    raise AssertionError(
        "A query produced more than the maximum possible "
        "30 pooled candidates."
    )


# ------------------------------------------------------------
# 13. Save frozen retrieval outputs
# ------------------------------------------------------------

retrieval_runs.to_parquet(
    RUNS_PATH,
    index=False,
)

retrieval_runs.to_csv(
    RUNS_CSV_PATH,
    index=False,
)

judgment_pool.to_parquet(
    POOL_PATH,
    index=False,
)

judgment_pool.to_csv(
    POOL_CSV_PATH,
    index=False,
)


# ------------------------------------------------------------
# 14. Reload and verify persistence
# ------------------------------------------------------------

verified_runs = pd.read_parquet(
    RUNS_PATH
)

verified_pool = pd.read_parquet(
    POOL_PATH
)

if len(verified_runs) != len(retrieval_runs):
    raise AssertionError(
        "Raw retrieval-run persistence verification failed."
    )

if len(verified_pool) != len(judgment_pool):
    raise AssertionError(
        "Judgment-pool persistence verification failed."
    )

if verified_pool["human_relevance"].notna().any():
    raise AssertionError(
        "Persisted judgment pool unexpectedly contains labels."
    )


# ------------------------------------------------------------
# 15. Display only protocol-level summaries
# ------------------------------------------------------------
# Do not display passage text or method ranks before beginning
# blinded adjudication.
# ------------------------------------------------------------

display(
    Markdown(
        """
## Frozen held-out retrieval pool created

The v1 artifacts remain preserved. The executable v2 design was
documented and frozen before the first held-out retrieval. Candidate
passages have been pooled and deduplicated without assigning labels.
"""
    )
)


display(
    heldout_queries_v2[
        [
            "query_id",
            "ticker",
            "report_year",
            "sec_filing_year",
            "report_date",
            "filing_date",
            "document_id",
        ]
    ]
)


pool_summary = (
    judgment_pool.groupby("query_id")
    .agg(
        unique_candidates=(
            "corpus_row",
            "nunique",
        ),
        candidates_in_all_three=(
            "retrieved_by_count",
            lambda series: int(
                series.eq(3).sum()
            ),
        ),
        candidates_in_one_method=(
            "retrieved_by_count",
            lambda series: int(
                series.eq(1).sum()
            ),
        ),
    )
    .reset_index()
)

display(pool_summary)


print("RAW RETRIEVAL ROWS:", len(retrieval_runs))
print(
    "UNIQUE POOLED JUDGMENTS:",
    len(judgment_pool),
)
print(
    "POOL SIZE RANGE PER QUERY:",
    int(query_pool_counts.min()),
    "to",
    int(query_pool_counts.max()),
)
print(
    "UNLABELED ROWS:",
    int(
        judgment_pool[
            "human_relevance"
        ].isna().sum()
    ),
)
print()
print("v1 config preserved:", V1_CONFIG_PATH)
print("v1 questions preserved:", V1_QUERIES_PATH)
print("v2 config:", V2_CONFIG_PATH)
print("v2 mapped questions:", V2_QUERIES_PATH)
print("amendment record:", AMENDMENT_PATH)
print("raw rankings:", RUNS_PATH)
print("blind judgment pool:", POOL_PATH)
print()
print(
    "READY FOR BLINDED MANUAL ADJUDICATION"
)


## Frozen held-out retrieval pool created

The v1 artifacts remain preserved. The executable v2 design was
documented and frozen before the first held-out retrieval. Candidate
passages have been pooled and deduplicated without assigning labels.


,query_id,ticker,report_year,sec_filing_year,report_date,filing_date,document_id
0,H01,AAPL,2025,2025,2025-09-27,2025-10-31,aapl_10k_000032019325000079
1,H02,AAPL,2024,2024,2024-09-28,2024-11-01,aapl_10k_000032019324000123
2,H03,MSFT,2025,2025,2025-06-30,2025-07-30,msft_10k_000095017025100235
3,H04,MSFT,2024,2024,2024-06-30,2024-07-30,msft_10k_000095017024087843
4,H05,AMZN,2024,2025,2024-12-31,2025-02-07,amzn_10k_000101872425000004
5,H06,AMZN,2023,2024,2023-12-31,2024-02-02,amzn_10k_000101872424000008
6,H07,GOOGL,2024,2025,2024-12-31,2025-02-05,googl_10k_000165204425000014
7,H08,GOOGL,2023,2024,2023-12-31,2024-01-31,googl_10k_000165204424000022
8,H09,NVDA,2025,2025,2025-01-26,2025-02-26,nvda_10k_000104581025000023
9,H10,NVDA,2024,2024,2024-01-28,2024-02-21,nvda_10k_000104581024000029


,query_id,unique_candidates,candidates_in_all_three,candidates_in_one_method
0,H01,18,3,9
1,H02,19,1,9
2,H03,16,4,6
3,H04,19,2,10
4,H05,15,5,5
5,H06,18,3,9
6,H07,15,6,6
7,H08,21,1,13
8,H09,17,4,8
9,H10,16,5,7


RAW RETRIEVAL ROWS: 360
UNIQUE POOLED JUDGMENTS: 214
POOL SIZE RANGE PER QUERY: 15 to 21
UNLABELED ROWS: 214

v1 config preserved: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_config_v1.json
v1 questions preserved: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_queries_v1.parquet
v2 config: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_config_v2.json
v2 mapped questions: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_queries_v2.parquet
amendment record: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_preexecution_amendment_v2.json
raw rankings: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_runs_v2.parquet
blind judgment pool: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_judgment_pool_v2.parquet

READY FOR BLINDED MANUAL ADJUDICATION


In [ ]:
# ============================================================
# BLINDED, RESUMABLE HELD-OUT RELEVANCE ADJUDICATION
# ============================================================
# Label scale:
#   2 = Directly relevant
#       The passage directly answers the question or contains
#       the specific evidence needed to answer it.
#
#   1 = Partially relevant
#       The passage contains useful related evidence but does
#       not adequately answer the question by itself.
#
#   0 = Irrelevant
#       The passage does not materially help answer the question.
#
# Important:
#   - Retrieval method, rank, and score remain hidden.
#   - Every label is saved immediately.
#   - The original frozen unlabeled pool is never overwritten.
# ============================================================

from pathlib import Path
import html

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output


# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

CHECKPOINT_DIR = Path(CHECKPOINT_DIR)

FROZEN_POOL_PATH = (
    CHECKPOINT_DIR
    / "heldout_judgment_pool_v2.parquet"
)

WORKING_JUDGMENTS_PATH = (
    CHECKPOINT_DIR
    / "heldout_judgments_working_v2.parquet"
)

WORKING_JUDGMENTS_CSV_PATH = (
    CHECKPOINT_DIR
    / "heldout_judgments_working_v2.csv"
)


if not FROZEN_POOL_PATH.exists():
    raise FileNotFoundError(
        f"Frozen judgment pool not found: {FROZEN_POOL_PATH}"
    )


# ------------------------------------------------------------
# 2. Load or initialize resumable workspace
# ------------------------------------------------------------

if WORKING_JUDGMENTS_PATH.exists():
    adjudication_df = pd.read_parquet(
        WORKING_JUDGMENTS_PATH
    )

    print(
        "Resuming existing adjudication workspace:",
        WORKING_JUDGMENTS_PATH,
    )
else:
    adjudication_df = pd.read_parquet(
        FROZEN_POOL_PATH
    ).copy()

    adjudication_df.to_parquet(
        WORKING_JUDGMENTS_PATH,
        index=False,
    )

    adjudication_df.to_csv(
        WORKING_JUDGMENTS_CSV_PATH,
        index=False,
    )

    print(
        "Created new adjudication workspace:",
        WORKING_JUDGMENTS_PATH,
    )


required_columns = [
    "query_id",
    "query",
    "text",
    "review_order",
    "human_relevance",
    "review_reason",
]

missing_columns = [
    column
    for column in required_columns
    if column not in adjudication_df.columns
]

if missing_columns:
    raise RuntimeError(
        "Judgment workspace is missing required columns: "
        f"{missing_columns}"
    )


adjudication_df["human_relevance"] = pd.array(
    adjudication_df["human_relevance"],
    dtype="Int64",
)

adjudication_df["review_reason"] = (
    adjudication_df["review_reason"]
    .fillna("")
    .astype(str)
)


# Keep the frozen randomized ordering.
adjudication_df = (
    adjudication_df
    .sort_values(
        [
            "query_id",
            "review_order",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. Persistence helper
# ------------------------------------------------------------

def save_adjudication_workspace():
    adjudication_df.to_parquet(
        WORKING_JUDGMENTS_PATH,
        index=False,
    )

    adjudication_df.to_csv(
        WORKING_JUDGMENTS_CSV_PATH,
        index=False,
    )


# ------------------------------------------------------------
# 4. Navigation state
# ------------------------------------------------------------

unlabeled_indices = adjudication_df.index[
    adjudication_df["human_relevance"].isna()
].tolist()

if unlabeled_indices:
    initial_index = unlabeled_indices[0]
else:
    initial_index = len(adjudication_df) - 1

state = {
    "current_index": max(initial_index, 0),
}


# ------------------------------------------------------------
# 5. Build interface widgets
# ------------------------------------------------------------

progress_html = widgets.HTML()
question_html = widgets.HTML()
passage_html = widgets.HTML()
existing_label_html = widgets.HTML()
message_html = widgets.HTML()

reason_box = widgets.Textarea(
    placeholder=(
        "Optional short reason, especially for ambiguous "
        "0/1 or 1/2 decisions"
    ),
    description="Reason:",
    layout=widgets.Layout(
        width="100%",
        height="80px",
    ),
    style={
        "description_width": "70px",
    },
)

irrelevant_button = widgets.Button(
    description="0 — Irrelevant",
    button_style="danger",
    layout=widgets.Layout(width="180px"),
)

partial_button = widgets.Button(
    description="1 — Partial",
    button_style="warning",
    layout=widgets.Layout(width="180px"),
)

direct_button = widgets.Button(
    description="2 — Direct",
    button_style="success",
    layout=widgets.Layout(width="180px"),
)

previous_button = widgets.Button(
    description="← Previous",
    layout=widgets.Layout(width="130px"),
)

skip_button = widgets.Button(
    description="Skip →",
    layout=widgets.Layout(width="130px"),
)

next_unlabeled_button = widgets.Button(
    description="Next unlabeled",
    button_style="info",
    layout=widgets.Layout(width="150px"),
)


# ------------------------------------------------------------
# 6. Rendering helpers
# ------------------------------------------------------------

def escaped_multiline(value):
    return html.escape(str(value)).replace(
        "\n",
        "<br>"
    )


def completed_count():
    return int(
        adjudication_df[
            "human_relevance"
        ].notna().sum()
    )


def current_row():
    return adjudication_df.iloc[
        state["current_index"]
    ]


def render_current():
    if len(adjudication_df) == 0:
        progress_html.value = (
            "<b>No pooled candidates are available.</b>"
        )
        return

    row = current_row()

    complete = completed_count()
    total = len(adjudication_df)
    remaining = total - complete

    query_complete = int(
        adjudication_df.loc[
            adjudication_df["query_id"].eq(
                row["query_id"]
            ),
            "human_relevance",
        ]
        .notna()
        .sum()
    )

    query_total = int(
        adjudication_df[
            "query_id"
        ].eq(row["query_id"]).sum()
    )

    progress_html.value = f"""
    <div style="
        padding:12px;
        border:1px solid #bdbdbd;
        border-radius:8px;
        margin-bottom:10px;
    ">
        <b>Overall progress:</b>
        {complete} / {total} labeled
        &nbsp;|&nbsp;
        {remaining} remaining
        <br>
        <b>Current question:</b>
        {html.escape(str(row["query_id"]))}
        &nbsp;|&nbsp;
        {query_complete} / {query_total} labeled
        &nbsp;|&nbsp;
        Blind candidate {int(row["review_order"])}
        of {query_total}
    </div>
    """

    question_html.value = f"""
    <div style="
        padding:14px;
        background:#eef4ff;
        border-left:5px solid #3367d6;
        margin-bottom:12px;
        font-size:16px;
        line-height:1.5;
    ">
        <b>Question</b><br>
        {escaped_multiline(row["query"])}
    </div>
    """

    passage_html.value = f"""
    <div style="
        padding:16px;
        background:#f8f8f8;
        border:1px solid #d0d0d0;
        border-radius:6px;
        max-height:500px;
        overflow-y:auto;
        white-space:normal;
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.6;
    ">
        <b>Candidate passage</b><br><br>
        {escaped_multiline(row["text"])}
    </div>
    """

    existing_label = row["human_relevance"]

    if pd.isna(existing_label):
        existing_label_html.value = (
            "<p><b>Current label:</b> Unlabeled</p>"
        )
    else:
        label_names = {
            0: "Irrelevant",
            1: "Partially relevant",
            2: "Directly relevant",
        }

        numeric_label = int(existing_label)

        existing_label_html.value = (
            "<p><b>Current label:</b> "
            f"{numeric_label} — "
            f"{label_names[numeric_label]}</p>"
        )

    reason_box.value = str(
        row["review_reason"]
    )

    message_html.value = ""


# ------------------------------------------------------------
# 7. Navigation helpers
# ------------------------------------------------------------

def move_to_next_unlabeled():
    current_position = state["current_index"]

    later_unlabeled = adjudication_df.index[
        (
            adjudication_df.index
            > current_position
        )
        & adjudication_df[
            "human_relevance"
        ].isna()
    ].tolist()

    if later_unlabeled:
        state["current_index"] = later_unlabeled[0]
        render_current()
        return

    earlier_unlabeled = adjudication_df.index[
        (
            adjudication_df.index
            < current_position
        )
        & adjudication_df[
            "human_relevance"
        ].isna()
    ].tolist()

    if earlier_unlabeled:
        state["current_index"] = earlier_unlabeled[0]
        render_current()
        return

    render_current()

    message_html.value = """
    <div style="
        padding:12px;
        background:#e6f4ea;
        border:1px solid #34a853;
        border-radius:6px;
    ">
        <b>All pooled candidates have been labeled.</b>
    </div>
    """


def assign_label(label):
    row_index = state["current_index"]

    adjudication_df.at[
        row_index,
        "human_relevance",
    ] = int(label)

    adjudication_df.at[
        row_index,
        "review_reason",
    ] = reason_box.value.strip()

    save_adjudication_workspace()
    move_to_next_unlabeled()


# ------------------------------------------------------------
# 8. Button callbacks
# ------------------------------------------------------------

def on_irrelevant_clicked(_):
    assign_label(0)


def on_partial_clicked(_):
    assign_label(1)


def on_direct_clicked(_):
    assign_label(2)


def on_previous_clicked(_):
    if state["current_index"] > 0:
        state["current_index"] -= 1
        render_current()
    else:
        message_html.value = (
            "<p>Already at the first candidate.</p>"
        )


def on_skip_clicked(_):
    if state["current_index"] < len(
        adjudication_df
    ) - 1:
        state["current_index"] += 1
        render_current()
    else:
        state["current_index"] = 0
        render_current()


def on_next_unlabeled_clicked(_):
    move_to_next_unlabeled()


irrelevant_button.on_click(
    on_irrelevant_clicked
)

partial_button.on_click(
    on_partial_clicked
)

direct_button.on_click(
    on_direct_clicked
)

previous_button.on_click(
    on_previous_clicked
)

skip_button.on_click(
    on_skip_clicked
)

next_unlabeled_button.on_click(
    on_next_unlabeled_clicked
)


# ------------------------------------------------------------
# 9. Display interface
# ------------------------------------------------------------

label_buttons = widgets.HBox(
    [
        irrelevant_button,
        partial_button,
        direct_button,
    ],
    layout=widgets.Layout(
        justify_content="space-between",
        margin="12px 0 10px 0",
    ),
)

navigation_buttons = widgets.HBox(
    [
        previous_button,
        skip_button,
        next_unlabeled_button,
    ],
    layout=widgets.Layout(
        justify_content="flex-start",
        gap="10px",
        margin="6px 0 10px 0",
    ),
)

instructions_html = widgets.HTML(
    """
    <div style="
        padding:12px;
        background:#fff8e1;
        border-left:5px solid #f9ab00;
        margin-bottom:12px;
        line-height:1.5;
    ">
        <b>Judge only whether this passage helps answer the displayed
        question.</b><br>
        2 = direct answer/evidence;
        1 = useful but incomplete evidence;
        0 = no material answer support.
        Do not infer relevance from writing style or general discussion
        of the same company.
    </div>
    """
)

interface = widgets.VBox(
    [
        instructions_html,
        progress_html,
        question_html,
        passage_html,
        existing_label_html,
        reason_box,
        label_buttons,
        navigation_buttons,
        message_html,
    ]
)

render_current()
display(interface)

Resuming existing adjudication workspace: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_judgments_working_v2.parquet


In [ ]:
from pathlib import Path
from datetime import datetime
import time

drive_root = Path("/content/drive/MyDrive")
keywords = ("label", "annotation", "checkpoint", "relevance", "judgment", "qrel")
extensions = {".csv", ".json", ".jsonl", ".pkl", ".parquet"}

if not drive_root.exists():
    raise RuntimeError("Google Drive is not mounted. Run the Drive-mount cell first.")

matches = [
    p for p in drive_root.rglob("*")
    if p.is_file()
    and p.suffix.lower() in extensions
    and any(word in p.name.lower() for word in keywords)
]

matches.sort(key=lambda p: p.stat().st_mtime, reverse=True)

print("Most recently modified candidate checkpoint files:\n")

for p in matches[:15]:
    modified = datetime.fromtimestamp(p.stat().st_mtime)
    age_minutes = (time.time() - p.stat().st_mtime) / 60
    print(f"{modified:%Y-%m-%d %H:%M:%S} | {age_minutes:6.1f} min ago")
    print(f"  {p}")
    print(f"  Size: {p.stat().st_size:,} bytes\n")

if not matches:
    print("WARNING: No likely checkpoint file was found in Google Drive.")
elif (time.time() - matches[0].stat().st_mtime) / 60 <= 15:
    print("✅ A likely checkpoint was updated within the last 15 minutes.")
    print("You should be safe to disconnect after confirming it is the correct file.")
else:
    print("⚠️ The newest likely checkpoint is more than 15 minutes old.")
    print("Do not disconnect yet—we need to explicitly save your current labels.")

Most recently modified candidate checkpoint files:

2026-08-09 23:39:30 |    7.5 min ago
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_judgments_working_v2.parquet
  Size: 236,537 bytes

2026-08-09 23:39:30 |    7.5 min ago
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_judgments_working_v2.csv
  Size: 566,540 bytes

2026-08-06 08:27:43 | 5239.3 min ago
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_judgment_pool_v2.parquet
  Size: 236,148 bytes

2026-08-06 08:27:43 | 5239.3 min ago
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_judgment_pool_v2.csv
  Size: 566,185 bytes

2026-08-06 01:14:52 | 5672.1 min ago
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/groq_judge_calibration_20_checkpoint.parquet
  Size: 12,517 bytes

2026-08-06 00:54:36 | 5692.4 min ago
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/gemini_judge_calibration_20_checkpoint.parque

In [ ]:
# ============================================================
# FREEZE AND VALIDATE COMPLETED HELD-OUT HUMAN JUDGMENTS
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import pandas as pd


# ------------------------------------------------------------
# 1. Define source and final artifact paths
# ------------------------------------------------------------

CHECKPOINT_DIR = Path(CHECKPOINT_DIR)

WORKING_JUDGMENTS_PATH = (
    CHECKPOINT_DIR
    / "heldout_judgments_working_v2.parquet"
)

FINAL_JUDGMENTS_PATH = (
    CHECKPOINT_DIR
    / "heldout_human_judgments_final_v2.parquet"
)

FINAL_JUDGMENTS_CSV_PATH = (
    CHECKPOINT_DIR
    / "heldout_human_judgments_final_v2.csv"
)

FINAL_QRELS_PATH = (
    CHECKPOINT_DIR
    / "heldout_qrels_final_v2.parquet"
)

FINAL_QRELS_CSV_PATH = (
    CHECKPOINT_DIR
    / "heldout_qrels_final_v2.csv"
)

FINAL_MANIFEST_PATH = (
    CHECKPOINT_DIR
    / "heldout_human_judgments_final_v2_manifest.json"
)


# ------------------------------------------------------------
# 2. Load completed working judgments
# ------------------------------------------------------------

if not WORKING_JUDGMENTS_PATH.exists():
    raise FileNotFoundError(
        f"Working judgments not found: "
        f"{WORKING_JUDGMENTS_PATH}"
    )

completed_judgments = pd.read_parquet(
    WORKING_JUDGMENTS_PATH
)

completed_judgments["human_relevance"] = pd.array(
    completed_judgments["human_relevance"],
    dtype="Int64",
)


# ------------------------------------------------------------
# 3. Structural validation
# ------------------------------------------------------------

required_columns = {
    "query_id",
    "query",
    "corpus_row",
    "chunk_id",
    "human_relevance",
    "review_reason",
    "dense_rank",
    "bm25_rank",
    "hybrid_rank",
}

missing_columns = (
    required_columns
    - set(completed_judgments.columns)
)

if missing_columns:
    raise AssertionError(
        "Missing required judgment columns: "
        f"{sorted(missing_columns)}"
    )

if len(completed_judgments) != 214:
    raise AssertionError(
        "Expected exactly 214 pooled judgments, "
        f"found {len(completed_judgments)}."
    )

missing_label_count = int(
    completed_judgments[
        "human_relevance"
    ].isna().sum()
)

if missing_label_count != 0:
    raise AssertionError(
        f"{missing_label_count} judgments remain unlabeled."
    )

observed_labels = set(
    completed_judgments[
        "human_relevance"
    ].astype(int).unique()
)

if not observed_labels.issubset({0, 1, 2}):
    raise AssertionError(
        f"Unexpected relevance labels: {observed_labels}"
    )

duplicate_count = int(
    completed_judgments.duplicated(
        subset=[
            "query_id",
            "corpus_row",
        ]
    ).sum()
)

if duplicate_count != 0:
    raise AssertionError(
        "Duplicate query–corpus-row judgments found: "
        f"{duplicate_count}"
    )

expected_query_ids = {
    f"H{number:02d}"
    for number in range(1, 13)
}

observed_query_ids = set(
    completed_judgments["query_id"]
)

if observed_query_ids != expected_query_ids:
    raise AssertionError(
        "Held-out query IDs do not match H01–H12.\n"
        f"Observed: {sorted(observed_query_ids)}"
    )


# ------------------------------------------------------------
# 4. Build compact qrels
# ------------------------------------------------------------

qrels_columns = [
    "query_id",
    "corpus_row",
    "chunk_id",
    "human_relevance",
    "review_reason",
]

final_qrels = (
    completed_judgments[qrels_columns]
    .rename(
        columns={
            "human_relevance": "relevance_grade",
            "review_reason": "adjudication_note",
        }
    )
    .sort_values(
        [
            "query_id",
            "corpus_row",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

final_qrels["label_source"] = "human"
final_qrels["human_reviewed"] = True
final_qrels["rubric_version"] = (
    "heldout_relevance_v2_0_1_2"
)


# ------------------------------------------------------------
# 5. Refuse to overwrite an existing frozen final artifact
# ------------------------------------------------------------

final_paths = [
    FINAL_JUDGMENTS_PATH,
    FINAL_JUDGMENTS_CSV_PATH,
    FINAL_QRELS_PATH,
    FINAL_QRELS_CSV_PATH,
    FINAL_MANIFEST_PATH,
]

existing_final_paths = [
    path
    for path in final_paths
    if path.exists()
]

if existing_final_paths:
    raise FileExistsError(
        "A frozen final artifact already exists. "
        "Do not overwrite it silently:\n"
        + "\n".join(
            str(path)
            for path in existing_final_paths
        )
    )


# ------------------------------------------------------------
# 6. Save final artifacts
# ------------------------------------------------------------

completed_judgments.to_parquet(
    FINAL_JUDGMENTS_PATH,
    index=False,
)

completed_judgments.to_csv(
    FINAL_JUDGMENTS_CSV_PATH,
    index=False,
)

final_qrels.to_parquet(
    FINAL_QRELS_PATH,
    index=False,
)

final_qrels.to_csv(
    FINAL_QRELS_CSV_PATH,
    index=False,
)


# ------------------------------------------------------------
# 7. Verify saved artifacts
# ------------------------------------------------------------

verified_judgments = pd.read_parquet(
    FINAL_JUDGMENTS_PATH
)

verified_qrels = pd.read_parquet(
    FINAL_QRELS_PATH
)

assert len(verified_judgments) == 214
assert len(verified_qrels) == 214
assert verified_qrels["relevance_grade"].notna().all()

assert not verified_qrels.duplicated(
    [
        "query_id",
        "corpus_row",
    ]
).any()


# ------------------------------------------------------------
# 8. Record checksums and freeze metadata
# ------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


freeze_manifest = {
    "artifact_type": "heldout_human_qrels",
    "artifact_version": "v2",
    "frozen_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "judgment_count": int(
        len(verified_judgments)
    ),
    "query_count": int(
        verified_judgments[
            "query_id"
        ].nunique()
    ),
    "relevance_scale": {
        "0": "irrelevant",
        "1": "partially_relevant",
        "2": "directly_relevant",
    },
    "source_working_file": str(
        WORKING_JUDGMENTS_PATH
    ),
    "artifacts": {
        "judgments_parquet": {
            "path": str(FINAL_JUDGMENTS_PATH),
            "sha256": sha256_file(
                FINAL_JUDGMENTS_PATH
            ),
        },
        "qrels_parquet": {
            "path": str(FINAL_QRELS_PATH),
            "sha256": sha256_file(
                FINAL_QRELS_PATH
            ),
        },
    },
}

with open(
    FINAL_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        freeze_manifest,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 9. Display initial label audit
# ------------------------------------------------------------

label_distribution = (
    verified_qrels[
        "relevance_grade"
    ]
    .value_counts()
    .reindex(
        [0, 1, 2],
        fill_value=0,
    )
    .rename_axis("relevance_grade")
    .reset_index(name="judgment_count")
)

label_distribution["percentage"] = (
    100
    * label_distribution["judgment_count"]
    / len(verified_qrels)
).round(2)

query_label_audit = (
    verified_qrels
    .groupby("query_id")
    .agg(
        pooled_candidates=(
            "corpus_row",
            "size",
        ),
        irrelevant=(
            "relevance_grade",
            lambda values: int(
                values.eq(0).sum()
            ),
        ),
        partial=(
            "relevance_grade",
            lambda values: int(
                values.eq(1).sum()
            ),
        ),
        direct=(
            "relevance_grade",
            lambda values: int(
                values.eq(2).sum()
            ),
        ),
        useful_total=(
            "relevance_grade",
            lambda values: int(
                values.ge(1).sum()
            ),
        ),
    )
    .reset_index()
)

query_label_audit["has_useful_evidence"] = (
    query_label_audit[
        "useful_total"
    ].gt(0)
)

query_label_audit["has_direct_evidence"] = (
    query_label_audit[
        "direct"
    ].gt(0)
)

print("=" * 72)
print("HELD-OUT HUMAN JUDGMENTS: FROZEN AND VERIFIED")
print("=" * 72)
print(f"Questions:             {verified_qrels['query_id'].nunique()}")
print(f"Judgments:             {len(verified_qrels)}")
print(f"Missing labels:        {verified_qrels['relevance_grade'].isna().sum()}")
print(f"Duplicate judgments:   {duplicate_count}")
print(f"Allowed labels only:   {observed_labels.issubset({0, 1, 2})}")
print()
print("Final judgments:", FINAL_JUDGMENTS_PATH)
print("Final qrels:    ", FINAL_QRELS_PATH)
print("Freeze manifest:", FINAL_MANIFEST_PATH)

display(label_distribution)
display(query_label_audit)

HELD-OUT HUMAN JUDGMENTS: FROZEN AND VERIFIED
Questions:             12
Judgments:             214
Missing labels:        0
Duplicate judgments:   0
Allowed labels only:   True

Final judgments: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_human_judgments_final_v2.parquet
Final qrels:     /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_qrels_final_v2.parquet
Freeze manifest: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_human_judgments_final_v2_manifest.json


,relevance_grade,judgment_count,percentage
0,0,128,59.81
1,1,43,20.09
2,2,43,20.09


,query_id,pooled_candidates,irrelevant,partial,direct,useful_total,has_useful_evidence,has_direct_evidence
0,H01,18,7,6,5,11,True,True
1,H02,19,6,8,5,13,True,True
2,H03,16,8,5,3,8,True,True
3,H04,19,10,7,2,9,True,True
4,H05,15,9,3,3,6,True,True
5,H06,18,13,2,3,5,True,True
6,H07,15,14,0,1,1,True,True
7,H08,21,16,4,1,5,True,True
8,H09,17,9,3,5,8,True,True
9,H10,16,4,3,9,12,True,True


In [ ]:
# ============================================================
# HELD-OUT RETRIEVAL EVALUATION:
# PER-QUERY METRICS FOR BM25, DENSE, AND HYBRID RRF
# ============================================================

import math
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Load frozen judgments
# ------------------------------------------------------------

evaluation_df = pd.read_parquet(
    FINAL_JUDGMENTS_PATH
).copy()

evaluation_df["human_relevance"] = pd.array(
    evaluation_df["human_relevance"],
    dtype="Int64",
)

METHOD_RANK_COLUMNS = {
    "BM25": "bm25_rank",
    "Dense": "dense_rank",
    "Hybrid RRF": "hybrid_rank",
}

EXPECTED_QUERY_IDS = [
    f"H{number:02d}"
    for number in range(1, 13)
]

FINAL_CUTOFF = 10


# ------------------------------------------------------------
# 2. Validate frozen ranking data
# ------------------------------------------------------------

missing_rank_columns = (
    set(METHOD_RANK_COLUMNS.values())
    - set(evaluation_df.columns)
)

if missing_rank_columns:
    raise AssertionError(
        "Missing rank columns: "
        f"{sorted(missing_rank_columns)}"
    )

ranking_audit_records = []

for method, rank_column in METHOD_RANK_COLUMNS.items():
    ranked_rows = evaluation_df.loc[
        evaluation_df[rank_column].notna(),
        [
            "query_id",
            "corpus_row",
            "chunk_id",
            "human_relevance",
            rank_column,
        ],
    ].copy()

    ranked_rows[rank_column] = (
        ranked_rows[rank_column]
        .astype(int)
    )

    for query_id in EXPECTED_QUERY_IDS:
        query_results = ranked_rows.loc[
            ranked_rows["query_id"].eq(query_id)
        ].copy()

        observed_ranks = sorted(
            query_results[rank_column].tolist()
        )

        expected_ranks = list(
            range(1, FINAL_CUTOFF + 1)
        )

        ranking_audit_records.append(
            {
                "method": method,
                "query_id": query_id,
                "retrieved_results": len(query_results),
                "unique_corpus_rows": (
                    query_results["corpus_row"].nunique()
                ),
                "observed_ranks": observed_ranks,
                "has_exact_top_10": (
                    observed_ranks == expected_ranks
                ),
            }
        )

ranking_audit = pd.DataFrame(
    ranking_audit_records
)

if not ranking_audit["has_exact_top_10"].all():
    display(
        ranking_audit.loc[
            ~ranking_audit["has_exact_top_10"]
        ]
    )
    raise AssertionError(
        "At least one query–method pair does not contain "
        "exactly ranks 1 through 10."
    )

if not (
    ranking_audit["retrieved_results"].eq(10).all()
    and ranking_audit["unique_corpus_rows"].eq(10).all()
):
    raise AssertionError(
        "Duplicate or missing retrieved results detected."
    )


# ------------------------------------------------------------
# 3. Metric helper functions
# ------------------------------------------------------------

def precision_at_k(binary_relevance, k):
    values = np.asarray(
        binary_relevance[:k],
        dtype=float,
    )

    return float(values.sum() / k)


def recall_at_k(
    binary_relevance,
    total_relevant_in_pool,
    k,
):
    if total_relevant_in_pool <= 0:
        return np.nan

    retrieved_relevant = np.asarray(
        binary_relevance[:k],
        dtype=float,
    ).sum()

    return float(
        retrieved_relevant
        / total_relevant_in_pool
    )


def hit_at_k(binary_relevance, k):
    return float(
        np.asarray(
            binary_relevance[:k],
            dtype=float,
        ).sum() > 0
    )


def reciprocal_rank_at_k(binary_relevance, k):
    for rank, relevance in enumerate(
        binary_relevance[:k],
        start=1,
    ):
        if relevance:
            return float(1.0 / rank)

    return 0.0


def dcg_at_k(graded_relevance, k):
    grades = np.asarray(
        graded_relevance[:k],
        dtype=float,
    )

    if len(grades) == 0:
        return 0.0

    ranks = np.arange(
        1,
        len(grades) + 1,
        dtype=float,
    )

    gains = np.power(2.0, grades) - 1.0
    discounts = np.log2(ranks + 1.0)

    return float(
        np.sum(gains / discounts)
    )


def ndcg_at_k(
    retrieved_grades,
    pooled_grades,
    k,
):
    actual_dcg = dcg_at_k(
        retrieved_grades,
        k,
    )

    ideal_grades = sorted(
        pooled_grades,
        reverse=True,
    )[:k]

    ideal_dcg = dcg_at_k(
        ideal_grades,
        k,
    )

    if ideal_dcg == 0:
        return np.nan

    return float(
        actual_dcg / ideal_dcg
    )


# ------------------------------------------------------------
# 4. Compute query-level metrics
# ------------------------------------------------------------

per_query_metric_records = []

for query_id in EXPECTED_QUERY_IDS:
    pooled_query = evaluation_df.loc[
        evaluation_df["query_id"].eq(query_id)
    ].copy()

    pooled_grades = (
        pooled_query["human_relevance"]
        .astype(int)
        .tolist()
    )

    total_useful_in_pool = int(
        pooled_query["human_relevance"]
        .ge(1)
        .sum()
    )

    total_direct_in_pool = int(
        pooled_query["human_relevance"]
        .eq(2)
        .sum()
    )

    for method, rank_column in METHOD_RANK_COLUMNS.items():
        ranked_query = (
            pooled_query.loc[
                pooled_query[rank_column].notna()
            ]
            .sort_values(
                rank_column,
                kind="stable",
            )
            .copy()
        )

        retrieved_grades = (
            ranked_query["human_relevance"]
            .astype(int)
            .tolist()
        )

        useful_binary = [
            int(grade >= 1)
            for grade in retrieved_grades
        ]

        direct_binary = [
            int(grade == 2)
            for grade in retrieved_grades
        ]

        first_useful_rank = next(
            (
                rank
                for rank, relevant
                in enumerate(
                    useful_binary,
                    start=1,
                )
                if relevant
            ),
            None,
        )

        first_direct_rank = next(
            (
                rank
                for rank, relevant
                in enumerate(
                    direct_binary,
                    start=1,
                )
                if relevant
            ),
            None,
        )

        record = {
            "query_id": query_id,
            "method": method,

            # Pool composition
            "pooled_candidates": len(pooled_query),
            "pooled_useful": total_useful_in_pool,
            "pooled_direct": total_direct_in_pool,

            # Retrieved grade counts
            "retrieved_irrelevant_at_10": int(
                sum(
                    grade == 0
                    for grade in retrieved_grades[:10]
                )
            ),
            "retrieved_partial_at_10": int(
                sum(
                    grade == 1
                    for grade in retrieved_grades[:10]
                )
            ),
            "retrieved_direct_at_10": int(
                sum(
                    grade == 2
                    for grade in retrieved_grades[:10]
                )
            ),

            # Useful evidence: grades 1 or 2
            "useful_precision_at_5": precision_at_k(
                useful_binary,
                5,
            ),
            "useful_precision_at_10": precision_at_k(
                useful_binary,
                10,
            ),
            "useful_pool_recall_at_5": recall_at_k(
                useful_binary,
                total_useful_in_pool,
                5,
            ),
            "useful_pool_recall_at_10": recall_at_k(
                useful_binary,
                total_useful_in_pool,
                10,
            ),
            "useful_hit_at_5": hit_at_k(
                useful_binary,
                5,
            ),
            "useful_hit_at_10": hit_at_k(
                useful_binary,
                10,
            ),
            "useful_mrr_at_10": reciprocal_rank_at_k(
                useful_binary,
                10,
            ),

            # Direct evidence: grade 2 only
            "direct_precision_at_5": precision_at_k(
                direct_binary,
                5,
            ),
            "direct_precision_at_10": precision_at_k(
                direct_binary,
                10,
            ),
            "direct_pool_recall_at_5": recall_at_k(
                direct_binary,
                total_direct_in_pool,
                5,
            ),
            "direct_pool_recall_at_10": recall_at_k(
                direct_binary,
                total_direct_in_pool,
                10,
            ),
            "direct_hit_at_5": hit_at_k(
                direct_binary,
                5,
            ),
            "direct_hit_at_10": hit_at_k(
                direct_binary,
                10,
            ),
            "direct_mrr_at_10": reciprocal_rank_at_k(
                direct_binary,
                10,
            ),

            # Graded relevance: 0, 1, 2
            "graded_ndcg_at_5": ndcg_at_k(
                retrieved_grades,
                pooled_grades,
                5,
            ),
            "graded_ndcg_at_10": ndcg_at_k(
                retrieved_grades,
                pooled_grades,
                10,
            ),

            # Diagnostic ranks
            "first_useful_rank": first_useful_rank,
            "first_direct_rank": first_direct_rank,
        }

        per_query_metric_records.append(record)


heldout_per_query_metrics = pd.DataFrame(
    per_query_metric_records
).sort_values(
    [
        "query_id",
        "method",
    ],
    kind="stable",
).reset_index(drop=True)


# ------------------------------------------------------------
# 5. Validate metric output
# ------------------------------------------------------------

assert len(heldout_per_query_metrics) == 36

assert (
    heldout_per_query_metrics["query_id"].nunique()
    == 12
)

assert (
    heldout_per_query_metrics["method"].nunique()
    == 3
)

bounded_metric_columns = [
    column
    for column in heldout_per_query_metrics.columns
    if (
        "_precision_" in column
        or "_recall_" in column
        or "_hit_" in column
        or "_mrr_" in column
        or "_ndcg_" in column
    )
]

for column in bounded_metric_columns:
    valid_values = (
        heldout_per_query_metrics[column]
        .dropna()
        .between(0.0, 1.0)
    )

    if not valid_values.all():
        raise AssertionError(
            f"Metric outside [0, 1]: {column}"
        )


# ------------------------------------------------------------
# 6. Save query-level metrics
# ------------------------------------------------------------

PER_QUERY_METRICS_PATH = (
    CHECKPOINT_DIR
    / "heldout_retrieval_per_query_metrics_v2.parquet"
)

PER_QUERY_METRICS_CSV_PATH = (
    CHECKPOINT_DIR
    / "heldout_retrieval_per_query_metrics_v2.csv"
)

heldout_per_query_metrics.to_parquet(
    PER_QUERY_METRICS_PATH,
    index=False,
)

heldout_per_query_metrics.to_csv(
    PER_QUERY_METRICS_CSV_PATH,
    index=False,
)


# ------------------------------------------------------------
# 7. Display compact query-level results
# ------------------------------------------------------------

compact_per_query_results = (
    heldout_per_query_metrics[
        [
            "query_id",
            "method",
            "pooled_useful",
            "pooled_direct",
            "retrieved_partial_at_10",
            "retrieved_direct_at_10",
            "useful_pool_recall_at_10",
            "direct_hit_at_5",
            "direct_mrr_at_10",
            "graded_ndcg_at_10",
            "first_direct_rank",
        ]
    ]
    .copy()
)

metric_display_columns = [
    "useful_pool_recall_at_10",
    "direct_hit_at_5",
    "direct_mrr_at_10",
    "graded_ndcg_at_10",
]

compact_per_query_results[
    metric_display_columns
] = (
    compact_per_query_results[
        metric_display_columns
    ]
    .round(4)
)

print("=" * 72)
print("HELD-OUT PER-QUERY RETRIEVAL METRICS")
print("=" * 72)
print("Query–method evaluations:", len(heldout_per_query_metrics))
print("Questions:", heldout_per_query_metrics["query_id"].nunique())
print("Methods:", heldout_per_query_metrics["method"].nunique())
print("All methods contain exact top-10 rankings: True")
print()
print("Saved:", PER_QUERY_METRICS_PATH)

display(compact_per_query_results)

HELD-OUT PER-QUERY RETRIEVAL METRICS
Query–method evaluations: 36
Questions: 12
Methods: 3
All methods contain exact top-10 rankings: True

Saved: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_per_query_metrics_v2.parquet


,query_id,method,pooled_useful,pooled_direct,retrieved_partial_at_10,retrieved_direct_at_10,useful_pool_recall_at_10,direct_hit_at_5,direct_mrr_at_10,graded_ndcg_at_10,first_direct_rank
0,H01,BM25,11,5,4,3,0.6364,1.0,0.3333,0.5486,3.0
1,H01,Dense,11,5,3,3,0.5455,1.0,0.5000,0.5723,2.0
2,H01,Hybrid RRF,11,5,3,3,0.5455,1.0,1.0000,0.6114,1.0
3,H02,BM25,13,5,5,1,0.4615,0.0,0.1667,0.3737,6.0
4,H02,Dense,13,5,3,5,0.6154,1.0,0.5000,0.8123,2.0
5,H02,Hybrid RRF,13,5,5,4,0.6923,1.0,1.0000,0.8971,1.0
6,H03,BM25,8,3,2,1,0.3750,0.0,0.1000,0.1806,10.0
7,H03,Dense,8,3,5,3,1.0000,1.0,0.3333,0.7684,3.0
8,H03,Hybrid RRF,8,3,3,2,0.6250,1.0,0.2500,0.4637,4.0
9,H04,BM25,9,2,5,1,0.6667,1.0,1.0000,0.6986,1.0


In [ ]:
# ============================================================
# HELD-OUT RETRIEVAL EVALUATION:
# MACRO AGGREGATION, CONFIDENCE INTERVALS,
# WINNER COUNTS, AND PAIRED METHOD COMPARISONS
# ============================================================

from itertools import product
from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Load the exact full-precision per-query metrics
# ------------------------------------------------------------

metrics_df = pd.read_parquet(
    PER_QUERY_METRICS_PATH
).copy()

EXPECTED_METHODS = [
    "BM25",
    "Dense",
    "Hybrid RRF",
]

PRIMARY_METRICS = {
    "useful_pool_recall_at_10":
        "Useful pool Recall@10",
    "direct_hit_at_5":
        "Direct Hit@5",
    "direct_mrr_at_10":
        "Direct MRR@10",
    "graded_ndcg_at_10":
        "Graded nDCG@10",
}

SECONDARY_METRICS = {
    "useful_precision_at_5":
        "Useful Precision@5",
    "useful_precision_at_10":
        "Useful Precision@10",
    "useful_pool_recall_at_5":
        "Useful pool Recall@5",
    "useful_hit_at_5":
        "Useful Hit@5",
    "useful_hit_at_10":
        "Useful Hit@10",
    "useful_mrr_at_10":
        "Useful MRR@10",
    "direct_precision_at_5":
        "Direct Precision@5",
    "direct_precision_at_10":
        "Direct Precision@10",
    "direct_pool_recall_at_5":
        "Direct pool Recall@5",
    "direct_pool_recall_at_10":
        "Direct pool Recall@10",
    "direct_hit_at_10":
        "Direct Hit@10",
    "graded_ndcg_at_5":
        "Graded nDCG@5",
}

ALL_REPORTED_METRICS = {
    **PRIMARY_METRICS,
    **SECONDARY_METRICS,
}

assert len(metrics_df) == 36
assert set(metrics_df["method"]) == set(EXPECTED_METHODS)
assert metrics_df["query_id"].nunique() == 12

method_query_counts = (
    metrics_df
    .groupby("method")["query_id"]
    .nunique()
    .reindex(EXPECTED_METHODS)
)

assert method_query_counts.eq(12).all()


# ------------------------------------------------------------
# 2. Deterministic bootstrap confidence intervals
# ------------------------------------------------------------

BOOTSTRAP_ITERATIONS = 20_000
BOOTSTRAP_SEED = 42
CONFIDENCE_LEVEL = 0.95


def bootstrap_mean_ci(
    values,
    iterations=BOOTSTRAP_ITERATIONS,
    confidence_level=CONFIDENCE_LEVEL,
    seed=BOOTSTRAP_SEED,
):
    """
    Query-level nonparametric bootstrap CI for a macro mean.

    The query is the resampling unit. Individual retrieved passages
    are not treated as independent observations.
    """

    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)

    sampled_indices = rng.integers(
        low=0,
        high=len(values),
        size=(iterations, len(values)),
    )

    bootstrap_means = values[
        sampled_indices
    ].mean(axis=1)

    alpha = 1.0 - confidence_level

    lower = np.quantile(
        bootstrap_means,
        alpha / 2.0,
    )

    upper = np.quantile(
        bootstrap_means,
        1.0 - alpha / 2.0,
    )

    return float(lower), float(upper)


# ------------------------------------------------------------
# 3. Build macro-aggregated metric table
# ------------------------------------------------------------

aggregate_records = []

for method_index, method in enumerate(EXPECTED_METHODS):
    method_rows = (
        metrics_df.loc[
            metrics_df["method"].eq(method)
        ]
        .sort_values("query_id")
        .copy()
    )

    for metric_index, (
        metric_column,
        metric_label,
    ) in enumerate(ALL_REPORTED_METRICS.items()):

        values = (
            method_rows[metric_column]
            .astype(float)
            .to_numpy()
        )

        ci_lower, ci_upper = bootstrap_mean_ci(
            values,
            seed=(
                BOOTSTRAP_SEED
                + method_index * 100
                + metric_index
            ),
        )

        aggregate_records.append(
            {
                "method": method,
                "metric": metric_column,
                "metric_label": metric_label,
                "query_count": int(
                    np.isfinite(values).sum()
                ),
                "macro_mean": float(
                    np.nanmean(values)
                ),
                "median": float(
                    np.nanmedian(values)
                ),
                "standard_deviation": float(
                    np.nanstd(values, ddof=1)
                ),
                "ci_95_lower": ci_lower,
                "ci_95_upper": ci_upper,
            }
        )

heldout_aggregate_metrics = pd.DataFrame(
    aggregate_records
)


# ------------------------------------------------------------
# 4. Create compact headline table
# ------------------------------------------------------------

headline_results = (
    heldout_aggregate_metrics.loc[
        heldout_aggregate_metrics[
            "metric"
        ].isin(PRIMARY_METRICS)
    ]
    .pivot(
        index="metric_label",
        columns="method",
        values="macro_mean",
    )
    .reindex(
        list(PRIMARY_METRICS.values())
    )
    .reindex(
        columns=EXPECTED_METHODS
    )
)

headline_results["Best method"] = (
    headline_results[
        EXPECTED_METHODS
    ]
    .idxmax(axis=1)
)

headline_results_display = (
    headline_results.copy()
)

headline_results_display[
    EXPECTED_METHODS
] = (
    headline_results_display[
        EXPECTED_METHODS
    ]
    .round(4)
)


# ------------------------------------------------------------
# 5. Build mean-with-confidence-interval table
# ------------------------------------------------------------

primary_ci_rows = []

for metric_column, metric_label in PRIMARY_METRICS.items():
    for method in EXPECTED_METHODS:
        row = heldout_aggregate_metrics.loc[
            heldout_aggregate_metrics[
                "metric"
            ].eq(metric_column)
            & heldout_aggregate_metrics[
                "method"
            ].eq(method)
        ].iloc[0]

        primary_ci_rows.append(
            {
                "metric": metric_label,
                "method": method,
                "macro_mean": row["macro_mean"],
                "ci_95_lower": row["ci_95_lower"],
                "ci_95_upper": row["ci_95_upper"],
                "mean_and_95_ci": (
                    f"{row['macro_mean']:.4f} "
                    f"[{row['ci_95_lower']:.4f}, "
                    f"{row['ci_95_upper']:.4f}]"
                ),
            }
        )

primary_metric_intervals = pd.DataFrame(
    primary_ci_rows
)


# ------------------------------------------------------------
# 6. Count query-level wins and ties
# ------------------------------------------------------------

WIN_TOLERANCE = 1e-12

winner_records = []

for metric_column, metric_label in PRIMARY_METRICS.items():
    metric_pivot = (
        metrics_df.pivot(
            index="query_id",
            columns="method",
            values=metric_column,
        )
        .reindex(columns=EXPECTED_METHODS)
    )

    for query_id, row in metric_pivot.iterrows():
        best_value = row.max()

        winning_methods = [
            method
            for method in EXPECTED_METHODS
            if np.isclose(
                row[method],
                best_value,
                atol=WIN_TOLERANCE,
                rtol=0.0,
            )
        ]

        winner_records.append(
            {
                "query_id": query_id,
                "metric": metric_column,
                "metric_label": metric_label,
                "best_value": float(best_value),
                "winner_count": len(winning_methods),
                "winner_type": (
                    "unique"
                    if len(winning_methods) == 1
                    else "tie"
                ),
                "winning_methods": " | ".join(
                    winning_methods
                ),
            }
        )

query_level_winners = pd.DataFrame(
    winner_records
)

unique_win_records = []

for metric_column, metric_label in PRIMARY_METRICS.items():
    metric_winners = query_level_winners.loc[
        query_level_winners["metric"].eq(
            metric_column
        )
    ]

    for method in EXPECTED_METHODS:
        unique_wins = int(
            (
                metric_winners["winner_type"].eq(
                    "unique"
                )
                & metric_winners[
                    "winning_methods"
                ].eq(method)
            ).sum()
        )

        tie_participations = int(
            (
                metric_winners["winner_type"].eq(
                    "tie"
                )
                & metric_winners[
                    "winning_methods"
                ].str.split(" | ", regex=False)
                .apply(lambda winners: method in winners)
            ).sum()
        )

        unique_win_records.append(
            {
                "metric": metric_label,
                "method": method,
                "unique_query_wins": unique_wins,
                "tie_participations": tie_participations,
            }
        )

winner_summary = pd.DataFrame(
    unique_win_records
)


# ------------------------------------------------------------
# 7. Exact paired sign-flip/randomization test
# ------------------------------------------------------------

def exact_paired_randomization_test(
    differences,
):
    """
    Two-sided exact paired randomization test of mean difference.

    With 12 queries, all 2^12 sign assignments are feasible.
    This does not assume normally distributed query metrics.
    """

    differences = np.asarray(
        differences,
        dtype=float,
    )

    differences = differences[
        np.isfinite(differences)
    ]

    observed_mean_difference = float(
        differences.mean()
    )

    absolute_differences = np.abs(
        differences
    )

    possible_means = []

    for signs in product(
        [-1.0, 1.0],
        repeat=len(differences),
    ):
        signed_differences = (
            absolute_differences
            * np.asarray(signs)
        )

        possible_means.append(
            signed_differences.mean()
        )

    possible_means = np.asarray(
        possible_means,
        dtype=float,
    )

    p_value = float(
        np.mean(
            np.abs(possible_means)
            >= (
                abs(observed_mean_difference)
                - 1e-15
            )
        )
    )

    return p_value


def paired_bootstrap_difference_ci(
    differences,
    iterations=BOOTSTRAP_ITERATIONS,
    confidence_level=CONFIDENCE_LEVEL,
    seed=BOOTSTRAP_SEED,
):
    differences = np.asarray(
        differences,
        dtype=float,
    )

    differences = differences[
        np.isfinite(differences)
    ]

    rng = np.random.default_rng(seed)

    sampled_indices = rng.integers(
        low=0,
        high=len(differences),
        size=(iterations, len(differences)),
    )

    bootstrap_differences = differences[
        sampled_indices
    ].mean(axis=1)

    alpha = 1.0 - confidence_level

    return (
        float(
            np.quantile(
                bootstrap_differences,
                alpha / 2.0,
            )
        ),
        float(
            np.quantile(
                bootstrap_differences,
                1.0 - alpha / 2.0,
            )
        ),
    )


# Hybrid is compared to both individual retrievers.
PAIRWISE_COMPARISONS = [
    ("Hybrid RRF", "BM25"),
    ("Hybrid RRF", "Dense"),
    ("Dense", "BM25"),
]

paired_comparison_records = []

for metric_index, (
    metric_column,
    metric_label,
) in enumerate(PRIMARY_METRICS.items()):

    metric_pivot = (
        metrics_df.pivot(
            index="query_id",
            columns="method",
            values=metric_column,
        )
        .reindex(
            columns=EXPECTED_METHODS
        )
        .sort_index()
    )

    for comparison_index, (
        method_a,
        method_b,
    ) in enumerate(PAIRWISE_COMPARISONS):

        differences = (
            metric_pivot[method_a]
            - metric_pivot[method_b]
        ).to_numpy(dtype=float)

        difference_ci_lower, difference_ci_upper = (
            paired_bootstrap_difference_ci(
                differences,
                seed=(
                    BOOTSTRAP_SEED
                    + metric_index * 100
                    + comparison_index
                ),
            )
        )

        method_a_wins = int(
            np.sum(differences > WIN_TOLERANCE)
        )

        method_b_wins = int(
            np.sum(differences < -WIN_TOLERANCE)
        )

        ties = int(
            np.sum(
                np.isclose(
                    differences,
                    0.0,
                    atol=WIN_TOLERANCE,
                    rtol=0.0,
                )
            )
        )

        paired_comparison_records.append(
            {
                "metric": metric_column,
                "metric_label": metric_label,
                "method_a": method_a,
                "method_b": method_b,
                "mean_a_minus_b": float(
                    differences.mean()
                ),
                "difference_ci_95_lower":
                    difference_ci_lower,
                "difference_ci_95_upper":
                    difference_ci_upper,
                "method_a_query_wins":
                    method_a_wins,
                "method_b_query_wins":
                    method_b_wins,
                "ties": ties,
                "exact_randomization_p_value":
                    exact_paired_randomization_test(
                        differences
                    ),
            }
        )

heldout_paired_comparisons = pd.DataFrame(
    paired_comparison_records
)

heldout_paired_comparisons[
    "difference_ci_excludes_zero"
] = (
    (
        heldout_paired_comparisons[
            "difference_ci_95_lower"
        ] > 0
    )
    | (
        heldout_paired_comparisons[
            "difference_ci_95_upper"
        ] < 0
    )
)


# ------------------------------------------------------------
# 8. Add method-level failure counts
# ------------------------------------------------------------

failure_summary = (
    metrics_df.groupby("method")
    .agg(
        queries=("query_id", "nunique"),
        no_useful_in_top_5=(
            "useful_hit_at_5",
            lambda values: int(
                values.eq(0).sum()
            ),
        ),
        no_useful_in_top_10=(
            "useful_hit_at_10",
            lambda values: int(
                values.eq(0).sum()
            ),
        ),
        no_direct_in_top_5=(
            "direct_hit_at_5",
            lambda values: int(
                values.eq(0).sum()
            ),
        ),
        no_direct_in_top_10=(
            "direct_hit_at_10",
            lambda values: int(
                values.eq(0).sum()
            ),
        ),
        mean_first_direct_rank=(
            "first_direct_rank",
            "mean",
        ),
        median_first_direct_rank=(
            "first_direct_rank",
            "median",
        ),
    )
    .reindex(EXPECTED_METHODS)
    .reset_index()
)

failure_summary[
    [
        "mean_first_direct_rank",
        "median_first_direct_rank",
    ]
] = (
    failure_summary[
        [
            "mean_first_direct_rank",
            "median_first_direct_rank",
        ]
    ]
    .round(4)
)


# ------------------------------------------------------------
# 9. Save all aggregate artifacts
# ------------------------------------------------------------

AGGREGATE_METRICS_PATH = (
    CHECKPOINT_DIR
    / "heldout_retrieval_aggregate_metrics_v2.csv"
)

PAIRWISE_COMPARISONS_PATH = (
    CHECKPOINT_DIR
    / "heldout_retrieval_paired_comparisons_v2.csv"
)

WINNER_SUMMARY_PATH = (
    CHECKPOINT_DIR
    / "heldout_retrieval_winner_summary_v2.csv"
)

QUERY_WINNERS_PATH = (
    CHECKPOINT_DIR
    / "heldout_retrieval_query_winners_v2.csv"
)

FAILURE_SUMMARY_PATH = (
    CHECKPOINT_DIR
    / "heldout_retrieval_failure_summary_v2.csv"
)

SUMMARY_JSON_PATH = (
    CHECKPOINT_DIR
    / "heldout_retrieval_summary_v2.json"
)

heldout_aggregate_metrics.to_csv(
    AGGREGATE_METRICS_PATH,
    index=False,
)

heldout_paired_comparisons.to_csv(
    PAIRWISE_COMPARISONS_PATH,
    index=False,
)

winner_summary.to_csv(
    WINNER_SUMMARY_PATH,
    index=False,
)

query_level_winners.to_csv(
    QUERY_WINNERS_PATH,
    index=False,
)

failure_summary.to_csv(
    FAILURE_SUMMARY_PATH,
    index=False,
)


def dataframe_records_for_json(dataframe):
    return json.loads(
        dataframe.to_json(
            orient="records",
            double_precision=15,
        )
    )


summary_payload = {
    "evaluation_version": "heldout_v2",
    "question_count": 12,
    "judgment_count": 214,
    "methods": EXPECTED_METHODS,
    "primary_metric_definitions": PRIMARY_METRICS,
    "recall_scope": (
        "pool-relative recall over the union of "
        "BM25, dense, and hybrid top-10 results"
    ),
    "bootstrap": {
        "iterations": BOOTSTRAP_ITERATIONS,
        "seed": BOOTSTRAP_SEED,
        "confidence_level": CONFIDENCE_LEVEL,
        "resampling_unit": "query",
    },
    "aggregate_primary_metrics":
        dataframe_records_for_json(
            heldout_aggregate_metrics.loc[
                heldout_aggregate_metrics[
                    "metric"
                ].isin(PRIMARY_METRICS)
            ]
        ),
    "paired_primary_comparisons":
        dataframe_records_for_json(
            heldout_paired_comparisons
        ),
    "failure_summary":
        dataframe_records_for_json(
            failure_summary
        ),
}

with open(
    SUMMARY_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary_payload,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 10. Validate saved aggregate outputs
# ------------------------------------------------------------

assert AGGREGATE_METRICS_PATH.exists()
assert PAIRWISE_COMPARISONS_PATH.exists()
assert WINNER_SUMMARY_PATH.exists()
assert QUERY_WINNERS_PATH.exists()
assert FAILURE_SUMMARY_PATH.exists()
assert SUMMARY_JSON_PATH.exists()

assert len(
    heldout_aggregate_metrics.loc[
        heldout_aggregate_metrics[
            "metric"
        ].isin(PRIMARY_METRICS)
    ]
) == 12

assert len(heldout_paired_comparisons) == 12


# ------------------------------------------------------------
# 11. Display final aggregation outputs
# ------------------------------------------------------------

paired_display = (
    heldout_paired_comparisons[
        [
            "metric_label",
            "method_a",
            "method_b",
            "mean_a_minus_b",
            "difference_ci_95_lower",
            "difference_ci_95_upper",
            "method_a_query_wins",
            "method_b_query_wins",
            "ties",
            "exact_randomization_p_value",
        ]
    ]
    .copy()
)

paired_numeric_columns = [
    "mean_a_minus_b",
    "difference_ci_95_lower",
    "difference_ci_95_upper",
    "exact_randomization_p_value",
]

paired_display[
    paired_numeric_columns
] = (
    paired_display[
        paired_numeric_columns
    ]
    .round(4)
)

print("=" * 72)
print("HELD-OUT RETRIEVAL AGGREGATION COMPLETE")
print("=" * 72)
print(f"Questions:                 {metrics_df['query_id'].nunique()}")
print(f"Methods:                   {metrics_df['method'].nunique()}")
print(f"Bootstrap iterations:      {BOOTSTRAP_ITERATIONS:,}")
print("Bootstrap unit:            query")
print("Paired test:               exact sign-flip randomization")
print("Recall interpretation:     pool-relative")
print()
print("Saved aggregate metrics:  ", AGGREGATE_METRICS_PATH)
print("Saved pairwise results:   ", PAIRWISE_COMPARISONS_PATH)
print("Saved summary JSON:       ", SUMMARY_JSON_PATH)

print()
print("PRIMARY HELD-OUT RESULTS")
display(headline_results_display)

print()
print("PRIMARY METRIC 95% BOOTSTRAP INTERVALS")
display(
    primary_metric_intervals[
        [
            "metric",
            "method",
            "mean_and_95_ci",
        ]
    ]
)

print()
print("QUERY-LEVEL WINNER COUNTS")
display(winner_summary)

print()
print("PAIRED QUERY-LEVEL COMPARISONS")
display(paired_display)

print()
print("METHOD FAILURE SUMMARY")
display(failure_summary)

HELD-OUT RETRIEVAL AGGREGATION COMPLETE
Questions:                 12
Methods:                   3
Bootstrap iterations:      20,000
Bootstrap unit:            query
Paired test:               exact sign-flip randomization
Recall interpretation:     pool-relative

Saved aggregate metrics:   /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_aggregate_metrics_v2.csv
Saved pairwise results:    /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_paired_comparisons_v2.csv
Saved summary JSON:        /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_summary_v2.json

PRIMARY HELD-OUT RESULTS


method,BM25,Dense,Hybrid RRF,Best method
metric_label,,,,
Useful pool Recall@10,0.5686,0.7456,0.6071,Dense
Direct Hit@5,0.6667,0.7500,0.9167,Hybrid RRF
Direct MRR@10,0.4944,0.4510,0.6736,Hybrid RRF
Graded nDCG@10,0.5118,0.5821,0.5986,Hybrid RRF



PRIMARY METRIC 95% BOOTSTRAP INTERVALS


,metric,method,mean_and_95_ci
0,Useful pool Recall@10,BM25,"0.5686 [0.4339, 0.7089]"
1,Useful pool Recall@10,Dense,"0.7456 [0.6056, 0.8753]"
2,Useful pool Recall@10,Hybrid RRF,"0.6071 [0.4656, 0.7289]"
3,Direct Hit@5,BM25,"0.6667 [0.4167, 0.9167]"
4,Direct Hit@5,Dense,"0.7500 [0.5000, 1.0000]"
5,Direct Hit@5,Hybrid RRF,"0.9167 [0.7500, 1.0000]"
6,Direct MRR@10,BM25,"0.4944 [0.2931, 0.7083]"
7,Direct MRR@10,Dense,"0.4510 [0.2945, 0.6204]"
8,Direct MRR@10,Hybrid RRF,"0.6736 [0.4722, 0.8611]"
9,Graded nDCG@10,BM25,"0.5118 [0.3689, 0.6629]"



QUERY-LEVEL WINNER COUNTS


,metric,method,unique_query_wins,tie_participations
0,Useful pool Recall@10,BM25,3,1
1,Useful pool Recall@10,Dense,5,2
2,Useful pool Recall@10,Hybrid RRF,2,2
3,Direct Hit@5,BM25,0,9
4,Direct Hit@5,Dense,0,10
5,Direct Hit@5,Hybrid RRF,0,12
6,Direct MRR@10,BM25,3,2
7,Direct MRR@10,Dense,2,2
8,Direct MRR@10,Hybrid RRF,4,3
9,Graded nDCG@10,BM25,3,0



PAIRED QUERY-LEVEL COMPARISONS


,metric_label,method_a,method_b,mean_a_minus_b,difference_ci_95_lower,difference_ci_95_upper,method_a_query_wins,method_b_query_wins,ties,exact_randomization_p_value
0,Useful pool Recall@10,Hybrid RRF,BM25,0.0385,-0.1021,0.1668,6,4,2,0.6191
1,Useful pool Recall@10,Hybrid RRF,Dense,-0.1385,-0.3144,0.0168,4,5,3,0.1562
2,Useful pool Recall@10,Dense,BM25,0.1770,-0.0614,0.3965,7,3,2,0.1738
3,Direct Hit@5,Hybrid RRF,BM25,0.2500,0.0000,0.5000,3,0,9,0.2500
4,Direct Hit@5,Hybrid RRF,Dense,0.1667,0.0000,0.4167,2,0,10,0.5000
5,Direct Hit@5,Dense,BM25,0.0833,-0.2500,0.4167,3,2,7,1.0000
6,Direct MRR@10,Hybrid RRF,BM25,0.1792,-0.0833,0.4236,7,3,2,0.2285
7,Direct MRR@10,Hybrid RRF,Dense,0.2226,0.0159,0.4255,7,3,2,0.0781
8,Direct MRR@10,Dense,BM25,-0.0434,-0.3073,0.2170,5,5,2,0.7656
9,Graded nDCG@10,Hybrid RRF,BM25,0.0868,-0.0833,0.2484,8,4,0,0.3496



METHOD FAILURE SUMMARY


,method,queries,no_useful_in_top_5,no_useful_in_top_10,no_direct_in_top_5,no_direct_in_top_10,mean_first_direct_rank,median_first_direct_rank
0,BM25,12,2,0,4,0,3.7500,3.5
1,Dense,12,1,0,3,0,3.6667,2.0
2,Hybrid RRF,12,1,1,1,1,1.7273,1.0


In [ ]:
# ============================================================
# HELD-OUT RETRIEVAL EVALUATION:
# QUERY-METADATA AUDIT AND H12 FAILURE INSPECTION
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Load frozen judgments and per-query metrics
# ------------------------------------------------------------

judgments_df = pd.read_parquet(
    FINAL_JUDGMENTS_PATH
).copy()

metrics_df = pd.read_parquet(
    PER_QUERY_METRICS_PATH
).copy()


# ------------------------------------------------------------
# 2. Inspect available evaluation columns
# ------------------------------------------------------------

print("=" * 72)
print("AVAILABLE FROZEN-JUDGMENT COLUMNS")
print("=" * 72)

for column_number, column_name in enumerate(
    judgments_df.columns,
    start=1,
):
    print(f"{column_number:>2}. {column_name}")


# ------------------------------------------------------------
# 3. Identify likely query-level metadata columns
# ------------------------------------------------------------

possible_query_metadata_columns = [
    "query_id",
    "query",
    "query_text",
    "question",
    "question_text",
    "ticker",
    "company",
    "filing_type",
    "form_type",
    "category",
    "query_category",
    "question_category",
    "difficulty",
    "query_type",
    "question_type",
    "expected_answer",
    "reference_answer",
]

available_query_metadata_columns = [
    column
    for column in possible_query_metadata_columns
    if column in judgments_df.columns
]

if "query_id" not in available_query_metadata_columns:
    raise AssertionError(
        "The frozen judgment table is missing query_id."
    )

query_metadata = (
    judgments_df[
        available_query_metadata_columns
    ]
    .drop_duplicates()
    .sort_values("query_id")
    .reset_index(drop=True)
)

print()
print("=" * 72)
print("AVAILABLE QUERY-LEVEL METADATA")
print("=" * 72)
print(
    "Detected metadata columns:",
    available_query_metadata_columns,
)
print()

display(query_metadata)


# ------------------------------------------------------------
# 4. Show complete H12 pooled evidence
# ------------------------------------------------------------

H12_ID = "H12"

h12_rows = judgments_df.loc[
    judgments_df["query_id"].eq(H12_ID)
].copy()

if h12_rows.empty:
    raise AssertionError("H12 was not found.")


preferred_diagnostic_columns = [
    "query_id",
    "query",
    "query_text",
    "question",
    "question_text",
    "ticker",
    "company",
    "filing_type",
    "form_type",
    "category",
    "query_category",
    "question_category",
    "corpus_row",
    "chunk_id",
    "human_relevance",
    "bm25_rank",
    "dense_rank",
    "hybrid_rank",
    "bm25_score",
    "dense_score",
    "hybrid_score",
    "rrf_score",
    "section",
    "section_name",
    "item",
    "page",
    "text",
    "chunk_text",
]

h12_display_columns = [
    column
    for column in preferred_diagnostic_columns
    if column in h12_rows.columns
]

h12_sort_columns = [
    column
    for column in [
        "human_relevance",
        "hybrid_rank",
        "dense_rank",
        "bm25_rank",
    ]
    if column in h12_rows.columns
]

h12_sort_ascending = [
    False if column == "human_relevance" else True
    for column in h12_sort_columns
]

h12_diagnostic = (
    h12_rows[
        h12_display_columns
    ]
    .sort_values(
        h12_sort_columns,
        ascending=h12_sort_ascending,
        na_position="last",
        kind="stable",
    )
    .reset_index(drop=True)
)

print()
print("=" * 72)
print("H12 COMPLETE POOLED-EVIDENCE DIAGNOSTIC")
print("=" * 72)
print("Pooled candidates:", len(h12_rows))
print(
    "Useful pooled candidates:",
    int(h12_rows["human_relevance"].ge(1).sum()),
)
print(
    "Direct pooled candidates:",
    int(h12_rows["human_relevance"].eq(2).sum()),
)
print()

display(h12_diagnostic)


# ------------------------------------------------------------
# 5. Isolate the relevant H12 passages
# ------------------------------------------------------------

h12_relevant = h12_rows.loc[
    h12_rows["human_relevance"].ge(1),
    h12_display_columns,
].copy()

h12_relevant = h12_relevant.sort_values(
    [
        column
        for column in [
            "human_relevance",
            "dense_rank",
            "bm25_rank",
            "hybrid_rank",
        ]
        if column in h12_relevant.columns
    ],
    ascending=[
        False
        if column == "human_relevance"
        else True
        for column in [
            column
            for column in [
                "human_relevance",
                "dense_rank",
                "bm25_rank",
                "hybrid_rank",
            ]
            if column in h12_relevant.columns
        ]
    ],
    na_position="last",
    kind="stable",
)

print()
print("=" * 72)
print("H12 RELEVANT PASSAGES ONLY")
print("=" * 72)

display(h12_relevant)


# ------------------------------------------------------------
# 6. Display all method rankings for H12
# ------------------------------------------------------------

method_rank_columns = {
    "BM25": "bm25_rank",
    "Dense": "dense_rank",
    "Hybrid RRF": "hybrid_rank",
}

ranking_text_columns = [
    column
    for column in [
        "query_id",
        "corpus_row",
        "chunk_id",
        "human_relevance",
        "section",
        "section_name",
        "item",
        "page",
        "text",
        "chunk_text",
    ]
    if column in h12_rows.columns
]

h12_method_rankings = {}

for method, rank_column in method_rank_columns.items():
    method_columns = list(
        dict.fromkeys(
            ranking_text_columns
            + [rank_column]
        )
    )

    ranked_results = (
        h12_rows.loc[
            h12_rows[rank_column].notna(),
            method_columns,
        ]
        .sort_values(
            rank_column,
            kind="stable",
        )
        .reset_index(drop=True)
    )

    h12_method_rankings[method] = ranked_results

    print()
    print("-" * 72)
    print(f"H12 — {method.upper()} TOP 10")
    print("-" * 72)

    display(ranked_results)


# ------------------------------------------------------------
# 7. Compare H12 metric values directly
# ------------------------------------------------------------

h12_metrics = (
    metrics_df.loc[
        metrics_df["query_id"].eq(H12_ID),
        [
            "query_id",
            "method",
            "retrieved_irrelevant_at_10",
            "retrieved_partial_at_10",
            "retrieved_direct_at_10",
            "useful_precision_at_10",
            "useful_pool_recall_at_10",
            "direct_hit_at_5",
            "direct_hit_at_10",
            "direct_mrr_at_10",
            "graded_ndcg_at_10",
            "first_direct_rank",
        ],
    ]
    .copy()
)

metric_columns_to_round = [
    column
    for column in h12_metrics.columns
    if column not in [
        "query_id",
        "method",
    ]
]

h12_metrics[
    metric_columns_to_round
] = (
    h12_metrics[
        metric_columns_to_round
    ]
    .round(4)
)

print()
print("=" * 72)
print("H12 METHOD-LEVEL METRICS")
print("=" * 72)

display(h12_metrics)

AVAILABLE FROZEN-JUDGMENT COLUMNS
 1. query_id
 2. corpus_row
 3. chunk_id
 4. document_id
 5. chunk_index
 6. text
 7. ticker
 8. company
 9. form_type
10. filing_date
11. report_date
12. accession_number
13. source_url
14. bm25_rank
15. dense_rank
16. hybrid_rank
17. bm25_score
18. dense_score
19. hybrid_score
20. query
21. question_type
22. difficulty
23. report_year
24. retrieved_by_count
25. review_order
26. human_relevance
27. review_reason

AVAILABLE QUERY-LEVEL METADATA
Detected metadata columns: ['query_id', 'query', 'ticker', 'company', 'form_type', 'difficulty', 'question_type']



,query_id,query,ticker,company,form_type,difficulty,question_type
0,H01,"According to Apple's 2025 Form 10-K, what fact...",AAPL,Apple Inc.,10-K,medium,reported_driver
1,H02,"According to Apple's 2024 Form 10-K, how did n...",AAPL,Apple Inc.,10-K,medium,geographic_or_segment_comparison
2,H03,"According to Microsoft's 2025 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,medium,segment_driver
3,H04,"According to Microsoft's 2024 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,medium,capital_expenditure
4,H05,"According to Amazon's 2024 Form 10-K, how did ...",AMZN,AMAZON COM INC,10-K,medium,segment_comparison
5,H06,"According to Amazon's 2023 Form 10-K, what fac...",AMZN,AMAZON COM INC,10-K,hard,cash_flow_driver
6,H07,"According to Alphabet's 2024 Form 10-K, what f...",GOOGL,Alphabet Inc.,10-K,medium,revenue_driver
7,H08,"According to Alphabet's 2023 Form 10-K, what r...",GOOGL,Alphabet Inc.,10-K,hard,risk_factor
8,H09,"According to NVIDIA's 2025 Form 10-K, what fac...",NVDA,NVIDIA CORP,10-K,medium,segment_driver
9,H10,"According to NVIDIA's 2024 Form 10-K, what sup...",NVDA,NVIDIA CORP,10-K,hard,supply_risk



H12 COMPLETE POOLED-EVIDENCE DIAGNOSTIC
Pooled candidates: 20
Useful pooled candidates: 4
Direct pooled candidates: 2



,query_id,query,ticker,company,form_type,corpus_row,chunk_id,human_relevance,bm25_rank,dense_rank,hybrid_rank,bm25_score,dense_score,hybrid_score,text
0,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2402,msft_10k_000095017023035122__chunk_0055,2,<NA>,7,<NA>,NaN,0.696734,NaN,"At times, to achieve commercial objectives, we..."
1,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2396,msft_10k_000095017023035122__chunk_0049,2,6,<NA>,<NA>,16.895615,NaN,NaN,It may take longer than expected to realize th...
2,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2394,msft_10k_000095017023035122__chunk_0047,1,<NA>,8,<NA>,NaN,0.691760,NaN,These include unauthorized use of accounts thr...
3,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2403,msft_10k_000095017023035122__chunk_0056,1,<NA>,10,<NA>,NaN,0.686068,NaN,Despite our efforts to protect customer and us...
4,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2425,msft_10k_000095017023035122__chunk_0078,0,4,3,1,17.678169,0.704512,0.059715,Our Board of Directors declared the following ...
5,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2351,msft_10k_000095017023035122__chunk_0004,0,<NA>,5,2,NaN,0.698241,0.052962,These forward-looking statements generally are...
6,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2426,msft_10k_000095017023035122__chunk_0079,0,10,6,3,15.949256,0.697655,0.052778,OVERVIEW Microsoft is a technology company who...
7,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2532,msft_10k_000095017023035122__chunk_0185,0,7,<NA>,4,16.850392,NaN,0.045546,"Also, projections of any evaluation of effecti..."
8,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2538,msft_10k_000095017023035122__chunk_0191,0,<NA>,<NA>,5,NaN,NaN,0.044798,"Bank National Association, as Trustee, to the ..."
9,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2406,msft_10k_000095017023035122__chunk_0059,0,<NA>,<NA>,6,NaN,NaN,0.044089,Some AI scenarios present ethical issues or ma...



H12 RELEVANT PASSAGES ONLY


,query_id,query,ticker,company,form_type,corpus_row,chunk_id,human_relevance,bm25_rank,dense_rank,hybrid_rank,bm25_score,dense_score,hybrid_score,text
201,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2402,msft_10k_000095017023035122__chunk_0055,2,<NA>,7,<NA>,NaN,0.696734,NaN,"At times, to achieve commercial objectives, we..."
196,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2396,msft_10k_000095017023035122__chunk_0049,2,6,<NA>,<NA>,16.895615,NaN,NaN,It may take longer than expected to realize th...
197,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2394,msft_10k_000095017023035122__chunk_0047,1,<NA>,8,<NA>,NaN,0.691760,NaN,These include unauthorized use of accounts thr...
195,H12,"According to Microsoft's 2023 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,2403,msft_10k_000095017023035122__chunk_0056,1,<NA>,10,<NA>,NaN,0.686068,NaN,Despite our efforts to protect customer and us...


... (4 outputs left out)


------------------------------------------------------------------------
H12 — HYBRID RRF TOP 10
------------------------------------------------------------------------


,query_id,corpus_row,chunk_id,human_relevance,text,hybrid_rank
0,H12,2425,msft_10k_000095017023035122__chunk_0078,0,Our Board of Directors declared the following ...,1
1,H12,2351,msft_10k_000095017023035122__chunk_0004,0,These forward-looking statements generally are...,2
2,H12,2426,msft_10k_000095017023035122__chunk_0079,0,OVERVIEW Microsoft is a technology company who...,3
3,H12,2532,msft_10k_000095017023035122__chunk_0185,0,"Also, projections of any evaluation of effecti...",4
4,H12,2538,msft_10k_000095017023035122__chunk_0191,0,"Bank National Association, as Trustee, to the ...",5
5,H12,2406,msft_10k_000095017023035122__chunk_0059,0,Some AI scenarios present ethical issues or ma...,6
6,H12,2429,msft_10k_000095017023035122__chunk_0082,0,"Third, we are consolidating our leases to crea...",7
7,H12,2523,msft_10k_000095017023035122__chunk_0176,0,"Long-lived assets, excluding financial instrum...",8
8,H12,2361,msft_10k_000095017023035122__chunk_0014,0,We are 28 percent of the way for Hispanic and ...,9
9,H12,2539,msft_10k_000095017023035122__chunk_0192,0,"and The Bank of New York Mellon Trust Company,...",10



H12 METHOD-LEVEL METRICS


,query_id,method,retrieved_irrelevant_at_10,retrieved_partial_at_10,retrieved_direct_at_10,useful_precision_at_10,useful_pool_recall_at_10,direct_hit_at_5,direct_hit_at_10,direct_mrr_at_10,graded_ndcg_at_10,first_direct_rank
33,H12,BM25,9,0,1,0.1,0.25,0.0,1.0,0.1667,0.1835,6.0
34,H12,Dense,7,2,1,0.3,0.75,0.0,1.0,0.1429,0.2755,7.0
35,H12,Hybrid RRF,10,0,0,0.0,0.00,0.0,0.0,0.0000,0.0000,NaN


In [ ]:
# ============================================================
# H12 FULL FUSION-CANDIDATE RECOVERY
# BM25 TOP 20 + DENSE TOP 30
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

H12_ID = "H12"

REQUIRED_DEPTHS = {
    "BM25": 20,
    "Dense": 30,
}

METHOD_ALIASES = {
    "bm25": "BM25",
    "sparse": "BM25",
    "dense": "Dense",
    "faiss": "Dense",
    "semantic": "Dense",
}

IDENTIFIER_CANDIDATES = [
    "corpus_row",
    "chunk_id",
    "document_id",
    "doc_id",
]

TEXT_CANDIDATES = [
    "chunk_text",
    "text",
    "passage",
    "content",
]


# ------------------------------------------------------------
# 2. Load the frozen H12 judgments
# ------------------------------------------------------------

judgments_df = pd.read_parquet(
    FINAL_JUDGMENTS_PATH
).copy()

h12_judgments = judgments_df.loc[
    judgments_df["query_id"].astype(str).eq(H12_ID)
].copy()

if h12_judgments.empty:
    raise AssertionError(
        f"{H12_ID} was not found in the frozen judgments."
    )

if "human_relevance" not in h12_judgments.columns:
    raise AssertionError(
        "The frozen judgments do not contain "
        "'human_relevance'."
    )

identifier_column = next(
    (
        column
        for column in IDENTIFIER_CANDIDATES
        if column in h12_judgments.columns
    ),
    None,
)

if identifier_column is None:
    raise AssertionError(
        "Could not identify a passage identifier. "
        f"Checked: {IDENTIFIER_CANDIDATES}"
    )

text_column = next(
    (
        column
        for column in TEXT_CANDIDATES
        if column in h12_judgments.columns
    ),
    None,
)

h12_useful_columns = [
    "query_id",
    identifier_column,
    "human_relevance",
]

for optional_column in [
    "bm25_rank",
    "dense_rank",
    "hybrid_rank",
    "section",
    "section_name",
    "item",
    text_column,
]:
    if (
        optional_column is not None
        and optional_column in h12_judgments.columns
        and optional_column not in h12_useful_columns
    ):
        h12_useful_columns.append(optional_column)

h12_useful = (
    h12_judgments.loc[
        h12_judgments["human_relevance"].ge(1),
        h12_useful_columns,
    ]
    .drop_duplicates(
        subset=[identifier_column]
    )
    .sort_values(
        [
            "human_relevance",
            identifier_column,
        ],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

if len(h12_useful) != 4:
    print(
        "Warning: expected four useful H12 passages, "
        f"but found {len(h12_useful)}."
    )

useful_identifiers = set(
    h12_useful[identifier_column]
    .astype(str)
)


# ------------------------------------------------------------
# 3. Helper functions for candidate-table normalization
# ------------------------------------------------------------

def normalize_method(value):
    normalized = str(value).strip().lower()

    for alias, canonical_name in METHOD_ALIASES.items():
        if alias in normalized:
            return canonical_name

    return None


def load_candidate_table(path):
    suffix = path.suffix.lower()

    try:
        if suffix == ".parquet":
            return pd.read_parquet(path)

        if suffix == ".csv":
            return pd.read_csv(path)

    except Exception as error:
        print(
            f"Skipped unreadable file: {path.name} "
            f"({type(error).__name__}: {error})"
        )

    return None


def normalize_long_format_candidate_table(
    dataframe,
    source_path,
):
    """
    Normalize tables shaped approximately as:

        query_id | method | rank | corpus_row/chunk_id | ...

    Returns None when the table is not compatible.
    """

    query_column = next(
        (
            column
            for column in [
                "query_id",
                "question_id",
            ]
            if column in dataframe.columns
        ),
        None,
    )

    method_column = next(
        (
            column
            for column in [
                "method",
                "retriever",
                "retrieval_method",
            ]
            if column in dataframe.columns
        ),
        None,
    )

    rank_column = next(
        (
            column
            for column in [
                "rank",
                "retrieval_rank",
                "result_rank",
                "candidate_rank",
            ]
            if column in dataframe.columns
        ),
        None,
    )

    table_identifier_column = next(
        (
            column
            for column in IDENTIFIER_CANDIDATES
            if column in dataframe.columns
        ),
        None,
    )

    if any(
        column is None
        for column in [
            query_column,
            method_column,
            rank_column,
            table_identifier_column,
        ]
    ):
        return None

    h12_rows = dataframe.loc[
        dataframe[query_column]
        .astype(str)
        .eq(H12_ID)
    ].copy()

    if h12_rows.empty:
        return None

    h12_rows["normalized_method"] = (
        h12_rows[method_column]
        .apply(normalize_method)
    )

    h12_rows = h12_rows.loc[
        h12_rows["normalized_method"].isin(
            REQUIRED_DEPTHS
        )
    ].copy()

    if h12_rows.empty:
        return None

    h12_rows["candidate_rank"] = pd.to_numeric(
        h12_rows[rank_column],
        errors="coerce",
    )

    h12_rows["passage_identifier"] = (
        h12_rows[table_identifier_column]
        .astype(str)
    )

    h12_rows["candidate_source"] = str(
        source_path
    )

    keep_columns = [
        "normalized_method",
        "candidate_rank",
        "passage_identifier",
        "candidate_source",
    ]

    table_text_column = next(
        (
            column
            for column in TEXT_CANDIDATES
            if column in h12_rows.columns
        ),
        None,
    )

    if table_text_column is not None:
        h12_rows["candidate_text"] = (
            h12_rows[table_text_column]
        )
        keep_columns.append("candidate_text")

    return h12_rows[keep_columns]


def normalize_wide_format_candidate_table(
    dataframe,
    source_path,
):
    """
    Normalize tables shaped approximately as:

        query_id | corpus_row | bm25_rank | dense_rank | ...

    Returns None when the table is not compatible.
    """

    query_column = next(
        (
            column
            for column in [
                "query_id",
                "question_id",
            ]
            if column in dataframe.columns
        ),
        None,
    )

    table_identifier_column = next(
        (
            column
            for column in IDENTIFIER_CANDIDATES
            if column in dataframe.columns
        ),
        None,
    )

    rank_mapping = {
        "BM25": next(
            (
                column
                for column in [
                    "bm25_rank",
                    "sparse_rank",
                ]
                if column in dataframe.columns
            ),
            None,
        ),
        "Dense": next(
            (
                column
                for column in [
                    "dense_rank",
                    "faiss_rank",
                    "semantic_rank",
                ]
                if column in dataframe.columns
            ),
            None,
        ),
    }

    if (
        query_column is None
        or table_identifier_column is None
        or all(
            column is None
            for column in rank_mapping.values()
        )
    ):
        return None

    h12_rows = dataframe.loc[
        dataframe[query_column]
        .astype(str)
        .eq(H12_ID)
    ].copy()

    if h12_rows.empty:
        return None

    table_text_column = next(
        (
            column
            for column in TEXT_CANDIDATES
            if column in h12_rows.columns
        ),
        None,
    )

    normalized_parts = []

    for method, rank_column in rank_mapping.items():
        if rank_column is None:
            continue

        method_rows = h12_rows.loc[
            h12_rows[rank_column].notna()
        ].copy()

        if method_rows.empty:
            continue

        normalized = pd.DataFrame(
            {
                "normalized_method": method,
                "candidate_rank": pd.to_numeric(
                    method_rows[rank_column],
                    errors="coerce",
                ),
                "passage_identifier": (
                    method_rows[
                        table_identifier_column
                    ].astype(str)
                ),
                "candidate_source": str(
                    source_path
                ),
            }
        )

        if table_text_column is not None:
            normalized["candidate_text"] = (
                method_rows[
                    table_text_column
                ].to_numpy()
            )

        normalized_parts.append(normalized)

    if not normalized_parts:
        return None

    return pd.concat(
        normalized_parts,
        ignore_index=True,
    )


# ------------------------------------------------------------
# 4. Search saved evaluation artifacts
# ------------------------------------------------------------

candidate_directories = []

for possible_directory in [
    globals().get("CHECKPOINT_DIR"),
    globals().get("EVALUATION_DIR"),
    globals().get("ARTIFACT_DIR"),
    Path(FINAL_JUDGMENTS_PATH).parent,
]:
    if possible_directory is None:
        continue

    possible_directory = Path(
        possible_directory
    )

    if (
        possible_directory.exists()
        and possible_directory
        not in candidate_directories
    ):
        candidate_directories.append(
            possible_directory
        )

artifact_paths = []

for directory in candidate_directories:
    artifact_paths.extend(
        directory.rglob("*.parquet")
    )
    artifact_paths.extend(
        directory.rglob("*.csv")
    )

artifact_paths = sorted(
    set(artifact_paths)
)

normalized_candidate_tables = []

for artifact_path in artifact_paths:
    candidate_df = load_candidate_table(
        artifact_path
    )

    if candidate_df is None or candidate_df.empty:
        continue

    long_table = (
        normalize_long_format_candidate_table(
            candidate_df,
            artifact_path,
        )
    )

    if long_table is not None:
        normalized_candidate_tables.append(
            long_table
        )

    wide_table = (
        normalize_wide_format_candidate_table(
            candidate_df,
            artifact_path,
        )
    )

    if wide_table is not None:
        normalized_candidate_tables.append(
            wide_table
        )

if normalized_candidate_tables:
    recovered_candidates = pd.concat(
        normalized_candidate_tables,
        ignore_index=True,
    )

    recovered_candidates = (
        recovered_candidates
        .dropna(
            subset=[
                "normalized_method",
                "candidate_rank",
                "passage_identifier",
            ]
        )
        .drop_duplicates(
            subset=[
                "normalized_method",
                "candidate_rank",
                "passage_identifier",
                "candidate_source",
            ]
        )
        .sort_values(
            [
                "normalized_method",
                "candidate_source",
                "candidate_rank",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

else:
    recovered_candidates = pd.DataFrame(
        columns=[
            "normalized_method",
            "candidate_rank",
            "passage_identifier",
            "candidate_source",
        ]
    )


# ------------------------------------------------------------
# 5. Identify sources with sufficient candidate depth
# ------------------------------------------------------------

source_depth_records = []

for (
    source,
    method,
), group in recovered_candidates.groupby(
    [
        "candidate_source",
        "normalized_method",
    ]
):
    observed_ranks = sorted(
        group["candidate_rank"]
        .dropna()
        .astype(int)
        .unique()
    )

    required_depth = REQUIRED_DEPTHS[method]

    source_depth_records.append(
        {
            "candidate_source": source,
            "method": method,
            "required_depth": required_depth,
            "maximum_observed_rank": (
                max(observed_ranks)
                if observed_ranks
                else np.nan
            ),
            "unique_ranks_observed": len(
                observed_ranks
            ),
            "covers_required_depth": (
                set(
                    range(
                        1,
                        required_depth + 1,
                    )
                )
                .issubset(set(observed_ranks))
            ),
        }
    )

source_depth_audit = pd.DataFrame(
    source_depth_records
)

if not source_depth_audit.empty:
    source_depth_audit = (
        source_depth_audit
        .sort_values(
            [
                "method",
                "covers_required_depth",
                "maximum_observed_rank",
            ],
            ascending=[
                True,
                False,
                False,
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 6. Select one valid full-depth source per method
# ------------------------------------------------------------

selected_sources = {}

for method, required_depth in REQUIRED_DEPTHS.items():
    valid_sources = source_depth_audit.loc[
        source_depth_audit["method"].eq(
            method
        )
        & source_depth_audit[
            "covers_required_depth"
        ]
    ]

    if not valid_sources.empty:
        selected_sources[method] = (
            valid_sources.iloc[0][
                "candidate_source"
            ]
        )


# ------------------------------------------------------------
# 7. Recover ranks of all useful H12 passages
# ------------------------------------------------------------

recovery_records = []

for _, useful_row in h12_useful.iterrows():
    passage_identifier = str(
        useful_row[identifier_column]
    )

    record = {
        identifier_column: useful_row[
            identifier_column
        ],
        "human_relevance": int(
            useful_row["human_relevance"]
        ),
    }

    if text_column is not None:
        record[text_column] = useful_row.get(
            text_column
        )

    for method, required_depth in REQUIRED_DEPTHS.items():
        rank_output_column = (
            "bm25_full_rank"
            if method == "BM25"
            else "dense_full_rank"
        )

        status_output_column = (
            "bm25_candidate_status"
            if method == "BM25"
            else "dense_candidate_status"
        )

        source = selected_sources.get(method)

        if source is None:
            record[rank_output_column] = np.nan
            record[status_output_column] = (
                "NOT RECOVERABLE: no saved "
                f"top-{required_depth} source"
            )
            continue

        matching_rows = recovered_candidates.loc[
            recovered_candidates[
                "candidate_source"
            ].eq(source)
            & recovered_candidates[
                "normalized_method"
            ].eq(method)
            & recovered_candidates[
                "passage_identifier"
            ].eq(passage_identifier)
        ]

        if matching_rows.empty:
            record[rank_output_column] = np.nan
            record[status_output_column] = (
                f"ABSENT FROM TOP {required_depth}"
            )
        else:
            recovered_rank = int(
                matching_rows[
                    "candidate_rank"
                ].min()
            )

            record[rank_output_column] = (
                recovered_rank
            )
            record[status_output_column] = (
                f"PRESENT AT RANK {recovered_rank}"
            )

    recovery_records.append(record)

h12_full_candidate_recovery = pd.DataFrame(
    recovery_records
)


# ------------------------------------------------------------
# 8. Add concise cross-retriever interpretation
# ------------------------------------------------------------

def interpret_candidate_recovery(row):
    bm25_rank = row["bm25_full_rank"]
    dense_rank = row["dense_full_rank"]

    bm25_available = pd.notna(bm25_rank)
    dense_available = pd.notna(dense_rank)

    bm25_recoverable = not str(
        row["bm25_candidate_status"]
    ).startswith("NOT RECOVERABLE")

    dense_recoverable = not str(
        row["dense_candidate_status"]
    ).startswith("NOT RECOVERABLE")

    if not (
        bm25_recoverable
        and dense_recoverable
    ):
        return (
            "Incomplete saved-candidate evidence"
        )

    if bm25_available and dense_available:
        return (
            "Present in both fusion candidate sets"
        )

    if bm25_available:
        return (
            "BM25-only fusion candidate"
        )

    if dense_available:
        return (
            "Dense-only fusion candidate"
        )

    return (
        "Absent from both saved fusion candidate sets"
    )


h12_full_candidate_recovery[
    "candidate_set_interpretation"
] = (
    h12_full_candidate_recovery.apply(
        interpret_candidate_recovery,
        axis=1,
    )
)


# ------------------------------------------------------------
# 9. Validate recovered ranks
# ------------------------------------------------------------

if "bm25_full_rank" in h12_full_candidate_recovery:
    assert (
        h12_full_candidate_recovery[
            "bm25_full_rank"
        ]
        .dropna()
        .le(REQUIRED_DEPTHS["BM25"])
        .all()
    )

if "dense_full_rank" in h12_full_candidate_recovery:
    assert (
        h12_full_candidate_recovery[
            "dense_full_rank"
        ]
        .dropna()
        .le(REQUIRED_DEPTHS["Dense"])
        .all()
    )


# ------------------------------------------------------------
# 10. Display results
# ------------------------------------------------------------

print("=" * 72)
print("H12 FULL FUSION-CANDIDATE RECOVERY")
print("=" * 72)
print(
    f"Useful passages:           "
    f"{len(h12_full_candidate_recovery)}"
)
print(
    "Required BM25 depth:       "
    f"{REQUIRED_DEPTHS['BM25']}"
)
print(
    "Required Dense depth:      "
    f"{REQUIRED_DEPTHS['Dense']}"
)
print(
    "BM25 source recovered:     "
    f"{'Yes' if 'BM25' in selected_sources else 'No'}"
)
print(
    "Dense source recovered:    "
    f"{'Yes' if 'Dense' in selected_sources else 'No'}"
)

print()
print("CANDIDATE-SOURCE DEPTH AUDIT")

if source_depth_audit.empty:
    print(
        "No compatible saved candidate tables were found."
    )
else:
    display(source_depth_audit)

print()
print("H12 USEFUL-PASSAGE FULL-RANK RECOVERY")

display_columns = [
    identifier_column,
    "human_relevance",
    "bm25_full_rank",
    "bm25_candidate_status",
    "dense_full_rank",
    "dense_candidate_status",
    "candidate_set_interpretation",
]

if text_column is not None:
    display_columns.append(text_column)

display(
    h12_full_candidate_recovery[
        display_columns
    ]
)

print()
print("SELECTED FULL-DEPTH SOURCES")

if selected_sources:
    for method, source in selected_sources.items():
        print(f"{method}: {source}")
else:
    print(
        "No saved artifact was proven to contain "
        "the complete BM25 top 20 or Dense top 30."
    )

H12 FULL FUSION-CANDIDATE RECOVERY
Useful passages:           4
Required BM25 depth:       20
Required Dense depth:      30
BM25 source recovered:     No
Dense source recovered:    No

CANDIDATE-SOURCE DEPTH AUDIT


,candidate_source,method,required_depth,maximum_observed_rank,unique_ranks_observed,covers_required_depth
0,/content/drive/MyDrive/FinanceRAG-Analyst/arti...,BM25,20,19,13,False
1,/content/drive/MyDrive/FinanceRAG-Analyst/arti...,BM25,20,19,13,False
2,/content/drive/MyDrive/FinanceRAG-Analyst/arti...,BM25,20,10,10,False
3,/content/drive/MyDrive/FinanceRAG-Analyst/arti...,BM25,20,10,10,False
4,/content/drive/MyDrive/FinanceRAG-Analyst/arti...,BM25,20,10,10,False
5,/content/drive/MyDrive/FinanceRAG-Analyst/arti...,BM25,20,10,10,False
6,/content/drive/MyDrive/FinanceRAG-Analyst/arti...,BM25,20,10,10,False
7,/content/drive/MyDrive/FinanceRAG-Analyst/arti...,BM25,20,10,10,False
8,/content/drive/MyDrive/FinanceRAG-Analyst/arti...,Dense,30,24,13,False
9,/content/drive/MyDrive/FinanceRAG-Analyst/arti...,Dense,30,24,13,False



H12 USEFUL-PASSAGE FULL-RANK RECOVERY


,corpus_row,human_relevance,bm25_full_rank,bm25_candidate_status,dense_full_rank,dense_candidate_status,candidate_set_interpretation,text
0,2396,2,NaN,NOT RECOVERABLE: no saved top-20 source,NaN,NOT RECOVERABLE: no saved top-30 source,Incomplete saved-candidate evidence,It may take longer than expected to realize th...
1,2402,2,NaN,NOT RECOVERABLE: no saved top-20 source,NaN,NOT RECOVERABLE: no saved top-30 source,Incomplete saved-candidate evidence,"At times, to achieve commercial objectives, we..."
2,2394,1,NaN,NOT RECOVERABLE: no saved top-20 source,NaN,NOT RECOVERABLE: no saved top-30 source,Incomplete saved-candidate evidence,These include unauthorized use of accounts thr...
3,2403,1,NaN,NOT RECOVERABLE: no saved top-20 source,NaN,NOT RECOVERABLE: no saved top-30 source,Incomplete saved-candidate evidence,Despite our efforts to protect customer and us...



SELECTED FULL-DEPTH SOURCES
No saved artifact was proven to contain the complete BM25 top 20 or Dense top 30.


### H12 full-candidate recovery limitation

The saved evaluation artifacts did not preserve a verifiably complete BM25
top-20 or dense top-30 candidate list for H12. Although some pooled artifacts
contained ranks as deep as BM25 rank 19 and dense rank 24, their rank sequences
were incomplete and therefore could not be treated as authoritative full-depth
retrieval outputs.

Consequently, the analysis cannot retrospectively determine whether each useful
H12 passage was completely absent from the other component retriever's fusion
candidate set or present only at a weak rank.

The observable failure mechanism remains clear at the evaluated top-10 depth:
BM25 retrieved direct evidence at rank 6, while dense retrieval independently
retrieved direct evidence at rank 7 and partial evidence at ranks 8 and 10.
Hybrid RRF nevertheless returned no useful evidence in its top 10. Recorded RRF
scores show that irrelevant passages receiving support from both component
retrievers displaced useful passages without sufficiently strong
cross-retriever agreement.

H12 is therefore reported as a false-consensus fusion failure, with the precise
full-depth component membership retained as an explicit artifact limitation.
The frozen benchmark and retrieval configuration were not modified or rerun for
this retrospective diagnostic.

In [ ]:
# ============================================================
# HELD-OUT RETRIEVAL EVALUATION:
# QUERY-LEVEL RETRIEVER COMPLEMENTARITY
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Load and validate frozen per-query metrics
# ------------------------------------------------------------

metrics_df = pd.read_parquet(
    PER_QUERY_METRICS_PATH
).copy()

required_columns = [
    "query_id",
    "method",
    "retrieved_partial_at_10",
    "retrieved_direct_at_10",
    "useful_precision_at_10",
    "useful_pool_recall_at_10",
    "direct_mrr_at_10",
    "graded_ndcg_at_10",
]

missing_columns = [
    column
    for column in required_columns
    if column not in metrics_df.columns
]

if missing_columns:
    raise AssertionError(
        "Per-query metrics are missing required columns: "
        f"{missing_columns}"
    )


# ------------------------------------------------------------
# 2. Normalize method names
# ------------------------------------------------------------

def normalize_method_name(value):
    normalized = (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    method_aliases = {
        "bm25": "BM25",
        "sparse": "BM25",
        "dense": "Dense",
        "faiss": "Dense",
        "semantic": "Dense",
        "hybrid": "Hybrid RRF",
        "hybrid_rrf": "Hybrid RRF",
        "rrf": "Hybrid RRF",
    }

    return method_aliases.get(
        normalized,
        str(value).strip(),
    )


metrics_df["method"] = (
    metrics_df["method"]
    .apply(normalize_method_name)
)

expected_methods = {
    "BM25",
    "Dense",
    "Hybrid RRF",
}

observed_methods = set(
    metrics_df["method"].dropna().unique()
)

missing_methods = (
    expected_methods
    - observed_methods
)

if missing_methods:
    raise AssertionError(
        "Expected methods were not found: "
        f"{sorted(missing_methods)}. "
        f"Observed methods: {sorted(observed_methods)}"
    )


# ------------------------------------------------------------
# 3. Validate one row per query-method combination
# ------------------------------------------------------------

duplicate_query_methods = (
    metrics_df
    .groupby(
        ["query_id", "method"]
    )
    .size()
    .loc[lambda values: values.ne(1)]
)

if not duplicate_query_methods.empty:
    raise AssertionError(
        "Expected exactly one row per query and method. "
        f"Problematic combinations:\n"
        f"{duplicate_query_methods}"
    )

methods_per_query = (
    metrics_df
    .groupby("query_id")["method"]
    .agg(set)
)

incomplete_queries = methods_per_query.loc[
    methods_per_query.apply(
        lambda methods: methods != expected_methods
    )
]

if not incomplete_queries.empty:
    raise AssertionError(
        "Some queries do not contain all three methods:\n"
        f"{incomplete_queries}"
    )


# ------------------------------------------------------------
# 4. Add derived useful-evidence indicators
# ------------------------------------------------------------

metrics_df["retrieved_useful_at_10"] = (
    metrics_df["retrieved_partial_at_10"]
    + metrics_df["retrieved_direct_at_10"]
)

metrics_df["found_useful_at_10"] = (
    metrics_df["retrieved_useful_at_10"]
    .gt(0)
)

metrics_df["found_direct_at_10"] = (
    metrics_df["retrieved_direct_at_10"]
    .gt(0)
)


# ------------------------------------------------------------
# 5. Pivot selected metrics to query-level wide format
# ------------------------------------------------------------

metric_columns = [
    "retrieved_useful_at_10",
    "found_useful_at_10",
    "found_direct_at_10",
    "useful_precision_at_10",
    "useful_pool_recall_at_10",
    "direct_mrr_at_10",
    "graded_ndcg_at_10",
]

wide_metrics = metrics_df.pivot(
    index="query_id",
    columns="method",
    values=metric_columns,
)

# Flatten columns to names such as:
# dense_direct_mrr_at_10
wide_metrics.columns = [
    (
        f"{method.lower().replace(' ', '_')}"
        f"__{metric}"
    )
    for metric, method in wide_metrics.columns
]

wide_metrics = (
    wide_metrics
    .reset_index()
    .sort_values(
        "query_id",
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Helpers for determining winners and ties
# ------------------------------------------------------------

METHOD_ORDER = [
    "BM25",
    "Dense",
    "Hybrid RRF",
]

METHOD_PREFIXES = {
    "BM25": "bm25",
    "Dense": "dense",
    "Hybrid RRF": "hybrid_rrf",
}


def metric_value(row, method, metric):
    return row[
        f"{METHOD_PREFIXES[method]}__{metric}"
    ]


def identify_best_methods(
    row,
    metric,
    tolerance=1e-12,
):
    values = {
        method: float(
            metric_value(
                row,
                method,
                metric,
            )
        )
        for method in METHOD_ORDER
    }

    best_value = max(values.values())

    winners = [
        method
        for method in METHOD_ORDER
        if np.isclose(
            values[method],
            best_value,
            atol=tolerance,
            rtol=0.0,
        )
    ]

    return " = ".join(winners)


# ------------------------------------------------------------
# 7. Build the compact complementarity table
# ------------------------------------------------------------

complementarity_records = []

for _, row in wide_metrics.iterrows():
    bm25_found_useful = bool(
        metric_value(
            row,
            "BM25",
            "found_useful_at_10",
        )
    )

    dense_found_useful = bool(
        metric_value(
            row,
            "Dense",
            "found_useful_at_10",
        )
    )

    hybrid_found_useful = bool(
        metric_value(
            row,
            "Hybrid RRF",
            "found_useful_at_10",
        )
    )

    bm25_direct_mrr = metric_value(
        row,
        "BM25",
        "direct_mrr_at_10",
    )

    dense_direct_mrr = metric_value(
        row,
        "Dense",
        "direct_mrr_at_10",
    )

    hybrid_direct_mrr = metric_value(
        row,
        "Hybrid RRF",
        "direct_mrr_at_10",
    )

    best_component_direct_mrr = max(
        bm25_direct_mrr,
        dense_direct_mrr,
    )

    hybrid_improved_direct_ranking = (
        hybrid_direct_mrr
        > best_component_direct_mrr + 1e-12
    )

    hybrid_matched_best_component = np.isclose(
        hybrid_direct_mrr,
        best_component_direct_mrr,
        atol=1e-12,
        rtol=0.0,
    )

    component_found_useful = (
        bm25_found_useful
        or dense_found_useful
    )

    hybrid_lost_all_component_useful_evidence = (
        component_found_useful
        and not hybrid_found_useful
    )

    if hybrid_improved_direct_ranking:
        hybrid_direct_outcome = (
            "Improved over both components"
        )
    elif hybrid_matched_best_component:
        hybrid_direct_outcome = (
            "Matched best component"
        )
    else:
        hybrid_direct_outcome = (
            "Below best component"
        )

    complementarity_records.append(
        {
            "query_id": row["query_id"],
            "best_useful_coverage_method": (
                identify_best_methods(
                    row,
                    "useful_pool_recall_at_10",
                )
            ),
            "best_direct_mrr_method": (
                identify_best_methods(
                    row,
                    "direct_mrr_at_10",
                )
            ),
            "best_ndcg_method": (
                identify_best_methods(
                    row,
                    "graded_ndcg_at_10",
                )
            ),
            "bm25_found_useful": (
                bm25_found_useful
            ),
            "dense_found_useful": (
                dense_found_useful
            ),
            "hybrid_found_useful": (
                hybrid_found_useful
            ),
            "hybrid_direct_ranking_outcome": (
                hybrid_direct_outcome
            ),
            "hybrid_lost_all_component_useful_evidence": (
                hybrid_lost_all_component_useful_evidence
            ),
        }
    )

query_complementarity = pd.DataFrame(
    complementarity_records
)


# ------------------------------------------------------------
# 8. Add useful query metadata when available
# ------------------------------------------------------------

judgments_df = pd.read_parquet(
    FINAL_JUDGMENTS_PATH
).copy()

possible_metadata_columns = [
    "query_id",
    "query",
    "query_text",
    "question",
    "question_text",
    "ticker",
    "company",
    "filing_type",
    "form_type",
    "question_type",
    "query_type",
    "difficulty",
]

available_metadata_columns = [
    column
    for column in possible_metadata_columns
    if column in judgments_df.columns
]

query_metadata = (
    judgments_df[
        available_metadata_columns
    ]
    .drop_duplicates()
    .copy()
)

metadata_conflicts = (
    query_metadata
    .groupby("query_id")
    .size()
    .loc[lambda values: values.gt(1)]
)

if not metadata_conflicts.empty:
    print(
        "Warning: conflicting query metadata was found "
        "for these query IDs:"
    )
    print(metadata_conflicts)

    query_metadata = (
        query_metadata
        .drop_duplicates(
            subset=["query_id"],
            keep="first",
        )
    )

query_complementarity = (
    query_metadata
    .merge(
        query_complementarity,
        on="query_id",
        how="right",
        validate="one_to_one",
    )
    .sort_values(
        "query_id",
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Calculate dataset-level complementarity counts
# ------------------------------------------------------------

query_count = len(
    query_complementarity
)

bm25_useful_count = int(
    query_complementarity[
        "bm25_found_useful"
    ].sum()
)

dense_useful_count = int(
    query_complementarity[
        "dense_found_useful"
    ].sum()
)

hybrid_useful_count = int(
    query_complementarity[
        "hybrid_found_useful"
    ].sum()
)

both_components_useful_count = int(
    (
        query_complementarity[
            "bm25_found_useful"
        ]
        & query_complementarity[
            "dense_found_useful"
        ]
    ).sum()
)

bm25_only_useful_count = int(
    (
        query_complementarity[
            "bm25_found_useful"
        ]
        & ~query_complementarity[
            "dense_found_useful"
        ]
    ).sum()
)

dense_only_useful_count = int(
    (
        ~query_complementarity[
            "bm25_found_useful"
        ]
        & query_complementarity[
            "dense_found_useful"
        ]
    ).sum()
)

neither_component_useful_count = int(
    (
        ~query_complementarity[
            "bm25_found_useful"
        ]
        & ~query_complementarity[
            "dense_found_useful"
        ]
    ).sum()
)

hybrid_improvement_count = int(
    query_complementarity[
        "hybrid_direct_ranking_outcome"
    ]
    .eq("Improved over both components")
    .sum()
)

hybrid_match_count = int(
    query_complementarity[
        "hybrid_direct_ranking_outcome"
    ]
    .eq("Matched best component")
    .sum()
)

hybrid_below_count = int(
    query_complementarity[
        "hybrid_direct_ranking_outcome"
    ]
    .eq("Below best component")
    .sum()
)

hybrid_loss_count = int(
    query_complementarity[
        "hybrid_lost_all_component_useful_evidence"
    ].sum()
)

complementarity_summary = pd.DataFrame(
    [
        {
            "diagnostic": (
                "BM25 found useful evidence"
            ),
            "query_count": bm25_useful_count,
        },
        {
            "diagnostic": (
                "Dense found useful evidence"
            ),
            "query_count": dense_useful_count,
        },
        {
            "diagnostic": (
                "Hybrid found useful evidence"
            ),
            "query_count": hybrid_useful_count,
        },
        {
            "diagnostic": (
                "Both components found useful evidence"
            ),
            "query_count": both_components_useful_count,
        },
        {
            "diagnostic": (
                "Only BM25 found useful evidence"
            ),
            "query_count": bm25_only_useful_count,
        },
        {
            "diagnostic": (
                "Only Dense found useful evidence"
            ),
            "query_count": dense_only_useful_count,
        },
        {
            "diagnostic": (
                "Neither component found useful evidence"
            ),
            "query_count": neither_component_useful_count,
        },
        {
            "diagnostic": (
                "Hybrid improved Direct MRR "
                "over both components"
            ),
            "query_count": hybrid_improvement_count,
        },
        {
            "diagnostic": (
                "Hybrid matched the best component "
                "on Direct MRR"
            ),
            "query_count": hybrid_match_count,
        },
        {
            "diagnostic": (
                "Hybrid fell below the best component "
                "on Direct MRR"
            ),
            "query_count": hybrid_below_count,
        },
        {
            "diagnostic": (
                "Hybrid lost all useful evidence found "
                "by at least one component"
            ),
            "query_count": hybrid_loss_count,
        },
    ]
)

complementarity_summary["query_fraction"] = (
    complementarity_summary["query_count"]
    / query_count
)


# ------------------------------------------------------------
# 10. Method winner counts, with ties preserved
# ------------------------------------------------------------

def count_winner_membership(column):
    records = []

    for method in METHOD_ORDER:
        count = int(
            query_complementarity[column]
            .fillna("")
            .str.split(" = ")
            .apply(
                lambda winners: method in winners
            )
            .sum()
        )

        records.append(
            {
                "metric": column,
                "method": method,
                "queries_won_or_tied": count,
            }
        )

    return records


winner_count_records = []

for winner_column in [
    "best_useful_coverage_method",
    "best_direct_mrr_method",
    "best_ndcg_method",
]:
    winner_count_records.extend(
        count_winner_membership(
            winner_column
        )
    )

method_winner_counts = pd.DataFrame(
    winner_count_records
)


# ------------------------------------------------------------
# 11. Validation checks
# ------------------------------------------------------------

if query_count != 12:
    print(
        "Warning: expected 12 held-out queries, "
        f"but found {query_count}."
    )

assert (
    bm25_only_useful_count
    + dense_only_useful_count
    + both_components_useful_count
    + neither_component_useful_count
    == query_count
)

assert (
    hybrid_improvement_count
    + hybrid_match_count
    + hybrid_below_count
    == query_count
)

assert not query_complementarity[
    [
        "best_useful_coverage_method",
        "best_direct_mrr_method",
        "best_ndcg_method",
    ]
].isna().any().any()


# ------------------------------------------------------------
# 12. Display final tables
# ------------------------------------------------------------

boolean_columns = [
    "bm25_found_useful",
    "dense_found_useful",
    "hybrid_found_useful",
    "hybrid_lost_all_component_useful_evidence",
]

display_complementarity = (
    query_complementarity.copy()
)

display_complementarity[
    boolean_columns
] = (
    display_complementarity[
        boolean_columns
    ]
    .replace(
        {
            True: "Yes",
            False: "No",
        }
    )
)

print("=" * 88)
print("QUERY-LEVEL RETRIEVER COMPLEMENTARITY")
print("=" * 88)

display(display_complementarity)

print()
print("=" * 88)
print("COMPLEMENTARITY SUMMARY")
print("=" * 88)

display_summary = (
    complementarity_summary.copy()
)

display_summary["query_fraction"] = (
    display_summary["query_fraction"]
    .round(4)
)

display(display_summary)

print()
print("=" * 88)
print("METHOD WINNER OR TIE COUNTS")
print("=" * 88)

display(method_winner_counts)

QUERY-LEVEL RETRIEVER COMPLEMENTARITY


,query_id,query,ticker,company,form_type,question_type,difficulty,best_useful_coverage_method,best_direct_mrr_method,best_ndcg_method,bm25_found_useful,dense_found_useful,hybrid_found_useful,hybrid_direct_ranking_outcome,hybrid_lost_all_component_useful_evidence
0,H01,"According to Apple's 2025 Form 10-K, what fact...",AAPL,Apple Inc.,10-K,reported_driver,medium,BM25,Hybrid RRF,Hybrid RRF,Yes,Yes,Yes,Improved over both components,No
1,H02,"According to Apple's 2024 Form 10-K, how did n...",AAPL,Apple Inc.,10-K,geographic_or_segment_comparison,medium,Hybrid RRF,Hybrid RRF,Hybrid RRF,Yes,Yes,Yes,Improved over both components,No
2,H03,"According to Microsoft's 2025 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,segment_driver,medium,Dense,Dense,Dense,Yes,Yes,Yes,Below best component,No
3,H04,"According to Microsoft's 2024 Form 10-K, what ...",MSFT,MICROSOFT CORP,10-K,capital_expenditure,medium,BM25,BM25,BM25,Yes,Yes,Yes,Below best component,No
4,H05,"According to Amazon's 2024 Form 10-K, how did ...",AMZN,AMAZON COM INC,10-K,segment_comparison,medium,Dense,Dense,Dense,Yes,Yes,Yes,Below best component,No
5,H06,"According to Amazon's 2023 Form 10-K, what fac...",AMZN,AMAZON COM INC,10-K,cash_flow_driver,hard,Dense = Hybrid RRF,Dense = Hybrid RRF,Hybrid RRF,Yes,Yes,Yes,Matched best component,No
6,H07,"According to Alphabet's 2024 Form 10-K, what f...",GOOGL,Alphabet Inc.,10-K,revenue_driver,medium,BM25 = Dense = Hybrid RRF,BM25,BM25,Yes,Yes,Yes,Below best component,No
7,H08,"According to Alphabet's 2023 Form 10-K, what r...",GOOGL,Alphabet Inc.,10-K,risk_factor,hard,Dense,Hybrid RRF,Hybrid RRF,Yes,Yes,Yes,Improved over both components,No
8,H09,"According to NVIDIA's 2025 Form 10-K, what fac...",NVDA,NVIDIA CORP,10-K,segment_driver,medium,Dense,BM25 = Dense = Hybrid RRF,Dense,Yes,Yes,Yes,Matched best component,No
9,H10,"According to NVIDIA's 2024 Form 10-K, what sup...",NVDA,NVIDIA CORP,10-K,supply_risk,hard,Hybrid RRF,Hybrid RRF,Hybrid RRF,Yes,Yes,Yes,Improved over both components,No



COMPLEMENTARITY SUMMARY


,diagnostic,query_count,query_fraction
0,BM25 found useful evidence,12,1.0000
1,Dense found useful evidence,12,1.0000
2,Hybrid found useful evidence,11,0.9167
3,Both components found useful evidence,12,1.0000
4,Only BM25 found useful evidence,0,0.0000
5,Only Dense found useful evidence,0,0.0000
6,Neither component found useful evidence,0,0.0000
7,Hybrid improved Direct MRR over both components,4,0.3333
8,Hybrid matched the best component on Direct MRR,3,0.2500
9,Hybrid fell below the best component on Direct...,5,0.4167



METHOD WINNER OR TIE COUNTS


,metric,method,queries_won_or_tied
0,best_useful_coverage_method,BM25,4
1,best_useful_coverage_method,Dense,7
2,best_useful_coverage_method,Hybrid RRF,4
3,best_direct_mrr_method,BM25,5
4,best_direct_mrr_method,Dense,4
5,best_direct_mrr_method,Hybrid RRF,7
6,best_ndcg_method,BM25,3
7,best_ndcg_method,Dense,4
8,best_ndcg_method,Hybrid RRF,5


### Query-level retriever complementarity

Both component retrievers found at least one useful top-10 passage for all 12
held-out queries. Their complementarity therefore did not arise from exclusive
query coverage; instead, it appeared in which method achieved the strongest
useful-passage coverage and ranking for each query.

Dense retrieval achieved the best or tied-best pool-relative useful Recall@10
on 7 of 12 queries, compared with 4 queries each for BM25 and Hybrid RRF.
BM25 nevertheless produced the best or tied-best useful coverage on several
queries, including H01, H04, H07, and H11. This shows that the component
retrievers had different query-level strengths even though neither uniquely
rescued a query that the other completely missed.

Hybrid RRF achieved the best or tied-best Direct MRR@10 on 7 of 12 queries.
Relative to the stronger component retriever for each query, Hybrid improved
direct-evidence ranking on 4 queries, matched it on 3, and fell below it on 5.
Hybrid therefore improved or preserved the best component's direct-evidence
ranking on 7 of 12 queries, but did not consistently dominate both component
retrievers.

For graded nDCG@10, Hybrid was best or tied-best on 5 queries, Dense on 4, and
BM25 on 3. These winner counts include ties and therefore should not be summed
as mutually exclusive query totals.

H12 was the sole severe fusion failure. Both BM25 and Dense independently
retrieved useful evidence in their top 10, while Hybrid retrieved none.
Accordingly, Hybrid preserved some useful evidence found by the components on
11 of 12 queries, but RRF did not guarantee preservation of relevant component
results.

Overall, the query-level analysis supports using BM25 and Dense as complementary
retrieval signals, while treating RRF as a useful but imperfect rank-fusion
strategy. Its principal benefit in this benchmark was improved or preserved
early direct-evidence placement on a majority of queries, rather than uniformly
higher evidence coverage.

In [ ]:
# ============================================================
# HELD-OUT RETRIEVAL EVALUATION:
# FINAL ARTIFACT-INTEGRITY VALIDATION
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Resolve the required frozen artifacts
# ------------------------------------------------------------

required_artifacts = {
    "final_judgments": Path(FINAL_JUDGMENTS_PATH),
    "per_query_metrics": Path(PER_QUERY_METRICS_PATH),
}

missing_artifacts = {
    name: str(path)
    for name, path in required_artifacts.items()
    if not path.exists()
}

if missing_artifacts:
    raise FileNotFoundError(
        "Required frozen evaluation artifacts were not found:\n"
        + json.dumps(
            missing_artifacts,
            indent=2,
        )
    )


# ------------------------------------------------------------
# 2. Compute reproducibility metadata
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        while True:
            block = file.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


artifact_manifest_records = []

for artifact_name, artifact_path in required_artifacts.items():
    artifact_manifest_records.append(
        {
            "artifact": artifact_name,
            "path": str(artifact_path),
            "file_size_bytes": artifact_path.stat().st_size,
            "sha256": sha256_file(artifact_path),
        }
    )

artifact_manifest = pd.DataFrame(
    artifact_manifest_records
)


# ------------------------------------------------------------
# 3. Reload frozen artifacts independently
# ------------------------------------------------------------

final_judgments_check = pd.read_parquet(
    required_artifacts["final_judgments"]
).copy()

per_query_metrics_check = pd.read_parquet(
    required_artifacts["per_query_metrics"]
).copy()


# ------------------------------------------------------------
# 4. Validate judgment-table structure
# ------------------------------------------------------------

required_judgment_columns = [
    "query_id",
    "human_relevance",
]

missing_judgment_columns = [
    column
    for column in required_judgment_columns
    if column not in final_judgments_check.columns
]

if missing_judgment_columns:
    raise AssertionError(
        "Final judgments are missing required columns: "
        f"{missing_judgment_columns}"
    )

if final_judgments_check.empty:
    raise AssertionError(
        "The final judgment table is empty."
    )

if final_judgments_check[
    required_judgment_columns
].isna().any().any():
    raise AssertionError(
        "Null values were found in required judgment columns."
    )

observed_relevance_values = set(
    pd.to_numeric(
        final_judgments_check["human_relevance"],
        errors="raise",
    ).unique()
)

allowed_relevance_values = {
    0,
    1,
    2,
}

if not observed_relevance_values.issubset(
    allowed_relevance_values
):
    raise AssertionError(
        "Unexpected human-relevance labels found: "
        f"{sorted(observed_relevance_values)}"
    )


# ------------------------------------------------------------
# 5. Identify and validate the passage identifier
# ------------------------------------------------------------

identifier_candidates = [
    "corpus_row",
    "chunk_id",
    "passage_id",
    "document_id",
    "doc_id",
]

judgment_identifier_column = next(
    (
        column
        for column in identifier_candidates
        if column in final_judgments_check.columns
    ),
    None,
)

if judgment_identifier_column is None:
    raise AssertionError(
        "No passage identifier was found in the final "
        f"judgment table. Checked: {identifier_candidates}"
    )

duplicate_judgments = (
    final_judgments_check
    .groupby(
        [
            "query_id",
            judgment_identifier_column,
        ],
        dropna=False,
    )
    .size()
    .loc[lambda values: values.gt(1)]
)

if not duplicate_judgments.empty:
    raise AssertionError(
        "Duplicate query-passage judgments were found:\n"
        f"{duplicate_judgments}"
    )


# ------------------------------------------------------------
# 6. Validate per-query metric structure
# ------------------------------------------------------------

required_metric_columns = [
    "query_id",
    "method",
    "retrieved_partial_at_10",
    "retrieved_direct_at_10",
    "useful_precision_at_10",
    "useful_pool_recall_at_10",
    "direct_mrr_at_10",
    "graded_ndcg_at_10",
]

missing_metric_columns = [
    column
    for column in required_metric_columns
    if column not in per_query_metrics_check.columns
]

if missing_metric_columns:
    raise AssertionError(
        "Per-query metrics are missing required columns: "
        f"{missing_metric_columns}"
    )

if per_query_metrics_check.empty:
    raise AssertionError(
        "The per-query metric table is empty."
    )


# ------------------------------------------------------------
# 7. Normalize and validate retrieval methods
# ------------------------------------------------------------

def normalize_final_method_name(value):
    normalized = (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    aliases = {
        "bm25": "BM25",
        "sparse": "BM25",
        "dense": "Dense",
        "faiss": "Dense",
        "semantic": "Dense",
        "hybrid": "Hybrid RRF",
        "hybrid_rrf": "Hybrid RRF",
        "rrf": "Hybrid RRF",
    }

    return aliases.get(
        normalized,
        str(value).strip(),
    )


per_query_metrics_check["method"] = (
    per_query_metrics_check["method"]
    .apply(normalize_final_method_name)
)

expected_methods = {
    "BM25",
    "Dense",
    "Hybrid RRF",
}

observed_methods = set(
    per_query_metrics_check["method"]
    .dropna()
    .unique()
)

if observed_methods != expected_methods:
    raise AssertionError(
        "Method set does not match the frozen benchmark. "
        f"Expected {sorted(expected_methods)}, "
        f"observed {sorted(observed_methods)}."
    )


# ------------------------------------------------------------
# 8. Validate held-out query coverage
# ------------------------------------------------------------

judgment_query_ids = set(
    final_judgments_check["query_id"]
    .astype(str)
)

metric_query_ids = set(
    per_query_metrics_check["query_id"]
    .astype(str)
)

if judgment_query_ids != metric_query_ids:
    raise AssertionError(
        "Query IDs differ between judgments and metrics.\n"
        f"Only in judgments: "
        f"{sorted(judgment_query_ids - metric_query_ids)}\n"
        f"Only in metrics: "
        f"{sorted(metric_query_ids - judgment_query_ids)}"
    )

if len(metric_query_ids) != 12:
    raise AssertionError(
        "Expected exactly 12 held-out queries, "
        f"but found {len(metric_query_ids)}."
    )

duplicate_query_methods = (
    per_query_metrics_check
    .groupby(
        [
            "query_id",
            "method",
        ]
    )
    .size()
    .loc[lambda values: values.ne(1)]
)

if not duplicate_query_methods.empty:
    raise AssertionError(
        "Expected exactly one row per query-method pair:\n"
        f"{duplicate_query_methods}"
    )

methods_by_query = (
    per_query_metrics_check
    .groupby("query_id")["method"]
    .agg(set)
)

incomplete_queries = methods_by_query.loc[
    methods_by_query.apply(
        lambda methods: methods != expected_methods
    )
]

if not incomplete_queries.empty:
    raise AssertionError(
        "Some held-out queries do not contain all three "
        f"retrieval methods:\n{incomplete_queries}"
    )

expected_metric_rows = (
    len(metric_query_ids)
    * len(expected_methods)
)

if len(per_query_metrics_check) != expected_metric_rows:
    raise AssertionError(
        "Unexpected number of per-query metric rows. "
        f"Expected {expected_metric_rows}, "
        f"found {len(per_query_metrics_check)}."
    )


# ------------------------------------------------------------
# 9. Validate metric domains and internal relationships
# ------------------------------------------------------------

bounded_metric_columns = [
    "useful_precision_at_10",
    "useful_pool_recall_at_10",
    "direct_mrr_at_10",
    "graded_ndcg_at_10",
]

count_metric_columns = [
    "retrieved_partial_at_10",
    "retrieved_direct_at_10",
]

for column in (
    bounded_metric_columns
    + count_metric_columns
):
    per_query_metrics_check[column] = pd.to_numeric(
        per_query_metrics_check[column],
        errors="raise",
    )

if per_query_metrics_check[
    bounded_metric_columns
].isna().any().any():
    raise AssertionError(
        "Null values were found in bounded metric columns."
    )

for column in bounded_metric_columns:
    invalid_rows = per_query_metrics_check.loc[
        ~per_query_metrics_check[column]
        .between(
            0.0,
            1.0,
            inclusive="both",
        )
    ]

    if not invalid_rows.empty:
        raise AssertionError(
            f"{column} contains values outside [0, 1]."
        )

for column in count_metric_columns:
    values = per_query_metrics_check[column]

    if values.isna().any():
        raise AssertionError(
            f"{column} contains null values."
        )

    if values.lt(0).any():
        raise AssertionError(
            f"{column} contains negative counts."
        )

    if not np.allclose(
        values,
        np.floor(values),
    ):
        raise AssertionError(
            f"{column} contains non-integer counts."
        )

per_query_metrics_check["retrieved_useful_at_10"] = (
    per_query_metrics_check["retrieved_partial_at_10"]
    + per_query_metrics_check["retrieved_direct_at_10"]
)

if per_query_metrics_check[
    "retrieved_useful_at_10"
].gt(10).any():
    raise AssertionError(
        "Useful retrieved-passage count exceeds top-10 depth."
    )

precision_reconstructed = (
    per_query_metrics_check["retrieved_useful_at_10"]
    / 10.0
)

if not np.allclose(
    per_query_metrics_check[
        "useful_precision_at_10"
    ],
    precision_reconstructed,
    atol=1e-12,
    rtol=0.0,
):
    precision_mismatches = (
        per_query_metrics_check.loc[
            ~np.isclose(
                per_query_metrics_check[
                    "useful_precision_at_10"
                ],
                precision_reconstructed,
                atol=1e-12,
                rtol=0.0,
            ),
            [
                "query_id",
                "method",
                "retrieved_useful_at_10",
                "useful_precision_at_10",
            ],
        ]
    )

    raise AssertionError(
        "Useful Precision@10 is inconsistent with the "
        "retrieved useful-passage counts:\n"
        f"{precision_mismatches}"
    )

no_direct_evidence = (
    per_query_metrics_check[
        "retrieved_direct_at_10"
    ].eq(0)
)

if not np.allclose(
    per_query_metrics_check.loc[
        no_direct_evidence,
        "direct_mrr_at_10",
    ],
    0.0,
    atol=1e-12,
    rtol=0.0,
):
    raise AssertionError(
        "Direct MRR@10 must equal zero when no direct "
        "evidence was retrieved."
    )

direct_evidence_found = (
    per_query_metrics_check[
        "retrieved_direct_at_10"
    ].gt(0)
)

if per_query_metrics_check.loc[
    direct_evidence_found,
    "direct_mrr_at_10",
].le(0).any():
    raise AssertionError(
        "Direct MRR@10 must be positive when direct "
        "evidence was retrieved."
    )


# ------------------------------------------------------------
# 10. Reconstruct useful-pool recall from frozen judgments
# ------------------------------------------------------------

useful_pool_sizes = (
    final_judgments_check.loc[
        final_judgments_check[
            "human_relevance"
        ].ge(1)
    ]
    .groupby("query_id")
    .size()
    .rename("useful_pool_size")
)

missing_useful_pools = (
    metric_query_ids
    - set(
        useful_pool_sizes.index.astype(str)
    )
)

if missing_useful_pools:
    raise AssertionError(
        "Some queries have no useful judged passages: "
        f"{sorted(missing_useful_pools)}"
    )

metric_recall_check = (
    per_query_metrics_check.merge(
        useful_pool_sizes.reset_index(),
        on="query_id",
        how="left",
        validate="many_to_one",
    )
)

metric_recall_check["reconstructed_recall_at_10"] = (
    metric_recall_check["retrieved_useful_at_10"]
    / metric_recall_check["useful_pool_size"]
)

recall_matches = np.isclose(
    metric_recall_check[
        "useful_pool_recall_at_10"
    ],
    metric_recall_check[
        "reconstructed_recall_at_10"
    ],
    atol=1e-12,
    rtol=0.0,
)

if not recall_matches.all():
    recall_mismatches = (
        metric_recall_check.loc[
            ~recall_matches,
            [
                "query_id",
                "method",
                "retrieved_useful_at_10",
                "useful_pool_size",
                "useful_pool_recall_at_10",
                "reconstructed_recall_at_10",
            ],
        ]
    )

    raise AssertionError(
        "Useful-pool Recall@10 is inconsistent with the "
        "frozen judgment pool:\n"
        f"{recall_mismatches}"
    )


# ------------------------------------------------------------
# 11. Reconstruct the aggregate metric table
# ------------------------------------------------------------

aggregate_metric_columns = [
    "useful_precision_at_10",
    "useful_pool_recall_at_10",
    "direct_mrr_at_10",
    "graded_ndcg_at_10",
]

final_aggregate_metrics = (
    per_query_metrics_check
    .groupby(
        "method",
        as_index=False,
    )[aggregate_metric_columns]
    .mean()
    .rename(
        columns={
            "useful_precision_at_10": (
                "mean_useful_precision_at_10"
            ),
            "useful_pool_recall_at_10": (
                "mean_useful_pool_recall_at_10"
            ),
            "direct_mrr_at_10": (
                "mean_direct_mrr_at_10"
            ),
            "graded_ndcg_at_10": (
                "mean_graded_ndcg_at_10"
            ),
        }
    )
)

method_order = pd.CategoricalDtype(
    categories=[
        "BM25",
        "Dense",
        "Hybrid RRF",
    ],
    ordered=True,
)

final_aggregate_metrics["method"] = (
    final_aggregate_metrics["method"]
    .astype(method_order)
)

final_aggregate_metrics = (
    final_aggregate_metrics
    .sort_values("method")
    .reset_index(drop=True)
)

numeric_aggregate_columns = [
    column
    for column in final_aggregate_metrics.columns
    if column != "method"
]

final_aggregate_metrics[
    numeric_aggregate_columns
] = (
    final_aggregate_metrics[
        numeric_aggregate_columns
    ].round(4)
)


# ------------------------------------------------------------
# 12. Build compact benchmark-integrity summary
# ------------------------------------------------------------

relevance_distribution = (
    final_judgments_check[
        "human_relevance"
    ]
    .value_counts()
    .sort_index()
)

integrity_summary = pd.DataFrame(
    [
        {
            "check": "Frozen judgment artifact exists",
            "result": "PASS",
            "detail": str(
                required_artifacts[
                    "final_judgments"
                ]
            ),
        },
        {
            "check": "Frozen metric artifact exists",
            "result": "PASS",
            "detail": str(
                required_artifacts[
                    "per_query_metrics"
                ]
            ),
        },
        {
            "check": "Held-out query count",
            "result": "PASS",
            "detail": (
                f"{len(metric_query_ids)} queries"
            ),
        },
        {
            "check": "Retrieval-method coverage",
            "result": "PASS",
            "detail": (
                "BM25, Dense, Hybrid RRF "
                "for every query"
            ),
        },
        {
            "check": "Per-query metric row count",
            "result": "PASS",
            "detail": (
                f"{len(per_query_metrics_check)} rows"
            ),
        },
        {
            "check": "Unique query-passage judgments",
            "result": "PASS",
            "detail": (
                f"{len(final_judgments_check)} "
                "judgment rows"
            ),
        },
        {
            "check": "Allowed relevance labels",
            "result": "PASS",
            "detail": str(
                {
                    int(label): int(count)
                    for label, count
                    in relevance_distribution.items()
                }
            ),
        },
        {
            "check": "Metric value domains",
            "result": "PASS",
            "detail": (
                "All bounded metrics lie in [0, 1]"
            ),
        },
        {
            "check": "Precision reconstruction",
            "result": "PASS",
            "detail": (
                "Useful Precision@10 matches "
                "useful count / 10"
            ),
        },
        {
            "check": "Recall reconstruction",
            "result": "PASS",
            "detail": (
                "Useful-pool Recall@10 matches "
                "the frozen judgment pool"
            ),
        },
        {
            "check": "Direct-MRR consistency",
            "result": "PASS",
            "detail": (
                "Direct evidence counts and "
                "Direct MRR@10 are consistent"
            ),
        },
        {
            "check": "Artifact fingerprints",
            "result": "PASS",
            "detail": (
                "SHA-256 hashes computed"
            ),
        },
    ]
)


# ------------------------------------------------------------
# 13. Final completion assertion
# ------------------------------------------------------------

assert integrity_summary[
    "result"
].eq("PASS").all()

print("=" * 88)
print("FINAL NOTEBOOK 3 ARTIFACT-INTEGRITY VALIDATION")
print("=" * 88)
print(
    "Status: PASS."
)

print()
print("=" * 88)
print("INTEGRITY SUMMARY")
print("=" * 88)

display(integrity_summary)

print()
print("=" * 88)
print("FROZEN ARTIFACT MANIFEST")
print("=" * 88)

display(artifact_manifest)

print()
print("=" * 88)
print("RECONSTRUCTED FINAL AGGREGATE METRICS")
print("=" * 88)

display(final_aggregate_metrics)

FINAL NOTEBOOK 3 ARTIFACT-INTEGRITY VALIDATION
Status: PASS.

INTEGRITY SUMMARY


,check,result,detail
0,Frozen judgment artifact exists,PASS,/content/drive/MyDrive/FinanceRAG-Analyst/arti...
1,Frozen metric artifact exists,PASS,/content/drive/MyDrive/FinanceRAG-Analyst/arti...
2,Held-out query count,PASS,12 queries
3,Retrieval-method coverage,PASS,"BM25, Dense, Hybrid RRF for every query"
4,Per-query metric row count,PASS,36 rows
5,Unique query-passage judgments,PASS,214 judgment rows
6,Allowed relevance labels,PASS,"{0: 128, 1: 43, 2: 43}"
7,Metric value domains,PASS,"All bounded metrics lie in [0, 1]"
8,Precision reconstruction,PASS,Useful Precision@10 matches useful count / 10
9,Recall reconstruction,PASS,Useful-pool Recall@10 matches the frozen judgm...



FROZEN ARTIFACT MANIFEST


,artifact,path,file_size_bytes,sha256
0,final_judgments,/content/drive/MyDrive/FinanceRAG-Analyst/arti...,236537,f1af7a4b9b37894573bc1cb549edd644dc1389b60b93f8...
1,per_query_metrics,/content/drive/MyDrive/FinanceRAG-Analyst/arti...,18836,dba29271d7ecf4ce5970b06a202d97d57cd7a59422a066...



RECONSTRUCTED FINAL AGGREGATE METRICS


,method,mean_useful_precision_at_10,mean_useful_pool_recall_at_10,mean_direct_mrr_at_10,mean_graded_ndcg_at_10
0,BM25,0.3917,0.5686,0.4944,0.5118
1,Dense,0.5083,0.7456,0.4510,0.5821
2,Hybrid RRF,0.4417,0.6071,0.6736,0.5986


In [ ]:
# ============================================================
# NOTEBOOK 3: FREEZE FINAL EVALUATION CHECKPOINT
# ============================================================

from pathlib import Path
import json
from datetime import datetime, timezone

CHECKPOINT_DIR = Path("../artifacts/evaluation")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

AGGREGATE_METRICS_PATH = (
    CHECKPOINT_DIR / "heldout_aggregate_metrics.parquet"
)
ARTIFACT_MANIFEST_PATH = (
    CHECKPOINT_DIR / "heldout_artifact_manifest.parquet"
)
CHECKPOINT_METADATA_PATH = (
    CHECKPOINT_DIR / "notebook_03_checkpoint.json"
)

# Refuse to overwrite an existing frozen checkpoint.
for path in [
    AGGREGATE_METRICS_PATH,
    ARTIFACT_MANIFEST_PATH,
    CHECKPOINT_METADATA_PATH,
]:
    if path.exists():
        raise FileExistsError(
            f"Frozen checkpoint already exists and was not overwritten: {path}"
        )

final_aggregate_metrics.to_parquet(
    AGGREGATE_METRICS_PATH,
    index=False,
)

artifact_manifest.to_parquet(
    ARTIFACT_MANIFEST_PATH,
    index=False,
)

checkpoint_metadata = {
    "checkpoint_name": "notebook_03_heldout_retrieval_evaluation",
    "checkpoint_version": "v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "frozen",
    "heldout_query_count": int(
        per_query_metrics_check["query_id"].nunique()
    ),
    "judgment_count": int(len(final_judgments_check)),
    "metric_row_count": int(len(per_query_metrics_check)),
    "methods": [
        "BM25",
        "Dense",
        "Hybrid RRF",
    ],
    "relevance_scale": {
        "0": "irrelevant",
        "1": "partial evidence",
        "2": "direct evidence",
    },
    "selected_generation_retriever": "Hybrid RRF",
    "selection_rationale": (
        "Highest mean Direct MRR@10 and highest mean graded "
        "nDCG@10 on the frozen held-out benchmark. Dense remains "
        "the strongest method for useful evidence coverage."
    ),
    "source_artifacts": {
        row["artifact"]: {
            "path": row["path"],
            "file_size_bytes": int(row["file_size_bytes"]),
            "sha256": row["sha256"],
        }
        for row in artifact_manifest.to_dict("records")
    },
    "output_artifacts": {
        "aggregate_metrics": str(AGGREGATE_METRICS_PATH),
        "artifact_manifest": str(ARTIFACT_MANIFEST_PATH),
    },
}

CHECKPOINT_METADATA_PATH.write_text(
    json.dumps(
        checkpoint_metadata,
        indent=2,
    ),
    encoding="utf-8",
)

print("=" * 88)
print("NOTEBOOK 3 FINAL CHECKPOINT")
print("=" * 88)
print("Status: FROZEN")
print(f"Aggregate metrics: {AGGREGATE_METRICS_PATH}")
print(f"Artifact manifest: {ARTIFACT_MANIFEST_PATH}")
print(f"Checkpoint metadata: {CHECKPOINT_METADATA_PATH}")

NOTEBOOK 3 FINAL CHECKPOINT
Status: FROZEN
Aggregate metrics: ../artifacts/evaluation/heldout_aggregate_metrics.parquet
Artifact manifest: ../artifacts/evaluation/heldout_artifact_manifest.parquet
Checkpoint metadata: ../artifacts/evaluation/notebook_03_checkpoint.json
